<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/HIDROCARBUROS/PP_Produccion_Lotes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══ BLOQUE 1 · INSTALACIÓN Y EXTRACTOR (ejecutar una vez por sesión) ═══
!pip install pdfplumber pypdf requests certifi cryptography pandas ipywidgets google-api-python-client xlrd --quiet
import os, io, re, json, time, html, hashlib, calendar, statistics, unicodedata, warnings
from datetime import date
from urllib.parse import urljoin, unquote
import requests
import pandas as pd
import openpyxl
import pdfplumber
import ipywidgets as w
from IPython.display import display
print("✅ Librerías listas")

# ── Extractor de las tablas "PRODUCCIÓN FISCALIZADA MENSUAL" de HIDROCARBUROS LÍQUIDOS y de GAS NATURAL ──
# Lee la tabla por coordenadas (no como texto plano) para no mezclar columnas con gráficos o notas.
try:
    import pypdf
except ImportError:                                  # sin pypdf se recorren todas las páginas (más lento)
    pypdf = None

MESES = ["ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO", "JULIO", "AGOSTO",
         "SETIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"]
ALIAS_MES = {"SEPTIEMBRE": "SETIEMBRE", "ENE": "ENERO", "FEB": "FEBRERO", "MAR": "MARZO", "ABR": "ABRIL",
             "MAY": "MAYO", "JUN": "JUNIO", "JUL": "JULIO", "AGO": "AGOSTO", "SET": "SETIEMBRE",
             "SEP": "SETIEMBRE", "OCT": "OCTUBRE", "NOV": "NOVIEMBRE", "DIC": "DICIEMBRE"}
COL_EMPRESA = ("OPERADOR", "CONTRATISTA", "COMPANIA")
NUM = re.compile(r"^-?\d[\d,.]*$")
LOTE_OK = re.compile(r"^(\d{1,3}(-?[A-Z]{1,2}(/[A-Z])?)?|1-AB|[IVXL]+(/[IVXL]+)?|Z-\d{1,2}[A-Z]?)(\(T{1,2}\))?$")
OPER_OK = re.compile(r"^[A-ZÁÉÍÓÚÑÜ&.\-/]{2,}$")
ZONAS = ["SELVA NORTE", "SELVA CENTRAL", "SELVA SUR", "NOROESTE", "NOR-OESTE", "ZOCALO"]


def sin_tildes(t):
    return "".join(c for c in unicodedata.normalize("NFKD", t) if not unicodedata.combining(c)).upper()


def mes_de(t):
    t = sin_tildes(t).strip(".")
    t = ALIAS_MES.get(t, t)
    return t if t in MESES else None


def lote_norm(t):
    t = re.sub(r"\(\s*(LGN|COND\.?)\s*\)", "", t.upper())          # "31-C ( LGN )" -> "31-C"
    t = re.sub(r"\s*\(\d\)\s*$", "", t.strip())                     # "192 (1)" -> "192"
    t = t.rstrip(".,").replace(" ", "")
    t = re.sub(r"^31-?([A-Z])/(?:31-?)?([A-Z])$", r"31-\1/\2", t)    # "31B/D", "31B/31D" -> "31-B/D"
    return {"1AB": "1-AB"}.get(t, t)


def producto_de_lote(t):
    t = t.upper()
    return "CONDENSADO" if "COND" in t else ("LGN" if "LGN" in t else None)


def a_num(t, miles_punto=False):
    if t in ("-", "–"):
        return 0.0
    if miles_punto:
        return float(t.replace(".", "").replace(",", "."))
    return float(t.replace(",", ""))


ZONA_FIJA = {"1-AB": "SELVA NORTE", "192": "SELVA NORTE", "8": "SELVA NORTE", "67": "SELVA NORTE",
             "95": "SELVA NORTE", "102": "SELVA NORTE", "64": "SELVA NORTE", "39": "SELVA NORTE",
             "131": "SELVA CENTRAL", "31-B": "SELVA CENTRAL", "31-C": "SELVA CENTRAL", "31-D": "SELVA CENTRAL",
             "31-E": "SELVA CENTRAL", "31-B/D": "SELVA CENTRAL", "LOB/EA/LO": "NOROESTE",
             "88": "SELVA SUR", "56": "SELVA SUR", "57": "SELVA SUR", "58": "SELVA SUR"}


# Mismo campo con distinto contrato a lo largo del tiempo (para series históricas y rankings)
LOTE_HISTORICO = {"1-AB": "1-AB/192", "192": "1-AB/192",      # 1-AB terminó el 29/08/2015; 192 desde el 30/08/2015
                  "Z-2B": "Z-2B/Z-69", "Z-69": "Z-2B/Z-69",     # Z-2B pasó a Z-69 (PetroPerú, 2023)
                  "31-B": "31-B/D", "31-D": "31-B/D"}           # Maple: Maquía y Agua Caliente por separado hasta 2000


# Nombres de operadores: el PDF corta los nombres largos en dos renglones y tiene erratas; se homologan.
# Se conserva el nombre leído en OPERADOR_ORIGINAL.
OPERADOR_NORM = [
    (r"^PLUSPETRO$|^PLUSPETROL PERU( CORP\.?)?$", "PLUSPETROL"),
    (r"^PPN( EN( LIQUIDACI[OÓ]N)?)?$", "PLUSPETROL NORTE (EN LIQUIDACIÓN)"),
    (r"^UPLAND( OIL( AND( GAS)?)?)?$", "UPLAND OIL AND GAS"),
    (r"^CORPORACI[OÓ]N DEL( SUR)?$", "CORPORACIÓN DEL SUR"),
    (r"^PETROLERA MONTERR+ICO$|^PETROMONT$", "PETROLERA MONTERRICO"),   # PETROMONT = su nombre corto
    (r"^UNIENE?R?G[IÍ]A( ABC)?$", "UNIENERGÍA ABC"),
    (r"^UNIPETRO( ABC)?$", "UNIPETRO ABC"),
    (r"^PETRO-?TECH$", "PETROTECH"),
    (r"^G\.? ?M\.? ?P\.?( S\.? ?A\.?)?$", "GMP"),
    (r"^PETROPER[UÚ]$", "PETROPERÚ"),
    (r"^VIGO( ENERGY)?$", "VIGO ENERGY"),
    (r"^PACIFIC( STRATUS)?$", "PACIFIC STRATUS"),
    (r"^(DPB|BPD|M?M?BBL|LBB)$", ""),               # rótulos de gráficos leídos como operador
]

# Rótulos de los gráficos que el PDF deja junto a las tablas (no son operadores)
ETIQUETA_GRAFICO = re.compile(r"^\s*(DPB|BPD|M?M?BBL|LBB|MM|BLS?|P[AÁ]GINA\b.*)\s*$", re.I)

def norm_operador(nombre):
    """Homologa un nombre de operador; los consorcios 'A / B' se homologan por partes."""
    partes = []
    if nombre is None or (isinstance(nombre, float) and nombre != nombre):    # None / NaN
        return ""
    for parte in str(nombre).split("/"):
        x = re.sub(r"\s+", " ", parte).strip().upper()
        for patron, bueno in OPERADOR_NORM:
            if re.match(patron, x):
                x = bueno
                break
        if x and x not in ("NAN", "NONE"):
            partes.append(x)
    return " / ".join(partes)


def lote_historico(lote):
    base = re.sub(r"\(T+\)$", "", lote)                        # contratos temporales: I(T), VI(T), Z-69(T)
    return LOTE_HISTORICO.get(base, base)


def zona_de(lote):
    base = re.sub(r"\(T+\)$", "", lote)
    if base in ZONA_FIJA:
        return ZONA_FIJA[base]
    if base.startswith("Z-"):
        return "ZOCALO"
    if re.fullmatch(r"[IVXL]+(/[IVXL]+)?", base):
        return "NOROESTE"
    return ""


def paginas_candidatas(path, n, fluido="LIQUIDOS"):
    """Índices de páginas con la tabla mensual o el resumen, ubicados con una lectura rápida (pypdf).
    Los PDF anuales tienen ~90 páginas con gráficos y mapas: pdfplumber solo analiza estas."""
    if pypdf is None:
        return list(range(n))
    try:
        textos = [sin_tildes(p.extract_text() or "") for p in pypdf.PdfReader(path).pages]
    except Exception:
        return list(range(n))
    if fluido == "GAS":
        # en las páginas giradas pypdf devuelve cada palabra al revés ("LARUTAN"): se prueba también así
        al_reves = lambda t: " ".join(l[::-1] for l in t.splitlines())
        cand = [i for i, t in enumerate(textos)
                if any(("FISCALIZADA" in x and (("NATURAL" in x and "GAS" in x) or re.search(r"\((MSCF|MPC|MCF)\)", x)))
                       or ("MENSUAL" in x and re.search(r"\((MSCF|MPC|MCF)\)", x))
                       for x in (t, al_reves(t)))]
    else:
        cand = [i for i, t in enumerate(textos)
                if ("MENSUAL" in t and ("HIDROCARBUROS LIQUIDOS" in t or "FISCALIZADA" in t))
                or "RESUMEN DE PRODUCCION" in t or re.search(r"FISCALIZADA DE HIDROCARBUROS LIQUIDOS\s*-", t)]
    if not cand and sum(len(t) for t in textos) > 500:
        return list(range(n))                        # texto raro: por seguridad, todas
    return cand


def operadores_resumen(pdf, paginas=None):
    """{lote: operador} de las tablas resumen (anual 'RESUMEN DE PRODUCCIÓN…', mensual 4.1)."""
    out = {}
    patron = re.compile(r"^([A-ZÁÉÍÓÚÑÜ&.\- ]+?)\s*(?:\(\d\))?\s+(\S+)\s+[\d,.]+\s+[\d,.]+\s+[\d.,]+$")
    for ip in (paginas if paginas is not None else range(len(pdf.pages))):
        t = pdf.pages[ip].extract_text() or ""
        cab = sin_tildes(t[:200])
        if not (re.search(r"RESUMEN DE PRODUCCION", cab)
                or re.search(r"PRODUCCION FISCALIZADA DE (HIDROCARBUROS LIQUIDOS|GAS NATURAL) - [A-Z]+ \d{4}", cab)):
            continue
        for linea in t.splitlines():
            m = patron.match(linea.strip())
            if not m:
                continue
            oper, lote = m.group(1).strip(), lote_norm(m.group(2))
            for z in ["SELVA NORTE", "SELVA CENTRAL", "SELVA SUR", "NOROESTE", "ZÓCALO", "ZOCALO"]:
                oper = re.sub(r"^" + z + r"\s+", "", oper)
            if oper and LOTE_OK.match(lote) and OPER_OK.match(oper.split()[0]) and "TOTAL" not in oper:
                out[lote] = oper
    return out


def info_reporte(pdf):
    txt = sin_tildes(" ".join((pdf.pages[i].extract_text() or "") for i in range(min(2, len(pdf.pages)))))
    m = re.search(r"ESTADISTICA\s+ANUAL\s+DE\s+HIDROCARBUROS\s+(\d{4})", txt)
    if m:
        return "ANUAL", int(m.group(1)), 12
    m = re.search(r"ESTADISTICA\s+MENSUAL\s+DE\s+HIDROCARBUROS\s+(" + "|".join(MESES + ["SEPTIEMBRE"]) + r")\s+(\d{4})", txt)
    if m:
        return "MENSUAL", int(m.group(2)), MESES.index(mes_de(m.group(1))) + 1
    return None, None, None


def lineas(words, tol=2.5):
    out = []
    for w in sorted(words, key=lambda w: (round(w["top"]), w["x0"])):
        if out and abs(out[-1][0] - w["top"]) <= tol:
            out[-1][1].append(w)
        else:
            out.append([w["top"], [w]])
    return [(top, sorted(ws, key=lambda w: w["x0"])) for top, ws in out]


def palabras(pg):
    """Palabras con coordenadas "de lectura". Algunos anuales (p. ej. 2008) dibujan las tablas horizontales con
    el texto girado 90°: se leen en el sentido correcto y se rotan las coordenadas para que filas y columnas
    queden como en una página normal."""
    chars = pg.chars
    girados = [c for c in chars if not c.get("upright", True)]
    if len(girados) <= len(chars) / 2:
        return pg.extract_words()
    antihorario = sum(1 for c in girados if c["matrix"][1] > 0) >= len(girados) / 2
    if antihorario:                                  # se lee de abajo hacia arriba
        ws = pg.extract_words(char_dir_rotated="btt", line_dir_rotated="ltr")
        conv = lambda w: {"x0": pg.height - w["bottom"], "x1": pg.height - w["top"], "top": w["x0"], "bottom": w["x1"]}
    else:                                            # se lee de arriba hacia abajo
        ws = pg.extract_words(char_dir_rotated="ttb", line_dir_rotated="rtl")
        conv = lambda w: {"x0": w["top"], "x1": w["bottom"], "top": pg.width - w["x1"], "bottom": pg.width - w["x0"]}
    return [{**w, **conv(w)} for w in ws]


def titulo_tabla(ws, fluido="LIQUIDOS"):
    """Título de la página si es la tabla MENSUAL de producción fiscalizada (de líquidos o de gas); si no, None."""
    t = sin_tildes(" ".join(w["text"] for w in ws[:20]))
    if fluido == "GAS":
        # "PRODUCCIÓN FISCALIZADA MENSUAL DE GAS NATURAL 2021 (MPC)" / "PRODUCCION MENSUAL FISCALIZADA DE GAS NATURAL - 2001"
        # (se descartan la versión en MMBTU, la producción "de campo" y los balances)
        # ("… DE GAS NATURAL MENSUAL (MPC) - 2012" en los anuales con la página girada)
        cab = t.split("LOTE")[0]
        # 2008: "PRODUCCIÓN FISCALIZADA MENSUAL (Mscf) - 2008" (sin "gas": se reconoce por la unidad)
        # 2013: "PRODUCCIÓN MENSUAL (MPC) - 2013" (sin "fiscalizada" ni "gas"); 2024–2025: la tabla mensual en MPC se titula
        # "RESUMEN DE PRODUCCIÓN FISCALIZADA DE GAS NATURAL 2024". Se descartan las versiones en energía
        # ("PODER CALORÍFICO", "UNIDADES DE CALOR", MMBTU), la producción "de campo" y los balances.
        unidad = re.search(r"\((MSCF|MPC|MCF)\)", cab)
        es_gas = "GAS NATURAL" in cab or unidad
        mensual = "MENSUAL" in cab and ("FISCALIZADA" in cab or unidad)
        resumen = "RESUMEN DE PRODUCCION FISCALIZADA DE GAS NATURAL" in cab
        if es_gas and (mensual or resumen) and not re.search(r"MMBTU|CAMPO|BALANCE|BLS|LIQUIDOS|CALOR", cab):
            return t
        return None
    # 2016 titula "PRODUCCIÓN DE HIDROCARBUROS LIQUIDOS MENSUAL 2016" (sin "FISCALIZADA")
    if "MENSUAL" not in t or "CAMPO" in t or not ("FISCALIZADA" in t or "HIDROCARBUROS LIQUIDOS" in t):
        return None
    if re.search(r"MENSUAL DE GAS|GAS NATURAL", t.split("LOTE")[0][:150]):
        return None
    if "HIDROCARBUROS LIQUIDOS" in t or re.search(r"FISCALIZADA MENSUAL\s*-?\s*\d{4}", t):   # "… MENSUAL - 2010"
        return t
    return None


def pagina_tabla(pg, fluido="LIQUIDOS"):
    """Columnas de la tabla mensual por lote, o None si la página no la tiene."""
    ws = sorted(palabras(pg), key=lambda w: (round(w["top"]), w["x0"]))
    titulo = titulo_tabla(ws, fluido)
    if not titulo:
        return None
    for top, lw in lineas(ws):
        textos = [sin_tildes(w["text"]) for w in lw]
        meses = [mes_de(t) for t in textos]
        if "LOTE" in textos and "ENERO" in meses and "DICIEMBRE" in meses:
            col = {}
            for w, t, m in zip(lw, textos, meses):
                if m:
                    col[m] = w
                elif t in ("TOTAL", "LOTE", "ZONA") or t in COL_EMPRESA:
                    col["OPERADOR" if t in COL_EMPRESA else t] = w
            anio = re.findall(r"(?<!\d)(19\d\d|20\d\d)(?!\d)", titulo)
            return ws, top, col, (int(anio[0]) if anio else None)
    return None


def extraer(path, fluido="LIQUIDOS"):
    filas, totales = [], []
    anio_tabla = None
    with pdfplumber.open(path) as pdf:
        tipo, anio, mes_rep = info_reporte(pdf)
        candidatas = paginas_candidatas(path, len(pdf.pages), fluido)
        for ip in candidatas:
            pg = pdf.pages[ip]
            r = pagina_tabla(pg, fluido)
            if not r:
                continue
            ws, top_hdr, col, anio_tit = r
            if fluido == "GAS":
                # gas 2007 (SAPET VII/VI): el PDF imprime "-3616", "3-3402", "4-3034" por 3,616 / 33,402 / 43,034
                # (con esos valores la fila suma su TOTAL). En gas fiscalizado no hay negativos: se quita el guion.
                for w in ws:
                    if re.fullmatch(r"-\d{3,}|\d{1,3}-\d{3,4}", w["text"]):
                        w["text"] = w["text"].replace("-", "")
            anio_tabla = anio_tabla or anio_tit
            # separador de miles de la página: "19.993" (antiguos) o "19,993"
            nums = [w["text"] for w in ws if re.fullmatch(r"\d{1,3}([.,]\d{3})+(\.\d+)?", w["text"])]
            miles_punto = sum(1 for t in nums if re.fullmatch(r"\d{1,3}(\.\d{3})+", t)) > len(nums) / 2
            centros = {m: (col[m]["x0"] + col[m]["x1"]) / 2 for m in MESES + ["TOTAL"] if m in col}
            lim_lote_der = (col["LOTE"]["x1"] + col["ENERO"]["x0"]) / 2
            lim_oper_der = col["LOTE"]["x0"] - 12
            x_oper = col["OPERADOR"]["x0"] if "OPERADOR" in col else 0
            lim_zona_der = x_oper - 4 if "ZONA" in col else -1
            paso = (centros["DICIEMBRE"] - centros["ENERO"]) / 11
            producto, bloque = ("GAS NATURAL" if fluido == "GAS" else None), []
            etiquetas_oper, etiquetas_zona = [], []

            def cerrar_bloque():
                # filas sin operador: toman la etiqueta de operador suelta más cercana del bloque
                for f in bloque:
                    if not f["OPERADOR"] and etiquetas_oper:
                        f["OPERADOR"] = min(etiquetas_oper, key=lambda e: abs(e[0] - f["_top"]))[1]
                    if "ZONA" in col and etiquetas_zona:
                        f["ZONA"] = min(etiquetas_zona, key=lambda e: abs(e[0] - f["_top"]))[1]
                filas.extend(bloque)

            # gas 2011: el rótulo del lote va 3 pt más arriba que sus valores; se agrupa con más tolerancia
            for top, lw in lineas(ws, tol=3.6 if fluido == "GAS" else 2.5):
                if top <= top_hdr + 2:
                    continue
                txt = sin_tildes(" ".join(w["text"] for w in lw)).strip()
                izq = [w for w in lw if (w["x0"] + w["x1"]) / 2 < lim_lote_der]
                nums = [w for w in lw if (w["x0"] + w["x1"]) / 2 > lim_lote_der and (NUM.match(w["text"]) or w["text"] in "-–")]
                # quita "Pág. 24" u otro n° de página pegado a la izquierda del rótulo del producto
                cab = re.sub(r"^((PAG\.?|\d+)\s+)+", "", txt)
                hay_valores = any((w["x0"] + w["x1"]) / 2 > lim_lote_der for w in nums)
                if fluido != "GAS" and not hay_valores and (re.match(r"^PETROLEO(\s*\(B[A-Z]*\))?$", cab)
                                        or re.match(r"^LIQUIDOS DE GAS NATURAL", cab)):
                    cerrar_bloque(); bloque, etiquetas_oper = [], []
                    producto = "PETROLEO" if cab.startswith("PETROLEO") else "LGN"
                    continue
                if producto is None:
                    continue
                vals = {}
                for w in nums:
                    xc = (w["x0"] + w["x1"]) / 2
                    m = min(centros, key=lambda k: abs(centros[k] - xc))
                    if abs(centros[m] - xc) <= paso * 0.6:
                        try:
                            vals[m] = a_num(w["text"], miles_punto)
                        except ValueError:
                            pass
                txt_izq = sin_tildes(" ".join(w["text"] for w in izq)).strip()
                es_bd = re.search(r"\(\s*B/D\s*\)", txt_izq)
                if re.search(r"SUB-?TOTAL|^TOTAL|TOTAL(ES)?\s*$|PROMEDIO|\(BPD\)|MMPCD|MMCFD|\(MPCD\)", txt_izq) or es_bd:
                    if vals and not re.search(r"PROMEDIO|BPD|MMPCD|MMCFD|MPCD", txt_izq) and not es_bd:
                        totales.append({"PAGINA": ip + 1, "PRODUCTO": producto, "ETIQUETA": txt_izq, **vals})
                    continue
                lote_ws = [w for w in izq if w["x0"] >= lim_oper_der - 2 and not re.fullmatch(r"\(\d\)", w["text"])]
                oper_ws = [w for w in izq if w["x1"] < lim_oper_der and w["x0"] >= lim_zona_der
                           and OPER_OK.match(w["text"])]
                zona_ws = [w for w in izq if w["x1"] <= lim_zona_der + 2 and OPER_OK.match(w["text"])]
                zona_txt = sin_tildes(" ".join(w["text"] for w in zona_ws))
                if zona_txt and any(z in zona_txt for z in ZONAS):
                    etiquetas_zona.append((top, zona_txt.replace("NOR-OESTE", "NOROESTE")))
                lote_txt = " ".join(w["text"] for w in lote_ws)
                if lote_ws and vals and LOTE_OK.match(lote_norm(lote_txt)):
                    meses = {m: vals.get(m) for m in MESES}
                    bloque.append({"_top": top, "PAGINA": ip + 1,
                                   "PRODUCTO": producto_de_lote(lote_txt) or producto,
                                   "ZONA": "", "OPERADOR": " ".join(w["text"] for w in oper_ws).strip(),
                                   "LOTE_ORIGINAL": lote_txt, "LOTE": lote_norm(lote_txt),
                                   **meses, "TOTAL": vals.get("TOTAL")})
                elif oper_ws and not vals and not lote_ws and not ETIQUETA_GRAFICO.match(" ".join(w["text"] for w in oper_ws)):
                    etiquetas_oper.append((top, " ".join(w["text"] for w in oper_ws)))
            cerrar_bloque()
        if not tipo and filas:                     # portada sin el título esperado: año de la tabla
            tipo, anio, mes_rep = "ANUAL", anio_tabla, 12
        if not tipo or not anio:
            letras = sum(len(pg.extract_text() or "") for pg in pdf.pages[:15])
            return {"TIPO": None, "MOTIVO": "ESCANEADO (sin texto)" if letras < 200 else "FORMATO ANTIGUO (sin tabla reconocible)"}
        oper_res = operadores_resumen(pdf, candidatas)
        if fluido == "GAS" and not filas:          # PDF sin tabla de gas (o con otro diseño): no es un error
            return {"TIPO": tipo, "ANIO": anio, "MES_REPORTE": mes_rep, "filas": [], "totales": []}
    for f in filas:
        f["ZONA"] = zona_de(f["LOTE"]) or f["ZONA"]
        if f["OPERADOR"] == "" or len(f["OPERADOR"]) < 3:
            f["OPERADOR"] = oper_res.get(f["LOTE"], f["OPERADOR"])
    # imputación: si falta exactamente un mes (dentro del período) y el total lo permite
    hasta = 12 if tipo == "ANUAL" else mes_rep
    for f in filas:
        f.pop("_top", None)
        dentro = MESES[:hasta]
        vacios = [m for m in dentro if f[m] is None]
        suma = sum(f[m] or 0 for m in dentro)
        f["ESTADO"] = "OK"
        # gas: los meses traen decimales y el PDF redondea cada celda; la suma puede diferir del total en unas unidades
        tol = 6 if fluido == "GAS" else 0.5
        if f["TOTAL"] is not None and abs(suma - f["TOTAL"]) > tol:
            if len(vacios) == 1:
                f[vacios[0]] = f["TOTAL"] - suma
                f["ESTADO"] = f"IMPUTADO {vacios[0]}"
            else:
                f["ESTADO"] = "REVISAR"
        if f["TOTAL"] is None:                     # tabla sin columna TOTAL visible
            f["TOTAL"] = suma
        for m in dentro:
            if f[m] is None:
                f[m] = 0.0
    return {"TIPO": tipo, "ANIO": anio, "MES_REPORTE": mes_rep, "filas": filas, "totales": totales}


# ── Lector de los «Reportes de producción fiscalizada» DIARIOS (tabla día × lote, leída por coordenadas) ──

MESES_ES = ["ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO", "JULIO", "AGOSTO", "SETIEMBRE", "OCTUBRE",
            "NOVIEMBRE", "DICIEMBRE"]
NUMERO = re.compile(r"^-?\d{1,3}(,\d{3})*(\.\d+)?$|^-?\d+(\.\d+)?$")
LOTE_TXT = re.compile(r"^(\d{1,3}(-?[A-Z]{1,2})?|[IVXL]+|Z-\d{1,2}[A-Z]?|\d{1,3}_LIC|192_LIC)\.?$")
# Operadores que a veces salen en la cabecera sin el rótulo «Lote …» debajo (nombre largo en tres renglones)
LOTE_DE_OPERADOR = {"LATIN AMERICAN ENERGY": "VI", "LAET": "VI"}
NO_OPERADOR = {"LOTE", "DIA", "SUB", "TOTAL", "MCF", "MMBTU", "M3", "M3/D", ":", "LOTE:", "MES", "ANO", "SEPTIEMBRE"} | set(MESES_ES)


def _sin_tildes(t):
    return "".join(c for c in unicodedata.normalize("NFD", str(t)) if unicodedata.category(c) != "Mn").upper()


def lote_diario(t):
    """'X.' → 'X', '192_LIC' → '192', 'Z-2B' → 'Z-2B', 'I (TT)' → 'I(TT)'."""
    t = t.upper().replace(" ", "").rstrip(".")
    t = re.sub(r"_LIC$", "", t)
    return t


def _lineas(ws, tol=2.6):
    out = []
    for w in sorted(ws, key=lambda w: (w["top"], w["x0"])):
        if out and abs(out[-1][0] - w["top"]) <= tol:
            out[-1][1].append(w)
        else:
            out.append([w["top"], [w]])
    return [(t, sorted(l, key=lambda w: w["x0"])) for t, l in out]


def _cabecera(pg):
    """Título, unidad, año y mes de la página."""
    t = _sin_tildes(pg.extract_text() or "")
    tit = re.search(r"REPORTE DE PRODUCCION FISCALIZADA DE (HIDROCARBUROS LIQUIDOS|GAS NATURAL)[^(\n]*\(([^)]+)\)", t)
    anio = re.search(r"ANO\s+(\d{4})", t)
    mes = re.search(r"MES\s+(" + "|".join(MESES_ES + ["SEPTIEMBRE"]) + r")", t)
    rep = re.search(r"REPORTE AL\s*:\s*(\d{2}/\d{2}/\d{4})", t)
    if not (tit and anio and mes):
        return None
    unidad = tit.group(2).strip().upper().replace("BARRILES", "BBL").replace("M3/D", "M3")
    m = mes.group(1).replace("SEPTIEMBRE", "SETIEMBRE")
    return {"FLUIDO": "LIQUIDOS" if "LIQUIDOS" in tit.group(1) else "GAS NATURAL", "UNIDAD": unidad,
            "AÑO": int(anio.group(1)), "MES": MESES_ES.index(m) + 1, "REPORTE_AL": rep.group(1) if rep else ""}


def _num(t):
    return float(t.replace(",", ""))


def leer_pagina(pg):
    cab = _cabecera(pg)
    if not cab:
        return None
    # separar por tipo de letra: en el reporte en m3 los rótulos en negrita de UNNA («Lote III.») se montan sobre
    # los números del día 1 y, sin esto, salen palabras mezcladas como «1II6I.9,631»
    ws = pg.extract_words(extra_attrs=["fontname"])
    lineas = _lineas(ws)
    # 1. fila de lotes: la primera línea con 5+ códigos de lote (en gas van como "Lote : 31-C")
    fila_lotes = None
    for top, lw in lineas:
        cols = []
        for i, w in enumerate(lw):
            t = w["text"].upper()
            if LOTE_TXT.match(t) and not (i and lw[i - 1]["text"].upper() in ("AÑO", "ANO")):
                x0, x1, txt = w["x0"], w["x1"], w["text"]
                if i + 1 < len(lw) and re.fullmatch(r"\(T+\)", lw[i + 1]["text"]):     # "I (TT)"
                    x1, txt = lw[i + 1]["x1"], txt + lw[i + 1]["text"]
                if cab["FLUIDO"] == "GAS NATURAL" and i >= 2 and lw[i - 2]["text"].upper() == "LOTE":
                    x0 = lw[i - 2]["x0"]                     # el rótulo "Lote : 31-C" completo está centrado
                cols.append({"LOTE_ORIGINAL": txt, "LOTE": lote_diario(txt), "c": (x0 + x1) / 2, "top": top})
        if len(cols) >= 5:
            fila_lotes = (top, cols)
            break
    if not fila_lotes:
        return None
    top_l, cols = fila_lotes
    # rótulos de lote en otro renglón de la cabecera (operador con nombre largo en dos líneas: «UNNA ENERGIA» / «Lote III.»)
    for w in ws:
        t = w["text"].upper()
        if (0 < abs(w["top"] - top_l) <= 12 and LOTE_TXT.match(t) and "BOLD" in w.get("fontname", "").upper()
                and not NUMERO.match(w["text"]) and all(abs((w["x0"] + w["x1"]) / 2 - c["c"]) > 12 for c in cols)):
            vecinos = [v for v in ws if abs(v["top"] - w["top"]) < 1 and v["text"].upper() == "LOTE" and 0 < w["x0"] - v["x1"] < 20]
            if not vecinos:                          # sin «Lote» delante es una letra suelta del operador («I N EXPORT»)
                continue
            x0 = vecinos[0]["x0"]
            cols.append({"LOTE_ORIGINAL": w["text"], "LOTE": lote_diario(w["text"]), "c": (x0 + w["x1"]) / 2, "top": w["top"]})
    # 2. columnas SUB TOTAL / TOTAL (en la cabecera, hasta 20 pt arriba de la fila de lotes)
    tot = [w for w in ws if w["text"].upper() == "TOTAL" and top_l - 20 <= w["top"] <= top_l + 3
           and w["x0"] > min(c["c"] for c in cols)]
    subs = [w for w in ws if w["text"].upper() == "SUB" and top_l - 22 <= w["top"] <= top_l]
    for w in tot:
        es_sub = any(abs((s["x0"] + s["x1"]) / 2 - (w["x0"] + w["x1"]) / 2) < 15 and s["top"] < w["top"] for s in subs)
        cols.append({"LOTE_ORIGINAL": "SUB TOTAL" if es_sub else "TOTAL", "LOTE": "SUBTOTAL" if es_sub else "TOTAL",
                     "c": (w["x0"] + w["x1"]) / 2, "top": w["top"]})
    cols.sort(key=lambda c: c["c"])
    dif = [b["c"] - a["c"] for a, b in zip(cols, cols[1:])]
    ancho = statistics.median(dif) if dif else 30
    # 3. producto de cada columna: en líquidos, antes del primer SUB TOTAL = petróleo y condensado; luego LGN
    grupo = "PETROLEO Y CONDENSADO" if cab["FLUIDO"] == "LIQUIDOS" else "GAS NATURAL"
    for c in cols:
        c["PRODUCTO"] = grupo
        if c["LOTE"] == "SUBTOTAL" and grupo == "PETROLEO Y CONDENSADO":
            grupo = "LGN"
    # 4. operadores: palabras de la cabecera agrupadas en frases (un mismo operador puede abarcar varios lotes)
    cand = [w for w in ws if top_l - 14 <= w["top"] <= top_l + 20 and abs(w["top"] - top_l) > 2.5
            and not NUMERO.match(w["text"]) and _sin_tildes(w["text"]) not in NO_OPERADOR
            and not re.fullmatch(r"(19|20)\d\d", w["text"])
            and not LOTE_TXT.match(w["text"].upper())]
    frases = []
    for top, lw in _lineas(cand):
        for w in lw:
            txt = re.sub(r"MCF|MMBTU", "", w["text"], flags=re.I)        # "ENERMcfGIA" → "ENERGIA"
            if not txt:
                continue
            if frases and frases[-1]["top"] == top and w["x0"] - frases[-1]["x1"] < 4:
                frases[-1].update(x1=w["x1"], txt=frases[-1]["txt"] + " " + txt)
            else:
                frases.append({"top": top, "x0": w["x0"], "x1": w["x1"], "txt": txt})
    # una frase en el renglón de abajo que se superpone a otra de arriba la completa ("CORPORACIO" + "N DEL SUR")
    finales = []
    for f in sorted(frases, key=lambda f: f["top"]):
        arriba = [g for g in finales if g["top"] < f["top"] and f["x0"] < g["x1"] and f["x1"] > g["x0"]]
        if arriba:
            g = arriba[-1]
            g["txt"] = g["txt"] + ("" if g["txt"].endswith(("CORPORACIO",)) else " ") + f["txt"]
            g["x0"], g["x1"] = min(g["x0"], f["x0"]), max(g["x1"], f["x1"])
        else:
            finales.append(dict(f))
    for c in cols:
        if c["LOTE"] in ("SUBTOTAL", "TOTAL"):
            c["OPERADOR"] = ""
            continue
        dentro = [f for f in finales if f["x0"] - ancho / 2 <= c["c"] <= f["x1"] + ancho / 2]
        c["OPERADOR"] = min(dentro, key=lambda f: abs((f["x0"] + f["x1"]) / 2 - c["c"]))["txt"] if dentro else ""
    # operador de la cabecera sin columna de lote debajo (mayo 2026, m3: «LATIN AMERICAN ENERGY» sin «Lote VI.»):
    # se le crea su columna para no perder sus valores; el lote sale de LOTE_DE_OPERADOR o queda «SIN ROTULO»
    xs = [c["c"] for c in cols]
    for f in finales:
        fc = (f["x0"] + f["x1"]) / 2
        if min(abs(fc - x) for x in xs) > ancho * 0.6 and min(xs) < fc < max(xs):
            izq = max((c for c in cols if c["c"] < fc), key=lambda c: c["c"])
            nombre_op = re.sub(r"\s+", " ", f["txt"]).strip()
            clave = next((k for k in LOTE_DE_OPERADOR if _sin_tildes(nombre_op).split()[0] == k.split()[0]), None)
            lote = LOTE_DE_OPERADOR[clave] if clave else "SIN ROTULO"
            nombre_op = clave or nombre_op         # «LATIN ENERGY» (el «AMERICAN» cae en el renglón de lotes) → nombre completo
            cols.append({"LOTE_ORIGINAL": f"(sin rótulo: {nombre_op})", "LOTE": lote, "c": fc, "top": f["top"],
                         "PRODUCTO": izq["PRODUCTO"] if izq["LOTE"] != "SUBTOTAL" else "LGN", "OPERADOR": nombre_op})
            xs.append(fc)
    cols.sort(key=lambda c: c["c"])
    # 5. filas de días: los valores van alineados a la derecha de su celda
    bordes = [c["c"] + ancho / 2 for c in cols]
    filas = []
    x_min = cols[0]["c"] - ancho / 2
    for top, lw in lineas:
        if top <= top_l + 3:
            continue
        txt0 = _sin_tildes(lw[0]["text"])
        if txt0.startswith(("ACUMULADO", "TOTAL", "PROMEDIO", "OBSERVACIONES")):
            break
        dia = next((w for w in lw if w["x1"] < x_min + 2 and re.fullmatch(r"\d{1,2}", w["text"])), None)
        nums = [w for w in lw if w["x1"] >= x_min and NUMERO.match(w["text"])]
        if not nums:
            continue
        if dia is None:                       # en gas el n.º de día queda en otra línea, 1 pt más abajo
            dia = next((w for t2, l2 in lineas if 0 < abs(t2 - top) <= 3 for w in l2
                        if w["x1"] < x_min + 2 and re.fullmatch(r"\d{1,2}", w["text"])), None)
        if dia is None or not 1 <= int(dia["text"]) <= 31:
            continue
        valores = {}
        for w in nums:
            k = min(range(len(cols)), key=lambda i: abs(bordes[i] - w["x1"]))
            if abs(bordes[k] - w["x1"]) <= ancho / 2:
                valores[k] = _num(w["text"])
        filas.append({"DIA": int(dia["text"]), "valores": valores})
    return {**cab, "columnas": cols, "dias": filas}


def extraer_diario(path):
    """Lista de páginas leídas (un PDF de gas trae Mcf y MMBtu en páginas distintas)."""
    with pdfplumber.open(path) as pdf:
        return [p for p in (leer_pagina(pg) for pg in pdf.pages) if p]


def a_filas(pag, fuente=""):
    """Filas largas (día × lote) y control de cada día (suma de lotes vs SUB TOTAL / TOTAL del PDF)."""
    cols, filas, ctrl = pag["columnas"], [], []
    tol = 1.0 if pag["UNIDAD"] == "BBL" else 0.05
    for d in pag["dias"]:
        v = d["valores"]
        fecha = f"{d['DIA']:02d}/{pag['MES']:02d}/{pag['AÑO']}"
        lotes = [i for i, c in enumerate(cols) if c["LOTE"] not in ("SUBTOTAL", "TOTAL")]
        for i in lotes:
            c = cols[i]
            filas.append({"FECHA": fecha, "AÑO": pag["AÑO"], "MES": pag["MES"], "DIA": d["DIA"],
                          "FLUIDO": pag["FLUIDO"], "PRODUCTO": c["PRODUCTO"], "UNIDAD": pag["UNIDAD"],
                          "OPERADOR": c["OPERADOR"], "LOTE_ORIGINAL": c["LOTE_ORIGINAL"], "LOTE": c["LOTE"],
                          "VALOR": v.get(i, 0.0), "FUENTE": fuente})
        # control: cada SUB TOTAL contra los lotes de su producto; el TOTAL contra todos los lotes
        dif_max = 0.0
        for i, c in enumerate(cols):
            if c["LOTE"] == "SUBTOTAL" and i in v:
                s = sum(v.get(j, 0.0) for j in lotes if cols[j]["PRODUCTO"] == c["PRODUCTO"])
                dif_max = max(dif_max, abs(s - v[i]))
            if c["LOTE"] == "TOTAL" and i in v:
                dif_max = max(dif_max, abs(sum(v.get(j, 0.0) for j in lotes) - v[i]))
        ctrl.append({"FECHA": fecha, "FLUIDO": pag["FLUIDO"], "UNIDAD": pag["UNIDAD"],
                     "CELDAS": len(v), "DIF_MAX": round(dif_max, 4), "ESTADO": "OK" if dif_max <= tol else "REVISAR"})
    return filas, ctrl


# ── Tablas adicionales de los anuales: pozos perforados, regalías, canon/sobrecanon y precio de canasta ──

NUM_X = re.compile(r"^-?\d{1,3}([,.]\d{3})*([.,]\d+)?$|^-?\d+([.,]\d+)?$")
ANIO = re.compile(r"^(19|20)\d\d$")
ZONAS_TXT = {"NOR-OESTE", "NOROESTE", "NOR", "OESTE", "SELVA", "NORTE", "CENTRAL", "SUR", "ZOCALO", "ZÓCALO", "COSTA"}


def _num(t, miles_punto=False):
    t = t.strip()
    if t in ("-", "–", ""):
        return 0.0
    if miles_punto:
        t = t.replace(".", "").replace(",", ".")
    else:
        t = t.replace(",", "")
    return float(t)


def _miles_punto(ws):
    """¿Números con coma decimal ('8,17', '1.234,56')? (el anual 2001 usa ese formato)."""
    coma = sum(1 for w in ws if re.fullmatch(r"\d{1,3}(\.\d{3})*,\d{1,4}", w["text"]))
    punto = sum(1 for w in ws if re.fullmatch(r"\d{1,3}(,\d{3})*\.\d{1,4}", w["text"]))
    return coma > punto


def lote_px(t):
    """lote_norm más variantes de estas tablas: 'VII-VI' → 'VII/VI'."""
    t = re.sub(r"^31-?([A-Z])-31-?([A-Z])$", r"31-\1/\2", lote_norm(t.replace("_", "/")))
    t = re.sub(r"^31([A-E])$", r"31-\1", t)                       # "31C" → "31-C"
    return re.sub(r"^([IVXL]+)-([IVXL]+)$", r"\1/\2", t)


def separar_etiqueta(txt):
    """'OIG(4) X' → ('OIG', 'X'); 'UPLAND OIL AND GAS 8' → ('UPLAND OIL AND GAS', '8'); 'NOR-OESTE OLYMPIC XIII' → ('OLYMPIC', 'XIII')."""
    t = re.sub(r"\(\d\)", " ", txt)
    toks = [x for x in t.split() if sin_tildes(x) not in ZONAS_TXT]
    for n in (3, 2, 1):                      # el lote va al final: 'VII / VI', 'Z - 2B', '31-B/D', '192'
        if len(toks) >= n:
            cand = lote_px(" ".join(toks[-n:]))
            if LOTE_OK.match(cand):
                return " ".join(toks[:-n]).strip(" -:"), cand
    return " ".join(toks).strip(), ""


def asignar_lotes(filas_y, lotes_y):
    """Las celdas de lote combinadas abarcan varias filas y el texto queda centrado verticalmente en el bloque:
    reparte las filas (en orden) en bloques contiguos, uno por lote, cuyo centro quede lo más cerca de cada etiqueta."""
    n, m = len(filas_y), len(lotes_y)
    if not m or not n:
        return [None] * n
    if m > n:
        lotes_y = lotes_y[:n]
        m = n
    INF = float("inf")
    costo = [[INF] * (n + 1) for _ in range(m + 1)]
    corte = [[0] * (n + 1) for _ in range(m + 1)]
    costo[0][0] = 0.0
    for j in range(1, m + 1):
        for i in range(j, n + 1):
            for k in range(j - 1, i):          # filas k..i-1 → lote j-1
                if costo[j - 1][k] == INF:
                    continue
                c = costo[j - 1][k] + (sum(filas_y[k:i]) / (i - k) - lotes_y[j - 1]) ** 2
                if c < costo[j][i]:
                    costo[j][i], corte[j][i] = c, k
    out, i = [None] * n, n
    for j in range(m, 0, -1):
        k = corte[j][i]
        for r in range(k, i):
            out[r] = j - 1
        i = k
    return out


def tabla_por_columnas(pg, es_columna, tol=3.6, ws=None, requiere=None, modo_lote=True, num=NUM_X, corta_grafico=False,
                       etiquetas_lejos=False, para_en_total=True):
    """Lee una tabla cuyo encabezado es una línea con 4+ columnas reconocidas por `es_columna(texto)` (años, meses…).
    Devuelve (columnas, filas) con filas = [{"ETIQUETA", "OPERADOR", "LOTE", "TIPO_FILA", "valores": {columna: número}}];
    TIPO_FILA es DATO, SUBTOTAL o TOTAL. Si el encabezado tiene columna LOTE, las celdas de lote combinadas
    (un lote con varios operadores en el período) se reparten con `asignar_lotes`."""
    ws = ws if ws is not None else sorted(palabras(pg), key=lambda w: (round(w["top"]), w["x0"]))
    lin = lineas(ws, tol=tol)
    for i, (top, lw) in enumerate(lin):
        cols = [(w["text"], (w["x0"] + w["x1"]) / 2) for w in lw if es_columna(w["text"])]
        if len(cols) >= 4 and (not requiere or any(re.match(requiere, sin_tildes(w["text"])) for w in lw)):
            break
    else:
        return None, []
    tot = [w for w in lw if sin_tildes(w["text"]) in ("TOTAL", "PROMEDIO") and (w["x0"] + w["x1"]) / 2 > cols[-1][1]]
    if tot:
        cols.append((sin_tildes(tot[0]["text"]), (tot[0]["x0"] + tot[0]["x1"]) / 2))
    paso = (cols[-1][1] - cols[0][1]) / max(1, len(cols) - 1)
    x_min, x_max = cols[0][1] - paso * 0.6, cols[-1][1] + paso * 0.6
    x_izq = 0 if etiquetas_lejos else min(w["x0"] for w in lw) - 30   # fuera de esto: número de página, textos al margen
    w_lote = next((w for w in lw if sin_tildes(w["text"]) == "LOTE" and w["x1"] <= x_min), None) if modo_lote else None
    x_lote = w_lote["x0"] - 12 if w_lote else None
    mp = _miles_punto(ws)
    filas, sueltos, ops = [], [], []                      # etiquetas de lote / operador en líneas sin números
    for top, l in lin[i + 1:]:
        l = [w for w in l if x_izq <= w["x0"] and w["x1"] <= x_max + 4]
        txt = sin_tildes(" ".join(w["text"] for w in l))
        if re.match(r"^\(\d\)|^NOTA|^FUENTE|^PAGINA|^\* ", txt):
            break
        izq = [w for w in l if (w["x1"] <= x_min + 2 or (w["x0"] < x_min and not num.match(w["text"])))
               and sin_tildes(w["text"]) not in ZONAS_TXT]
        der = [w for w in l if w["x1"] > x_min + 2 and (num.match(w["text"]) or w["text"] in "-–")]
        if corta_grafico and filas and not der and izq and all(NUM_X.match(w["text"]) for w in izq):
            break                                         # empieza el eje de un gráfico debajo del cuadro
        if x_lote is not None:
            lote_w = [w for w in izq if w["x0"] >= x_lote]
            izq = [w for w in izq if w["x0"] < x_lote]
            if lote_w:
                lt = lote_px(" ".join(w["text"] for w in lote_w))
                if LOTE_OK.match(lt):
                    sueltos.append((min(w["top"] for w in lote_w), lt))
        if not der:
            if izq and filas and re.search(r"(/|\bY)$", filas[-1]["ETIQUETA"]):   # etiqueta partida: '31B-31D /' + '31C'
                filas[-1]["ETIQUETA"] += " " + " ".join(w["text"] for w in izq)
                if x_lote is None:
                    filas[-1]["OPERADOR"], filas[-1]["LOTE"] = separar_etiqueta(filas[-1]["ETIQUETA"])
                continue
            t_op = " ".join(w["text"] for w in izq).strip()
            if izq and x_lote is not None and re.search(r"[A-Z]{2}", sin_tildes(t_op)):
                ops.append((min(w["top"] for w in izq), t_op))
            continue
        vals = {}
        for w in der:
            xc = (w["x0"] + w["x1"]) / 2
            c = min(cols, key=lambda c: abs(c[1] - xc))
            if abs(c[1] - xc) <= paso * 0.6:
                try:
                    vals[c[0]] = _num(w["text"], mp)
                except ValueError:
                    pass
        etiqueta = " ".join(w["text"] for w in izq).strip()
        et = sin_tildes(etiqueta)
        tipo = "SUBTOTAL" if re.match(r"^SUB-?\s*TOTAL", et) else "TOTAL" if re.match(r"^TOTAL", et) else "DATO"
        if x_lote is None:
            op, lote = separar_etiqueta(etiqueta)
        else:
            op, lote = re.sub(r"\(\d\)", "", etiqueta).strip(), ""
        y = sum(w["top"] for w in der) / len(der)
        filas.append({"ETIQUETA": etiqueta, "OPERADOR": op, "LOTE": lote, "TIPO_FILA": tipo, "valores": vals, "_y": y})
        if tipo == "TOTAL" and para_en_total:
            break
    for f in filas:                                       # operador en celda combinada (en otra línea, sin números)
        if f["TIPO_FILA"] == "DATO" and not re.search(r"[A-Z]{2}", sin_tildes(f["OPERADOR"])) and ops:
            y, t_op = min(ops, key=lambda o: abs(o[0] - f["_y"]))
            if abs(y - f["_y"]) <= 14:
                f["OPERADOR"] = t_op
    if x_lote is not None:
        # cada sección (entre subtotales) reparte sus filas entre las etiquetas de lote que caen en ella
        ini = 0
        for k in range(len(filas) + 1):
            if k == len(filas) or filas[k]["TIPO_FILA"] != "DATO":
                bloque = filas[ini:k]
                if bloque:
                    y0 = bloque[0]["_y"] - 8
                    y1 = bloque[-1]["_y"] + 8
                    etq = sorted(e for e in sueltos if y0 <= e[0] <= y1)
                    asig = asignar_lotes([f["_y"] for f in bloque], [e[0] for e in etq])
                    for f, j in zip(bloque, asig):
                        f["LOTE"] = etq[j][1] if j is not None else ""
                ini = k + 1
    for f in filas:
        f.pop("_y")
    return [c[0] for c in cols], filas


_TEXTOS = {}


def textos_pdf(path):
    """Texto de cada página leído con pypdf (rápido), con las líneas también al revés por las páginas giradas."""
    if path not in _TEXTOS:
        import pypdf
        out = []
        for p in pypdf.PdfReader(path).pages:
            try:
                t = sin_tildes(p.extract_text() or "")
            except Exception:
                t = ""
            out.append(" ".join((t + " " + " ".join(l[::-1] for l in t.splitlines())).split()))
        _TEXTOS[path] = out
    return _TEXTOS[path]


def paginas_con(path, patron):
    """Índices de las páginas cuyo texto cumple `patron`."""
    return [i for i, t in enumerate(textos_pdf(path)) if re.search(patron, t)]


# ── Pozos perforados por lote (series de 10–11 años de cada anual) ─────────────
TIPOS_POZO = [("EXPLORATORIO", r"POZOS EXPLORATORIOS PERFORADOS POR CONTRATO"),
              ("CONFIRMATORIO", r"POZOS CONFIRMATORIOS PERFORADOS POR CONTRATO"),
              ("DESARROLLO", r"POZOS DE DESARROLLO PERFORADOS POR CONTRATO")]


def pozos_por_lote(path):
    """Filas {TIPO, AÑO, OPERADOR, LOTE, POZOS} de las tablas «POZOS … PERFORADOS POR CONTRATO aaaa - aaaa»
    y control {TIPO, AÑO, SUMA_LOTES, TOTAL_PDF} contra la fila TOTAL."""
    filas, control = [], []
    with pdfplumber.open(path) as pdf:
        for tipo, rx in TIPOS_POZO:
            for ip in paginas_con(path, rx + r"\s*\(?\s*(19|20)\d\d\s*-\s*(19|20)\d\d"):
                cols, fs = tabla_por_columnas(pdf.pages[ip], lambda t: bool(ANIO.match(t)))
                if not cols:
                    continue
                anios = [c for c in cols if ANIO.match(c)]
                tot = next((f for f in fs if f["TIPO_FILA"] == "TOTAL"), None)
                lotes = [f for f in fs if f["TIPO_FILA"] == "DATO"]
                for f in lotes:
                    for a in anios:
                        v = f["valores"].get(a, 0.0)
                        filas.append({"TIPO": tipo, "AÑO": int(a), "OPERADOR": f["OPERADOR"], "LOTE": f["LOTE"],
                                      "POZOS": int(round(v)), "PAGINA": ip + 1})
                for a in anios:
                    control.append({"TIPO": tipo, "AÑO": int(a), "SUMA_LOTES": int(round(sum(f["valores"].get(a, 0) for f in lotes))),
                                    "TOTAL_PDF": int(round(tot["valores"].get(a, 0))) if tot else None, "PAGINA": ip + 1})
        if not filas:                                     # anual 2005: tabla traspuesta en varias páginas
            acum, pags = {}, []
            for ip in paginas_con(path, r"NUMERO DE POZOS PERFORADOS POR CONTRATO\s*\(\s*(19|20)\d\d\s*-"):
                fs, ac = pozos_traspuesto(pdf.pages[ip])
                filas += [{**f, "PAGINA": ip + 1} for f in fs]
                acum.update(ac)
                pags.append(ip + 1)
            for (tipo, anio), v in sorted(acum.items()):
                control.append({"TIPO": tipo, "AÑO": anio, "SUMA_LOTES": sum(f["POZOS"] for f in filas if f["TIPO"] == tipo and f["AÑO"] == anio),
                                "TOTAL_PDF": v, "PAGINA": "-".join(map(str, pags))})
    return filas, control


# ── Pozos por lote, formato traspuesto del anual 2005 (lotes en columnas, años en filas; 1995–2005) ─────────
def rects_lectura(pg):
    """Rectángulos (bordes de celdas) en las mismas coordenadas "de lectura" que `palabras` (páginas giradas)."""
    chars = pg.chars
    girados = [c for c in chars if not c.get("upright", True)]
    if len(girados) <= len(chars) / 2:
        return [{"x0": r["x0"], "x1": r["x1"], "top": r["top"], "bottom": r["bottom"]} for r in pg.rects]
    if sum(1 for c in girados if c["matrix"][1] > 0) >= len(girados) / 2:
        return [{"x0": pg.height - r["bottom"], "x1": pg.height - r["top"], "top": r["x0"], "bottom": r["x1"]} for r in pg.rects]
    return [{"x0": r["top"], "x1": r["bottom"], "top": pg.width - r["x1"], "bottom": pg.width - r["x0"]} for r in pg.rects]


def pozos_traspuesto(pg):
    """Una página «NUMERO DE POZOS PERFORADOS POR CONTRATO (1995 - 2005)»: los bordes de las celdas definen las
    columnas (un lote cada una; el operador ocupa la celda combinada de arriba). Devuelve (filas, acumulado)
    con filas = [{TIPO, AÑO, OPERADOR, LOTE, POZOS}] y acumulado = {(TIPO, AÑO): total} si la página trae esa columna."""
    ws = sorted(palabras(pg), key=lambda w: (round(w["top"]), w["x0"]))
    lin = lineas(ws, tol=3.6)
    i_lote = next((i for i, (t, l) in enumerate(lin)
                   if sum(1 for w in l if LOTE_OK.match(lote_px(w["text"]))) >= 4), None)
    if i_lote is None:
        return [], {}
    y_lote = lin[i_lote][0]
    bordes = [r for r in rects_lectura(pg) if r["x1"] - r["x0"] < 3 and r["top"] <= y_lote + 2 and r["bottom"] >= y_lote + 4]
    xs = sorted({round((r["x0"] + r["x1"]) / 2) for r in bordes})
    xs = [x for k, x in enumerate(xs) if k == 0 or x - xs[k - 1] > 3]
    if len(xs) < 5:
        return [], {}
    y_op = min(r["top"] for r in bordes)                     # inicio de la franja de operadores
    altos = sorted({round((r["x0"] + r["x1"]) / 2) for r in bordes if r["top"] <= y_op + 3})
    centro = lambda w: (w["x0"] + w["x1"]) / 2
    cab = [w for w in ws if y_op - 12 <= w["top"] < y_lote - 3]
    columnas = []
    for a, b in zip(xs, xs[1:]):
        lote = " ".join(w["text"] for w in lin[i_lote][1] if a < centro(w) < b)
        oa = max([x for x in altos if x <= a + 1] or [a])
        ob = min([x for x in altos if x >= b - 1] or [b])
        op = " ".join(w["text"] for w in sorted(cab, key=lambda w: (round(w["top"]), w["x0"])) if oa < centro(w) < ob)
        op = re.sub(r"\s*\(\*\)", "", op).strip()
        if sin_tildes(op).startswith("ANO"):
            continue
        columnas.append({"a": a, "b": b, "LOTE": lote_px(lote) if lote else "", "OPERADOR": op})
    filas, acum, tipo = [], {}, None
    for top, l in lin[i_lote + 1:]:
        t0 = sin_tildes(l[0]["text"])
        if t0.startswith("EXPLORATORIO"):
            tipo = "EXPLORATORIO"
            continue
        if t0.startswith("DESARROLLO"):
            tipo = "DESARROLLO"
            continue
        if not tipo or not ANIO.match(l[0]["text"]):
            continue
        anio = int(l[0]["text"])
        for w in l[1:]:
            if not NUM_X.match(w["text"]):
                continue
            c = next((c for c in columnas if c["a"] < centro(w) < c["b"]), None)
            if c is None:
                continue
            v = int(_num(w["text"]))
            if not c["LOTE"] and re.search(r"ACUMULADO|TOTAL", sin_tildes(c["OPERADOR"])):
                acum[(tipo, anio)] = v
            elif c["LOTE"]:
                filas.append({"TIPO": tipo, "AÑO": anio, "OPERADOR": c["OPERADOR"], "LOTE": c["LOTE"], "POZOS": v})
    return filas, acum


# ── Regalías (ingresos al fisco) por lote e hidrocarburo ─────────────────────
PRODUCTOS_REG = {"PETROLEO": "PETRÓLEO", "LGN": "LGN", "GAS": "GAS NATURAL", "CONDENSADO": "CONDENSADO"}


def _monto(tokens):
    """'2' + '4,395,965.48' → 24395965.48 (los montos vienen partidos en dos palabras)."""
    t = "".join(tokens).replace(" ", "")
    return float(t.replace(",", "")) if re.fullmatch(r"\d[\d,]*(\.\d+)?", t) else None


def regalias_pagina(pg):
    """Tabla «INGRESOS AL FISCO - aaaa (US$)» / «INGRESO POR REGALÍAS POR CONTRATO - aaaa»: filas
    {MODALIDAD, HIDROCARBURO, LOTE, LOTE_PDF, REGALIA_US$} y el TOTAL del cuadro. Modalidad e hidrocarburo son
    celdas combinadas (texto centrado en su bloque de filas): se reparten con `asignar_lotes`."""
    ws = sorted(palabras(pg), key=lambda w: (round(w["top"]), w["x0"]))
    lin = lineas(ws, tol=3.6)
    i0 = next((i for i, (t, l) in enumerate(lin) if any(sin_tildes(w["text"]) == "LOTE" for w in l)
               and any(sin_tildes(w["text"]).startswith("REGALIA") for w in l)), None)
    if i0 is None:
        return [], None
    cab = lin[i0][1]
    x_lote = next(w for w in cab if sin_tildes(w["text"]) == "LOTE")["x0"] - 40
    x_hc = next((w["x0"] for w in cab if sin_tildes(w["text"]).startswith("HIDROCARBURO")), x_lote - 100) - 40
    x_reg = next(w for w in cab if sin_tildes(w["text"]).startswith("REGALIA"))["x0"] - 12
    x_pct = next((w["x0"] - 12 for w in cab if sin_tildes(w["text"]).startswith("PORCENTAJE")), x_reg)   # anual 2007
    filas, etq_mod, etq_hc, total, sin_lote, rotulos = [], [], [], None, [], []
    for top, l in lin[i0 + 1:]:
        txt = sin_tildes(" ".join(w["text"] for w in l))
        nums = [w for w in l if w["x0"] > x_reg and re.fullmatch(r"[\d,.]+", w["text"]) and re.search(r"\d", w["text"])]
        if re.match(r"^TOTAL", txt) or re.match(r"^TOTAL", sin_tildes(" ".join(w["text"] for w in l if w["x0"] > x_hc))):
            total = _monto([w["text"] for w in nums])
            break
        for w in l:
            t = sin_tildes(w["text"])
            if w["x0"] < x_hc and t in ("LICENCIA", "SERVICIOS"):
                etq_mod.append((w["top"], t))
            elif x_hc <= w["x0"] < x_lote and t in PRODUCTOS_REG:
                etq_hc.append((w["top"], PRODUCTOS_REG[t]))
        if "SUBTOTAL" in txt or "SUB-TOTAL" in txt:
            continue
        lw = [w for w in l if x_lote <= w["x0"] < x_pct]
        pct = [w for w in l if x_pct <= w["x0"] < x_reg and re.fullmatch(r"\d+(\.\d+)?", w["text"])]
        if nums and not lw:                               # monto sin lote: su rótulo va en otra línea (2015: 'III (T) Y III (*)')
            m_ = _monto([w["text"] for w in nums])
            if m_ is not None:
                sin_lote.append({"_y": sum(w["top"] for w in nums) / len(nums), "REGALIA_US$": m_,
                                 "REGALIA_PCT": float(pct[0]["text"]) if pct else None})
            continue
        if lw and not nums:
            rotulos.append((min(w["top"] for w in lw), " ".join(w["text"] for w in lw)))
            continue
        if not nums or not lw:
            continue
        # el lote es lo que va antes del primer número (también '1-AB (*)', 'I (TT)', '31 - E')
        nums_der = nums
        monto = _monto([w["text"] for w in nums_der])
        lote_pdf = " ".join(w["text"] for w in lw)
        base = re.sub(r"\(\*+\)|\(T{1,2}\)", "", lote_pdf)
        lote = lote_px(base)
        if monto is None or not LOTE_OK.match(lote):
            continue
        filas.append({"LOTE": lote, "LOTE_PDF": lote_pdf.strip(), "REGALIA_US$": monto,
                      "REGALIA_PCT": float(pct[0]["text"]) if pct else None,
                      "_y": sum(w["top"] for w in nums_der) / len(nums_der)})
    # cada rótulo suelto se reparte entre los montos sin lote más cercanos (arriba y abajo), en orden
    for y, txt in rotulos:
        cerca = sorted((f for f in sin_lote if abs(f["_y"] - y) <= 14 and "LOTE" not in f), key=lambda f: f["_y"])
        partes = [p for p in re.split(r"\s+Y\s+", txt, flags=re.I) if p.strip()]
        if not cerca:
            continue
        if len(partes) != len(cerca):
            partes = [txt] * len(cerca)
        for f, p in zip(cerca, partes):
            lote = lote_px(re.sub(r"\(\*+\)|\(T{1,2}\)", "", p))
            if LOTE_OK.match(lote):
                f.update({"LOTE": lote, "LOTE_PDF": p.strip()})
    filas += [f for f in sin_lote if "LOTE" in f]
    filas.sort(key=lambda f: f["_y"])
    ys = [f["_y"] for f in filas]
    for etq, campo in ((etq_mod, "MODALIDAD"), (etq_hc, "HIDROCARBURO")):
        if campo == "HIDROCARBURO" and etq_mod:
            # dentro de cada modalidad
            grupos = {}
            for k, f in enumerate(filas):
                grupos.setdefault(f["MODALIDAD"], []).append(k)
            for g, ks in grupos.items():
                yk = [ys[k] for k in ks]
                e = sorted(x for x in etq if min(yk) - 8 <= x[0] <= max(yk) + 8)
                for k, j in zip(ks, asignar_lotes(yk, [x[0] for x in e])):
                    filas[k][campo] = e[j][1] if j is not None else ""
        else:
            e = sorted(etq)
            for f, j in zip(filas, asignar_lotes(ys, [x[0] for x in e])):
                f[campo] = e[j][1] if j is not None else ("LICENCIA" if campo == "MODALIDAD" else "")
    for f in filas:
        f.pop("_y")
    return filas, total


def regalias_dos_columnas(pg):
    """Anuales 2001–2006: «INGRESOS POR CONTRATOS aaaa (MUS$)» en dos columnas de página (petróleo / LGN a la
    izquierda, gas / condensados a la derecha), por modalidad. Montos en miles de US$ → se pasan a US$."""
    ws = sorted(palabras(pg), key=lambda w: (round(w["top"]), w["x0"]))
    lin = lineas(ws, tol=3.6)
    x_med = next((w["x0"] - 8 for top, l in lin for w in l if sin_tildes(w["text"]) == "GAS:" and w["x0"] > 200), None)
    if x_med is None:
        return [], None
    textos = [w["text"] for w in ws]
    miles_punto = not any(re.fullmatch(r"\d{1,3}(,\d{3})+", t) for t in textos)
    prod_rx = {"PETROLEO:": "PETRÓLEO", "GAS:": "GAS NATURAL", "CONDENSADOS:": "CONDENSADO", "CONDENSADO:": "CONDENSADO", "LGN:": "LGN"}
    filas, total, modalidad, subtotales = [], None, "LICENCIA", []
    producto = {0: None, 1: None}
    for top, l in lin:
        txt = sin_tildes(" ".join(w["text"] for w in l))
        if "CONTRATOS DE LICENCIA" in txt:
            modalidad = "LICENCIA"
        elif "CONTRATOS DE SERVICIOS" in txt:
            modalidad = "SERVICIOS"
        if re.match(r"^TOTAL:", txt):
            total = _num(l[-1]["text"], miles_punto) * 1000
            continue
        if re.search(r"SUB\s*-?\s*TOTAL:", txt):         # 2004 no trae TOTAL general: se suman los subtotales
            subtotales.append(_num(l[-1]["text"], miles_punto) * 1000)
            continue
        for lado, mitad in ((0, [w for w in l if w["x0"] < x_med]), (1, [w for w in l if w["x0"] >= x_med])):
            if not mitad:
                continue
            t0 = sin_tildes(mitad[0]["text"])
            if t0 in prod_rx:
                producto[lado] = prod_rx[t0]
                continue
            if t0 in ("LOTE", "TOTAL", "SUB", "(*)") or not producto[lado]:
                continue
            ult = mitad[-1]["text"]
            if len(mitad) < 2 or not re.fullmatch(r"\d[\d.,]*", ult):
                continue
            lote_pdf = " ".join(w["text"] for w in mitad[:-1])
            lote = lote_px(re.sub(r"\(\*+\)", "", lote_pdf))
            if not LOTE_OK.match(lote):
                continue
            v = float(ult.replace(",", "")) if not miles_punto or not re.fullmatch(r"\d{1,3}(\.\d{3})+", ult) \
                else float(ult.replace(".", ""))
            filas.append({"MODALIDAD": modalidad, "HIDROCARBURO": producto[lado], "LOTE": lote, "LOTE_PDF": lote_pdf,
                          "REGALIA_US$": v * 1000, "REGALIA_PCT": None})
    return filas, (total if total is not None else (sum(subtotales) if subtotales else None))


def regalias(path):
    """Regalías del año de cada anual: filas {AÑO, MODALIDAD, HIDROCARBURO, LOTE, LOTE_PDF, REGALIA_US$, PAGINA}
    y control {AÑO, SUMA_LOTES, TOTAL_PDF, PAGINA}."""
    filas, control = [], []
    with pdfplumber.open(path) as pdf:
        rx = r"INGRESOS? (?:AL FISCO|POR REGALIAS|DE REGALIAS|POR CONTRATOS)[^0-9]{0,40}((?:19|20)\d\d)"
        for ip in paginas_con(path, rx):
            t = textos_pdf(path)[ip]
            m = re.search(rx, t)
            if "POR CONTRATOS" in m.group(0):
                fs, tot = regalias_dos_columnas(pdf.pages[ip])
            else:
                fs, tot = regalias_pagina(pdf.pages[ip])
            if not fs:
                continue
            anio = int(m.group(1))
            filas += [{"AÑO": anio, **f, "PAGINA": ip + 1} for f in fs]
            control.append({"AÑO": anio, "SUMA_LOTES": round(sum(f["REGALIA_US$"] for f in fs), 2),
                            "TOTAL_PDF": tot, "PAGINA": ip + 1})
    return filas, control


# ── Canon y sobrecanon por departamento ──────────────────────────────────────
def canon(path):
    """«DISTRIBUCIÓN DEL CANON / SOBRECANON aaaa - aaaa (MILLONES DE SOLES)»: filas {AÑO, DEPARTAMENTO, CANON, UNIDAD}
    y control {AÑO, SUMA_DEPARTAMENTOS, TOTAL_PDF}."""
    filas, control = [], []
    with pdfplumber.open(path) as pdf:
        for ip in paginas_con(path, r"DISTRIBUCION DEL CANON\s*/\s*SOBRECANON"):
            t = textos_pdf(path)[ip]
            m = re.search(r"\((MILLONES|MILES)[A-Z ]*SOLES\)", t)
            unidad = (m.group(1) + " DE SOLES") if m else ""
            cols, fs = tabla_por_columnas(pdf.pages[ip], lambda x: bool(ANIO.match(x)), requiere=r"DEPARTAMENTO|REGION")
            if not cols:
                continue
            anios = [c for c in cols if ANIO.match(c)]
            tot = next((f for f in fs if f["TIPO_FILA"] == "TOTAL"), None)
            deps = [f for f in fs if f["TIPO_FILA"] == "DATO" and f["ETIQUETA"]]
            for f in deps:
                for a in anios:
                    if a in f["valores"]:
                        filas.append({"AÑO": int(a), "DEPARTAMENTO": sin_tildes(f["ETIQUETA"]), "CANON": f["valores"][a],
                                      "UNIDAD": unidad, "PAGINA": ip + 1})
            for a in anios:
                control.append({"AÑO": int(a), "SUMA_DEPARTAMENTOS": round(sum(f["valores"].get(a, 0) for f in deps), 2),
                                "TOTAL_PDF": tot["valores"].get(a) if tot else None, "UNIDAD": unidad, "PAGINA": ip + 1})
    return filas, control


# ── Precio de la canasta de hidrocarburos líquidos por lote y mes ─────────────
PRECIO_X = re.compile(r"^\d{1,3}[.,]\d{2,4}$")      # precios con decimales: descarta textos superpuestos ('6677866778')


def _lotes_canasta(etiqueta):
    """'VII. Y V.' → [('VII',''), ('V','')]; 'Z-69-L' → [('Z-69','L')]; 'Z-2B(LGN)' → [('Z-2B','LGN')];
    '31B-31D / 31C' → [('31-B/D',''), ('31-C','')]; 'VII / VI' → [('VII/VI','')]."""
    et = re.sub(r"\(\*+\)|\(\d\)", "", sin_tildes(etiqueta))
    et = re.sub(r"^(LICENCIA|SERVICIOS|CONTRATO)\s+", "", et).strip()
    var = "LGN" if re.search(r"\(\s*LGN\s*\)", et) else ""
    et = re.sub(r"\(\s*LGN\s*\)", "", et)
    def uno(p):
        p = re.sub(r"\(T{1,2}\)", "", p).strip(" .")
        mv = re.match(r"^(.*?)-([LP])$", p)
        if mv and LOTE_OK.match(lote_px(mv.group(1))):
            return lote_px(mv.group(1)), mv.group(2)
        l = lote_px(p)
        return (l, var) if LOTE_OK.match(l) else None
    entero = uno(et)
    if entero:
        return [entero]
    return [x for x in (uno(p) for p in re.split(r"\s+Y\s+|\s*,\s*|\s+/\s+", et)) if x]


def canasta_traspuesta(pg):
    """Anuales antiguos (2001–2007): meses en filas y lotes en columnas ('MES | LOTE 1-AB | LOTE 31 B/D | …'), a veces
    en dos bloques (licencia / servicios). Las columnas de valores se agrupan por posición y se asignan en orden a
    las de la cabecera; se omiten las de gas natural. Devuelve [(lote, variante, etiqueta, {mes: precio}, promedio)]."""
    ws = sorted(palabras(pg), key=lambda w: (round(w["top"]), w["x0"]))
    lin = lineas(ws, tol=3.6)
    x_gas = None
    for top, l in lin:
        t = [sin_tildes(w["text"]) for w in l]
        if "GAS" in t and any(x.startswith("NATURAL") for x in t) and "HIDROCARBUROS" in t:
            x_gas = l[t.index("GAS")]["x0"] - 25            # el título de gas va centrado sobre sus columnas
    out = []
    cabs = [k for k, (top, l) in enumerate(lin) if sin_tildes(l[0]["text"]) == "MES" and len(l) > 3]
    for n, k in enumerate(cabs):
        cols, act = [], None
        for w in lin[k][1][1:]:
            t = sin_tildes(w["text"])
            if t in ("LOTE", "LOTES", "NOROESTE", "SELVA"):
                act = {"x0": w["x0"], "pal": [] if t in ("LOTE", "LOTES") else [t]}
                cols.append(act)
            elif act is not None:
                act["pal"].append(w["text"])
        if x_gas is not None:
            cols = [c for c in cols if c["x0"] < x_gas]
        fin = cabs[n + 1] if n + 1 < len(cabs) else len(lin)
        filas = {}
        for top, l in lin[k + 1:fin]:
            m = mes_de(l[0]["text"]) or ("PROMEDIO" if sin_tildes(l[0]["text"]) == "PROMEDIO" else None)
            if m:
                filas[m] = [w for w in l[1:] if PRECIO_X.match(w["text"]) or w["text"] in "-–"]
        xs = sorted(w["x0"] for k2, v in filas.items() if k2 != "PROMEDIO" for w in v)
        grupos = []
        for x in xs:
            if grupos and x - grupos[-1][-1] <= 15:
                grupos[-1].append(x)
            else:
                grupos.append([x])
        if x_gas is not None:
            grupos = [g for g in grupos if g[0] < x_gas + 15]
        if len(grupos) < len(cols):
            continue
        grupos = grupos[:len(cols)]                       # las de gas quedan a la derecha
        centros = [sum(g) / len(g) for g in grupos]
        for j, c in enumerate(cols):
            etiqueta = " ".join(c["pal"])
            lotes = [("NOROESTE", "")] if etiqueta == "NOROESTE" else _lotes_canasta(etiqueta)
            vals, prom = {}, None
            for m, v in filas.items():
                w = next((w for w in v if abs(w["x0"] - centros[j]) <= 15 + (8 if m == "PROMEDIO" else 0)), None)
                if w is None or w["text"] in "-–":
                    continue
                x = float(w["text"].replace(",", "."))
                if m == "PROMEDIO":
                    prom = x
                else:
                    vals[m] = x
            for lote, var in lotes:
                out.append((lote, var, etiqueta, vals, prom))
    return out


def promedio_grafico(pg):
    """Anuales 2007–2011: el cuadro no trae columna PROMEDIO, pero el gráfico de abajo imprime «PRECIO PROMEDIO» por
    lote, con los rótulos de lote en la línea de arriba. Devuelve {(lote, variante): promedio}."""
    ws = sorted(palabras(pg), key=lambda w: (round(w["top"]), w["x0"]))
    lin = lineas(ws, tol=3.6)
    out = {}
    for k, (top, l) in enumerate(lin):
        if not (sin_tildes(l[0]["text"]).startswith("PRECIO") and any(PRECIO_X.match(w["text"]) for w in l)):
            continue
        vals = [w for w in l if PRECIO_X.match(w["text"])]
        # rótulos: la línea más cercana de arriba con varios códigos de lote; las palabras pegadas forman un rótulo
        cab = next((l2 for t2, l2 in reversed(lin[max(0, k - 4):k])
                    if sum(1 for w in l2 if LOTE_OK.match(lote_px(w["text"]))) >= 4), None)
        if not cab:
            continue
        grupos = []
        for w in cab:
            if grupos and w["x0"] - grupos[-1][-1]["x1"] < 4:
                grupos[-1].append(w)
            else:
                grupos.append([w])
        for v in vals:
            g = min(grupos, key=lambda g: abs(g[0]["x0"] - v["x0"]))
            if abs(g[0]["x0"] - v["x0"]) > 14:
                continue
            for lote, var in _lotes_canasta(" ".join(w["text"] for w in g)):
                out[(lote, var)] = float(v["text"].replace(",", "."))
    return out


def canasta(path):
    """«VALOR MENSUAL DEL PRECIO DE CANASTA … aaaa (US$/bbl)»: filas {AÑO, MES, LOTE, VARIANTE, PRECIO_US$_BBL}
    (VARIANTE: L / P en Z-69, LGN en Z-2B) y control {AÑO, LOTE, PROMEDIO_CALC, PROMEDIO_PDF}."""
    filas, control = [], []
    rx = r"(VALOR(?:ES)? (?:MENSUAL )?(?:DEL PRECIO )?DE (?:LA )?CANASTA|CANASTA DE PRECIOS DE HIDROCARBUROS)"
    with pdfplumber.open(path) as pdf:
        for ip in paginas_con(path, rx):
            t = textos_pdf(path)[ip]
            m = re.search(rx + r"[^0-9]{0,90}((?:19|20)\d\d)", t)
            if not m:
                continue
            anio = int(m.group(2))
            cols, fs = tabla_por_columnas(pdf.pages[ip], lambda x: mes_de(x) is not None, modo_lote=False,
                                          num=PRECIO_X, corta_grafico=True)
            if not cols:
                for lote, var, etq, vals, prom in canasta_traspuesta(pdf.pages[ip]):
                    for m, v in vals.items():
                        filas.append({"AÑO": anio, "MES": MESES.index(m) + 1, "LOTE": lote, "VARIANTE": var,
                                      "LOTE_PDF": etq, "PRECIO_US$_BBL": v, "PAGINA": ip + 1})
                    if vals:
                        control.append({"AÑO": anio, "LOTE": lote, "VARIANTE": var, "MESES": len(vals),
                                        "PROMEDIO_CALC": round(sum(vals.values()) / len(vals), 4),
                                        "PROMEDIO_PDF": prom, "PAGINA": ip + 1,
                                        "PROMEDIO_FUENTE": "CUADRO" if prom else ""})
                continue
            meses = [c for c in cols if mes_de(c)]
            prom_g = promedio_grafico(pdf.pages[ip]) if "PROMEDIO" not in cols else {}
            for f in fs:
                if "PROMEDIO" in cols and not f["valores"].get("PROMEDIO"):
                    continue                              # restos de gráficos debajo del cuadro
                for lote, var in _lotes_canasta(f["ETIQUETA"]):
                    v_mes = {c: f["valores"][c] for c in meses if f["valores"].get(c)}
                    estado = {c: "OK" for c in v_mes}
                    prom = f["valores"].get("PROMEDIO") or prom_g.get((lote, var))
                    falta = [c for c in meses if c not in f["valores"]]
                    # un solo mes ilegible (texto superpuesto en el PDF) y promedio conocido: se completa por diferencia
                    if len(falta) == 1 and prom and len(v_mes) == 11:
                        v_mes[falta[0]] = round(12 * prom - sum(v_mes.values()), 2)
                        estado[falta[0]] = "IMPUTADO (12 × promedio − otros meses)"
                    for c in meses:
                        if c in v_mes:
                            filas.append({"AÑO": anio, "MES": MESES.index(mes_de(c)) + 1, "LOTE": lote, "VARIANTE": var,
                                          "LOTE_PDF": f["ETIQUETA"], "PRECIO_US$_BBL": v_mes[c], "ESTADO": estado[c],
                                          "PAGINA": ip + 1})
                    vals = list(v_mes.values())
                    presentes = sum(1 for c in meses if c in f["valores"] or c in v_mes)
                    # MESES_PDF: celdas de mes presentes, con precio o con «-» (el lote aún no producía): 12 = fila completa
                    control.append({"AÑO": anio, "LOTE": lote, "VARIANTE": var, "MESES": len(vals),
                                    "MESES_PDF": presentes, "IMPUTADOS": sum(1 for e in estado.values() if e != "OK"),
                                    "PROMEDIO_CALC": round(sum(vals) / len(vals), 4) if vals else None,
                                    # algunos promedios del PDF dividen entre 12 aunque un mes valga 0 (192 en 2018)
                                    "PROMEDIO_CALC_12": round(sum(vals) / presentes, 4) if presentes else None,
                                    "PROMEDIO_PDF": prom, "PAGINA": ip + 1,
                                    "PROMEDIO_FUENTE": "CUADRO" if f["valores"].get("PROMEDIO") else ("GRAFICO" if prom else "")})
    return filas, control


# ── Contratos vigentes y terminados (empresas, participación, vigencia) ────────
MES3 = {"ENE": 1, "FEB": 2, "MAR": 3, "ABR": 4, "MAY": 5, "JUN": 6, "JUL": 7, "AGO": 8,
        "SET": 9, "SEP": 9, "OCT": 10, "NOV": 11, "DIC": 12}
FECHA_C = r"\d{1,2}[-./][A-Z]{3}[-./]\d{2,4}|\d{1,2}[-./]\d{1,2}[-./]\d{2,4}"
_LOTE = r"(?P<lote>[IVXL]+ ?/ ?[IVXL]+|\d{2}[A-Z] Y \d{2}[A-Z]|\d{2,3}[A-Z]? ?/ ?\d{2,3}[A-Z]?|[A-Z0-9][A-Z0-9\-()\.]{0,11})(?: \(\*+\))?"
_ZONA = r"(?:(?:SELVA|SELVBA) (?:NORTE|CENTRAL|SUR)|NOR[- ]?OESTE|ZOCALO(?: NORTE)?|SIERRA (?:NORTE|SUR)|COSTA)"
_MOD = r"(?P<mod>LICENCIA|SERVICIOS?|OPERACIONES)"
_AREA = r"(?P<area>\d{1,3}(?:,\d{3})*\.\d+)"
FILA_VIG = re.compile(_LOTE + r" (?:" + _ZONA + r" )?(?P<s>" + FECHA_C + r")(?: \((?:" + FECHA_C + r")\))? (?:(?P<t>" + FECHA_C +
                      r")(?: ?\(\*+\))? )?" + _AREA + " " + _MOD)
FILA_EXP = re.compile(_LOTE + r" (?:" + _ZONA + r" )?" + _AREA + " " + _MOD)
FILA_TER = re.compile(_LOTE + r" (?P<s>" + FECHA_C + r") (?P<t>" + FECHA_C + r")")
SOCIO = re.compile(r"\s*/?\s*(?P<emp>[^/()]+?(?:\([^)%]*\)[^/()]*?)*)\s*\(\s*(?P<pct>\d+(?:[.,]\d+)?)\s*%\s*\)")
CABECERA = re.compile(r"FECHA DE AREA MODALIDAD SUSCRIPCION \(HAS?\.?\) CONTRATO|ZONA (?:CONTRATISTA|OPERADORA?|COMPANIAS) LOTE (?:UBICACION )?(?:FECHA DE SUSCRIPCION )?(?:FECHA DE TERMINO )?"
                      r"(?:ULTIMO DIA DE VIGENCIA )?(?:AREA \(HAS?\.?\) )?(?:MODALIDAD (?:DE )?CONTRATO)?")
RUIDO_EMP = re.compile(r"\b(" + _ZONA + r"|EXPLORACION|EXPLOTACION|TOTAL \d+[\d,.]*|"
                       r"CONTRATOS? (EN FASE DE|DE|TERMINADOS)[A-Z ]*|\(\d\)):?(?=\s|$)")


def fecha_ddmm(t):
    """'07-ABR-05' / '19-MAY-2054' / '28.2.2023' → '07/04/2005'."""
    p = re.split(r"[-./]", t.strip())
    if len(p) != 3:
        return None
    d, m, a = p
    m = MES3.get(m[:3], None) if not m.isdigit() else int(m)
    if not m:
        return None
    a = int(a)
    a = a + (1900 if a > 60 else 2000) if a < 100 else a
    return f"{int(d):02d}/{m:02d}/{a:04d}"


def socios(txt):
    """'PETROTAL PERU S.R.L. (100%)/...' → [(empresa, pct)]; sin % (ediciones antiguas) se separa por «/», con pct None."""
    txt = " ".join(txt.split()).strip(" /")
    out = [(m["emp"].strip(" /,"), float(m["pct"].replace(",", "."))) for m in SOCIO.finditer(txt)]
    if not out and txt:
        out = [(e.strip(), None) for e in txt.split("/") if e.strip()]
    return out


def _limpia_emp(t):
    t = CABECERA.sub(" ", t)
    t = " ".join(RUIDO_EMP.sub(" ", t).split())
    t = t.replace("PLU SPETROL", "PLUSPETROL")
    return re.sub(r"^(?:[-:]\s*)+", "", t).strip()


def contratos(path):
    """Contratos vigentes (explotación y exploración) y terminados en el año, de un anual.
    Filas: una por empresa socia (ESTADO, FASE, LOTE, EMPRESA, PARTICIPACION_PCT, FECHA_SUSCRIPCION, FECHA_TERMINO,
    AREA_HA, MODALIDAD, OPERADOR = 1.ª empresa listada, PAGINA). Control: lotes leídos vs «TOTAL n» de cada cuadro."""
    filas, control = [], []
    for ip, t in enumerate(textos_pdf(path)):
        n = len(t) // 2 + 50          # la mitad final es la copia al revés de las páginas giradas
        u = t[:n]
        if "SUSCRITO POR" in u or "CONVENIO" in u[:120]:
            continue
        ter = "ULTIMO DIA DE VIGENCIA" in u and "CONTRATOS TERMINADOS" in u
        vig = not ter and "MODALIDAD" in u and ("FECHA DE SUSCRIPCION" in u or "FECHA DE AREA MODALIDAD SUSCRIPCION" in u)
        exp = not ter and not vig and re.search(r"LOTE (?:UBICACION )?AREA \(HA", u) is not None
        if not (vig or exp or ter):
            continue
        sus = "CONTRATOS SUSCRITOS" in u                 # 2007–2008: cuadro de contratos firmados en el año
        estado = "TERMINADO" if ter else "SUSCRITO EN EL AÑO" if sus else "VIGENTE"
        explora = re.search(r"CONTRATOS (?:EN (?:FASE DE )?|(?:EN FASE )?DE )EXPLORACION", u)
        explota = re.search(r"CONTRATOS (?:EN (?:FASE DE )?|(?:EN FASE )?DE )EXPLOTACION", u)
        fase_txt = "EXPLORACION" if exp or (explora and not explota) else "EXPLOTACION"
        patron = FILA_TER if ter else FILA_EXP if exp else FILA_VIG
        fin = 0
        u = re.sub(r"(\d) -([A-Z]{3}-)", r"\1-\2", u)       # «27 -DIC-91»
        # cortes de pypdf dentro de una palabra o cifra: «LICE NCIA», «49, 821.139», «24 0,755.063»
        u = re.sub(r"L ?I ?C ?E ?N ?C ?I ?A\b", "LICENCIA", u)
        u = re.sub(r"(\d{1,3}), (\d{3}[.,])", r"\1,\2", u)
        u = re.sub(r"(?<= )(\d{1,2}) (\d,\d{3}\.\d{3}) (?=LICENCIA|SERVICIO|OPERACIONES)", r"\1\2 ", u)
        if ter or sus:               # el bloque EXPLORACION/EXPLOTACION cambia la fase en la misma página
            cortes = [(m.start(), m.group(0)) for m in re.finditer(r"\bEXPLORACION\b|\bEXPLOTACION\b", u)]
        lotes_pag = set()
        for m in patron.finditer(u):
            emp_txt = u[fin:m.start()]
            fin = m.end()
            fase = fase_txt
            if ter or sus:
                prev = [c for c in cortes if c[0] < m.start()]
                fase = prev[-1][1] if prev else fase_txt
            lote = lote_px(re.sub(r" ?/ ?| Y ", "/", m["lote"]))
            soc = socios(_limpia_emp(emp_txt))
            if not soc:
                continue
            lotes_pag.add(lote)
            for k, (e, pct) in enumerate(soc):
                filas.append({"ESTADO": estado, "FASE": fase, "LOTE": lote,
                              "EMPRESA": e, "PARTICIPACION_PCT": pct,
                              "FECHA_SUSCRIPCION": fecha_ddmm(m["s"]) if "s" in m.re.groupindex else None,
                              "FECHA_TERMINO": fecha_ddmm(m["t"]) if "t" in m.re.groupindex and m["t"] else None,
                              "AREA_HA": float(m["area"].replace(",", "")) if "area" in m.re.groupindex else None,
                              "MODALIDAD": m["mod"] if "mod" in m.re.groupindex else None,
                              "OPERADOR": soc[0][0], "PAGINA": ip + 1})
        if ter or sus:                # un TOTAL por fase (terminados) o por cuadro
            tots = [int(x) for x in re.findall(r"\bTOTAL (\d{1,3})(?![\d,.])", u)]
        else:                         # vigentes: «TOTAL n área»; 2018 trae un segundo cuadro (periodo de retención)
            tots = [int(x) for x in re.findall(r"\bTOTAL (\d{1,3}) \d[\d,.]*\d\b", u)]
        tot = sum(tots) if tots else None
        if lotes_pag or tot:
            control.append({"ESTADO_CONTRATO": estado, "FASE": "TODAS" if ter or sus else fase_txt, "LOTES": len(lotes_pag),
                            "TOTAL_PDF": tot, "PAGINA": ip + 1})
    return filas, control


def tablas_adicionales(path):
    """Las tablas de un anual: {"pozos": (filas, control), "regalias": …, "canon": …, "canasta": …, "contratos": …, "errores": {}}."""
    out, errores = {}, {}
    for nombre, f in (("pozos", pozos_por_lote), ("regalias", regalias), ("canon", canon), ("canasta", canasta),
                      ("contratos", contratos)):
        try:
            out[nombre] = f(path)
        except Exception as e:                            # una tabla con problemas no detiene las demás
            out[nombre] = ([], [])
            errores[nombre] = f"{type(e).__name__}: {e}"
    out["errores"] = errores
    _TEXTOS.pop(path, None)
    return out


# ── Resumen ejecutivo de las estadísticas MENSUALES: canon por departamento e ingresos para el Estado, mes a mes ──
DEPARTAMENTOS_CANON = {"LORETO", "UCAYALI", "PIURA", "TUMBES", "HUANUCO", "CUSCO", "JUNIN", "PASCO", "SAN MARTIN",
                       "MADRE DE DIOS", "PUNO", "AYACUCHO", "LAMBAYEQUE", "AMAZONAS"}


def resumen_mensual(path, anio_reporte=None):
    """Página «Resumen ejecutivo» de una estadística mensual: columnas ENERO…DICIEMBRE + TOTAL.
    Devuelve {"canon": [{AÑO, MES, DEPARTAMENTO, CANON_MM_SOLES}], "ingresos": [{AÑO, MES, MODALIDAD, INGRESO_MM_US$}],
    "control": [...], "anio_canon", "anio_ingresos"}. El año del canon sale de la nota «canon y sobrecanon … actualizada a
    <mes> <año>» (va con un mes de rezago: el reporte de enero 2026 trae los 12 meses de 2025)."""
    out = {"canon": [], "ingresos": [], "control": [], "anio_canon": None, "anio_ingresos": None}
    with pdfplumber.open(path) as pdf:
        for ip in paginas_con(path, r"DISTRIBUCI.N DEL CANON|CANON Y SOBRECANON"):
            t = textos_pdf(path)[ip]
            if not re.search(r"ENERO", t) or not re.search(r"INGRESO", t):
                continue
            cols, fs = tabla_por_columnas(pdf.pages[ip], lambda x: mes_de(x) is not None, modo_lote=False, etiquetas_lejos=True,
                                          para_en_total=False)
            if not cols:
                continue
            m = re.search(r"CANON Y SOBRECANON SE ENCUENTRA ACTUALIZADA A\s+(\w+)\s+((?:19|20)\d\d)", t)
            if anio_reporte is None:                       # «RESUMEN EJECUTIVO - ENERO 2026» (página anterior)
                m2 = next((re.search(r"RESUMEN EJECUTIVO\s*-\s*[A-Z]+\s+((?:19|20)\d\d)", x) for x in textos_pdf(path)
                           if re.search(r"RESUMEN EJECUTIVO\s*-\s*[A-Z]+\s+((?:19|20)\d\d)", x)), None)
                anio_reporte = int(m2.group(1)) if m2 else None
            anio_c = int(m.group(2)) if m else anio_reporte
            anio_i = anio_reporte
            meses = [c for c in cols if mes_de(c)]
            seccion, vistos = None, set()
            for f in fs:
                et = sin_tildes(f["ETIQUETA"])
                if "LICENCIA" in et:
                    seccion, clave = "I", "LICENCIA"
                elif "SERVICIO" in et:
                    seccion, clave = "I", "SERVICIOS"
                else:
                    dep = next((d for d in DEPARTAMENTOS_CANON if re.search(r"\b" + d + r"\b", et)), None)
                    if dep:
                        seccion, clave = "C", dep
                    elif re.search(r"\bTOTAL\b", et) and seccion in ("I", "C") and (seccion, "TOTAL") not in vistos:
                        clave = "TOTAL"
                    else:
                        continue
                vistos.add((seccion, clave))
                for c in meses:
                    if c not in f["valores"]:
                        continue
                    v = f["valores"][c]
                    mes = MESES.index(mes_de(c)) + 1
                    if seccion == "C" and anio_c:
                        out["canon"].append({"AÑO": anio_c, "MES": mes, "DEPARTAMENTO": clave, "CANON_MM_SOLES": v, "PAGINA": ip + 1})
                    elif seccion == "I" and anio_i:
                        out["ingresos"].append({"AÑO": anio_i, "MES": mes, "MODALIDAD": clave, "INGRESO_MM_US$": v, "PAGINA": ip + 1})
                if "TOTAL" in f["valores"]:                 # control: suma de meses = columna TOTAL
                    s = sum(f["valores"].get(c, 0) for c in meses)
                    out["control"].append({"TABLA": ("CANON MENSUAL " if seccion == "C" else "INGRESOS MENSUAL ") + clave,
                                           "AÑO": anio_c if seccion == "C" else anio_i, "SUMA": round(s, 2),
                                           "TOTAL_PDF": f["valores"]["TOTAL"], "PAGINA": ip + 1})
            out["anio_canon"], out["anio_ingresos"] = anio_c, anio_i
            # control: suma de departamentos = TOTAL de cada mes
            for mes in sorted({x["MES"] for x in out["canon"]}):
                deps = [x["CANON_MM_SOLES"] for x in out["canon"] if x["MES"] == mes and x["DEPARTAMENTO"] != "TOTAL"]
                tot = next((x["CANON_MM_SOLES"] for x in out["canon"] if x["MES"] == mes and x["DEPARTAMENTO"] == "TOTAL"), None)
                out["control"].append({"TABLA": f"CANON MENSUAL DEPARTAMENTOS MES {mes:02d}", "AÑO": anio_c,
                                       "SUMA": round(sum(deps), 2), "TOTAL_PDF": tot, "PAGINA": ip + 1})
            break
    _TEXTOS.pop(path, None)
    return out


# ── Lectores de los Excel de canon (Determinación por lote, Distribución por beneficiario y Municipalidades) ──

MESES_XLS = {"ENE": 1, "FEB": 2, "MAR": 3, "ABR": 4, "MAY": 5, "JUN": 6, "JUL": 7, "AGO": 8, "SET": 9, "SEP": 9,
             "OCT": 10, "NOV": 11, "DIC": 12}
DEP_LOCALIDAD = {"PUERTO INCA": "HUANUCO", "CUZCO": "CUSCO"}    # 2004–2015 escriben «Cuzco»
TOL_SOLES = 1.0                                   # redondeo de los Excel (S/ o US$)


def _txt(v):
    return re.sub(r"\s+", " ", sin_tildes(str(v))).strip() if isinstance(v, str) else ""


def _num(v):
    return float(v) if isinstance(v, (int, float)) and not isinstance(v, bool) else None


def _filas(ws):
    return [list(r) + [None] * 4 for r in ws.iter_rows(values_only=True)]


def _mes(v):
    m = re.match(r"([A-Z]{3})", _txt(v))
    return MESES_XLS.get(m.group(1)) if m else None


def _anio_hoja(titulo):
    t = str(titulo).strip()
    if re.fullmatch(r"(19|20)\d\d", t):
        return int(t)
    if re.fullmatch(r"9\d", t):                   # hojas «99», «93»…
        return 1900 + int(t)
    return None


def _libro(ruta):
    """Libro de openpyxl; los .xls (formato antiguo) se copian a un libro en memoria con xlrd."""
    if not str(ruta).lower().endswith(".xls"):
        return openpyxl.load_workbook(ruta, read_only=True, data_only=True)
    import xlrd
    libro, wb = xlrd.open_workbook(ruta), openpyxl.Workbook()
    wb.remove(wb.active)
    for sh in libro.sheets():
        ws = wb.create_sheet(sh.name[:31])
        for i in range(sh.nrows):
            fila = []
            for j in range(sh.ncols):
                c = sh.cell(i, j)
                v = xlrd.xldate_as_datetime(c.value, libro.datemode) if c.ctype == xlrd.XL_CELL_DATE else \
                    (None if c.ctype in (xlrd.XL_CELL_EMPTY, xlrd.XL_CELL_BLANK) else c.value)
                fila.append(v)
            ws.append(fila)
    return wb


def _tipo_beneficiario(t):
    if "29289" in t:
        return "LEY 29289"
    if re.search(r"CONC|MUNICIPAL", t):
        return "MUNICIPALIDADES"
    if re.search(r"AMAZONIA PERUANA|INST\.? ?PER|INST\.? ?INVEST|IIAP", t):
        return "IIAP"
    if re.search(r"UNIV|\bU\. ?N\.", t):
        return "UNIVERSIDADES"
    if re.search(r"INST", t):
        return "INSTITUTOS"
    if re.search(r"REGION|CORDE|GOBIERNO", t):
        return "GOBIERNO REGIONAL"
    return "OTRO"


def _cabecera_meses(fila):
    """{columna: mes} de la fila «PERIODO DE PAGO». El año de la etiqueta no se usa: hay hojas con la etiqueta
    del año siguiente copiada (Ucayali 2024 dice «Ene.2025»); el año es el de la hoja y lo confirman las fechas de pago."""
    return {j: _mes(v) for j, v in enumerate(fila) if j > 0 and _mes(v) is not None}


def _fechas(fila, cols):
    out = {}
    for j, m in cols.items():
        v = fila[j] if j < len(fila) else None
        out[m] = v.strftime("%d/%m/%Y") if hasattr(v, "strftime") else ""
    return out


def _limpia_nombre(t):
    t = re.sub(r"\(.*?\)", " ", t).replace("*", " ")
    return re.sub(r"\s+", " ", t).strip(" -")


def _provincia(t):
    """«A L T O    A M A Z O N A S» → «ALTO AMAZONAS»; «MAYNAS (36.72%)» → «MAYNAS»."""
    s = str(t).strip()
    tokens = s.split()
    if len(tokens) >= 3 and all(len(x) == 1 for x in tokens):
        s = " ".join(p.replace(" ", "") for p in re.split(r"\s{2,}", s))
    return _limpia_nombre(sin_tildes(s))


def _es_espaciado(t):
    tokens = str(t).split()
    return len(tokens) >= 3 and all(len(x) == 1 for x in tokens)


# ── Distribución: «Canon por Año <Dep.>» ──────────────────────────────────────────────────────────────────────────
def leer_distribucion(ruta, dep):
    benef, conc, ctrl = [], [], []
    for ws in _libro(ruta).worksheets:
        anio = _anio_hoja(ws.title)
        if anio is None:
            continue
        rows = _filas(ws)
        ip = next((i for i, r in enumerate(rows) if _txt(r[0]).startswith("PERIODO DE PAGO")), None)
        if ip is None:
            ctrl.append({"DEPARTAMENTO": dep, "AÑO": anio, "HOJA": ws.title, "PARTE": "BENEFICIARIOS",
                         "ESTADO": "SIN CABECERA «PERIODO DE PAGO»"})
            continue
        cols = _cabecera_meses(rows[ip])
        c0 = min(cols) if cols else 4
        fechas = _fechas(rows[ip + 1], cols) if _txt(rows[ip + 1][0]).startswith("FECHA") else {}
        filas_b, total, i = [], None, ip + 1
        for i in range(ip + 1, len(rows)):
            r = rows[i]
            textos = [_txt(x) for x in r[:c0] if isinstance(x, str)]
            vals = {m: _num(r[j]) for j, m in cols.items()}
            if any(x.startswith("TOTAL") for x in textos):
                total = vals
                break
            lab = _txt(r[0])
            if not textos and filas_b and any(v is not None for v in vals.values()):
                total = vals                          # Huánuco: la fila del total no lleva la palabra «TOTAL»
                break
            if not lab or lab.startswith("FECHA") or all(v is None for v in vals.values()):
                continue
            pct = next((r[j] for j in range(1, c0) if r[j] is not None), None)
            filas_b.append((lab, pct, vals))
        if total is None:
            ctrl.append({"DEPARTAMENTO": dep, "AÑO": anio, "HOJA": ws.title, "PARTE": "BENEFICIARIOS",
                         "ESTADO": "SIN FILA TOTAL"})
            continue
        meses = [m for m in sorted(set(cols.values())) if total.get(m) is not None]
        for m in meses:
            suma = sum((v.get(m) or 0) for _, _, v in filas_b)
            ok = abs(suma - total[m]) <= TOL_SOLES
            ctrl.append({"DEPARTAMENTO": dep, "AÑO": anio, "MES": m, "HOJA": ws.title, "PARTE": "BENEFICIARIOS",
                         "SUMA": round(suma, 2), "TOTAL_EXCEL": round(total[m], 2), "ESTADO": "OK" if ok else "REVISAR"})
            for lab, pct, v in filas_b:
                if v.get(m) is None:
                    continue
                benef.append({"DEPARTAMENTO": dep, "AÑO": anio, "MES": m, "FECHA": f"01/{m:02d}/{anio}",
                              "FECHA_PAGO": fechas.get(m, ""), "BENEFICIARIO": lab, "TIPO_BENEFICIARIO": _tipo_beneficiario(lab),
                              "PORCENTAJE": str(pct).strip() if pct is not None else "", "MONTO_SOLES": round(v[m], 2)})
        # detalle por concejo (1993–2002)
        idc = next((k for k in range(i + 1, len(rows)) if "DISTRIBUCION POR CONC" in _txt(rows[k][0])), None)
        if idc is None:
            continue
        prov, pct_prov, primero, filas_c, tot_c = "", "", False, [], None
        for r in rows[idc + 1:]:
            textos = [_txt(x) for x in r[:c0] if isinstance(x, str)]
            vals = {m: _num(r[j]) for j, m in cols.items()}
            if any(x.startswith("TOTAL") for x in textos):
                tot_c = vals
                break
            lab = _txt(r[0])
            if not lab:
                continue
            p = re.search(r"\(\s*([\d.,]+)\s*%\s*\)", lab)
            if p or all(v is None for v in vals.values()):        # «MAYNAS (36.72%)»: a veces trae 1 en los meses
                p = re.search(r"\(?\s*([\d.,]+)\s*%\s*\)?", lab)
                prov, pct_prov, primero = _provincia(r[0]), (p.group(1) + "%") if p else "", True
                continue
            filas_c.append((prov, pct_prov, _limpia_nombre(lab), "PROVINCIAL" if primero else "DISTRITAL", vals))
            primero = False
        for m in meses:
            suma = sum((v.get(m) or 0) for *_, v in filas_c)
            ref = (tot_c or {}).get(m)
            ctrl.append({"DEPARTAMENTO": dep, "AÑO": anio, "MES": m, "HOJA": ws.title, "PARTE": "CONCEJOS",
                         "SUMA": round(suma, 2), "TOTAL_EXCEL": round(ref, 2) if ref is not None else None,
                         "ESTADO": "OK" if ref is not None and abs(suma - ref) <= TOL_SOLES else "REVISAR"})
            for prov, pp, nom, niv, v in filas_c:
                if v.get(m) is None:
                    continue
                conc.append({"DEPARTAMENTO": dep, "AÑO": anio, "MES": m, "FECHA": f"01/{m:02d}/{anio}",
                             "FECHA_PAGO": fechas.get(m, ""), "SECCION": "CONCEJOS MUNICIPALES", "PROVINCIA": prov,
                             "PORCENTAJE_PROVINCIA": pp, "MUNICIPALIDAD": nom, "NIVEL": niv,
                             "MONTO_SOLES": round(v[m], 2), "FUENTE_EXCEL": "DISTRIBUCION"})
    return benef, conc, ctrl


# ── Canon Municipalidades: «Canon por Año <Dep.>» (año en curso) ─────────────────────────────────────────────────
def leer_municipal(ruta, dep):
    filas, ctrl = [], []
    for ws in _libro(ruta).worksheets:
        anio = _anio_hoja(ws.title)
        if anio is None:
            continue
        rows = _filas(ws)
        ips = [i for i, r in enumerate(rows) if _txt(r[0]).startswith("PERIODO DE PAGO")]
        if not ips:
            ctrl.append({"DEPARTAMENTO": dep, "AÑO": anio, "HOJA": ws.title, "PARTE": "MUNICIPAL",
                         "ESTADO": "SIN CABECERA «PERIODO DE PAGO»"})
            continue
        ip = ips[-1]                                  # bloque 2: «… GOBIERNOS REGIONALES, LOCALES Y COMUNIDADES»
        cols = _cabecera_meses(rows[ip])
        c0 = min(cols) if cols else 4
        fechas = _fechas(rows[ip + 1], cols) if _txt(rows[ip + 1][0]).startswith("FECHA") else {}
        sec, prov, primero, secciones, niveles = None, "", False, [], {}
        for i in range(ip + 2, len(rows)):
            r = rows[i]
            t0 = _txt(r[0])
            textos = [_txt(x) for x in r[:c0] if isinstance(x, str)]
            vals = {m: _num(r[j]) for j, m in cols.items()}
            hay = any(v is not None for v in vals.values())
            if any(x.startswith("TOTAL") for x in textos):
                if sec is not None:
                    secciones[-1][2] = vals
                    sec = None
                continue
            if "GOBIERNOS LOCALES" in t0:
                sec = "COMUNIDADES (5%)" if "COMUNIDADES" in t0 else "GOBIERNOS LOCALES"
                secciones.append([sec, [], None])
                prov, primero = "", False
                continue
            if sec is None or not t0:
                continue
            if not hay:
                sig = next((x for x in rows[i + 1:i + 4] if any(y is not None for y in x[:c0 + 12])), None)
                sig_hay = sig is not None and _txt(sig[0]) and any(_num(sig[j]) is not None for j in cols)
                if _es_espaciado(r[0]) or sig_hay:
                    prov, primero = _provincia(r[0]), True
                continue
            nom = _limpia_nombre(t0)
            if re.search(r"\(PROVINCIA", t0):
                niv = "PROVINCIAL"
                prov = prov or nom
            elif sec == "GOBIERNOS LOCALES":
                niv = "PROVINCIAL" if primero else "DISTRITAL"
            else:
                if not prov:                          # sección sin encabezados de provincia: la de la sección anterior
                    prov = next((p for (p, n) in niveles if n == nom), "")
                niv = niveles.get((prov, nom), "DISTRITAL")
            if sec == "GOBIERNOS LOCALES":
                niveles[(prov, nom)] = niv
            primero = False
            secciones[-1][1].append((prov, nom, niv, vals))
        if not secciones:
            ctrl.append({"DEPARTAMENTO": dep, "AÑO": anio, "HOJA": ws.title, "PARTE": "MUNICIPAL",
                         "ESTADO": "SIN SECCION «GOBIERNOS LOCALES»"})
        for sec, filas_s, tot in secciones:
            meses = [m for m in sorted(set(cols.values())) if (tot or {}).get(m) is not None]
            if tot is None:
                ctrl.append({"DEPARTAMENTO": dep, "AÑO": anio, "HOJA": ws.title, "PARTE": sec, "ESTADO": "SIN FILA TOTAL"})
            for m in meses:
                suma = sum((v.get(m) or 0) for *_, v in filas_s)
                if suma == 0 and tot[m] == 0:             # mes aún sin reparto municipal en el Excel (p. ej., oct–dic)
                    ctrl.append({"DEPARTAMENTO": dep, "AÑO": anio, "MES": m, "HOJA": ws.title, "PARTE": sec,
                                 "SUMA": 0, "TOTAL_EXCEL": 0, "ESTADO": "SIN REPARTO EN EL EXCEL (todo 0)"})
                    continue
                ctrl.append({"DEPARTAMENTO": dep, "AÑO": anio, "MES": m, "HOJA": ws.title, "PARTE": sec,
                             "SUMA": round(suma, 2), "TOTAL_EXCEL": round(tot[m], 2),
                             "ESTADO": "OK" if abs(suma - tot[m]) <= TOL_SOLES else "REVISAR"})
                for prov, nom, niv, v in filas_s:
                    if v.get(m) is None:
                        continue
                    filas.append({"DEPARTAMENTO": dep, "AÑO": anio, "MES": m, "FECHA": f"01/{m:02d}/{anio}",
                                  "FECHA_PAGO": fechas.get(m, ""), "SECCION": sec, "PROVINCIA": prov,
                                  "PORCENTAJE_PROVINCIA": "", "MUNICIPALIDAD": nom, "NIVEL": niv,
                                  "MONTO_SOLES": round(v[m], 2), "FUENTE_EXCEL": "MUNICIPAL"})
    return filas, ctrl


# ── Determinación: «Canon Petrolero <año>» ───────────────────────────────────────────────────────────────────────
HOJA_MES = re.compile(r"(ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SET|SEP|OCT|NOV|DIC)[A-Z]*")


def _lote_producto(item, unidad):
    s = _txt(str(item)).replace("(*)", " ")
    if isinstance(item, (int, float)):
        s = str(int(item))
    prod = "GAS" if "GAS" in s else "LGN" if "LGN" in s else "CONDENSADO" if "COND" in s else \
        ("GAS" if "BTU" in unidad else "PETROLEO")
    lote = re.split(r"[\s-]*\b(GAS|LGN|COND|CRUDO)\b", s)[0]
    lote = re.sub(r"\s+\(", "(", lote).strip(" -")
    lote = re.sub(r"^Z(\d)", r"Z-\1", lote)              # «Z1 Crudo» → Z-1
    return lote, prod


def _dep_loc(v):
    t = _txt(v)
    return DEP_LOCALIDAD.get(t, t)


def leer_determinacion(ruta, anio):
    filas, soles, ctrl = [], [], []
    for ws in _libro(ruta).worksheets:
        nombre = _txt(ws.title).replace(" ", "")
        if not HOJA_MES.fullmatch(nombre):
            continue                                  # «TOTAL», «Ene-2», «Abr (2)», «Nov-original», «cuadre»…
        mes = MESES_XLS[nombre[:3]]
        rows = _filas(ws)
        ih = next((i for i, r in enumerate(rows[:20]) if "LOTE" in _txt(r[0])), None)
        col = {}
        if ih is not None:
            for r in rows[max(0, ih - 2):ih + 1]:
                for j, v in enumerate(r):
                    t = _txt(v)
                    for clave, pat in (("VP", "VALOR DE LA PRODUCCION"), ("CANON", "CANON Y SOBRECANON"),
                                       ("LOC1", "LOCALIDAD 1"), ("LOC2", "LOCALIDAD 2"), ("UNID", "UNIDADES"),
                                       ("MOD", "MODALIDAD"), ("PRECIO", "CANASTA O VALOR VENTA"), ("DED", "DEDUCCION")):
                        if t.startswith(pat) and clave not in col:
                            col[clave] = j
        if not {"VP", "CANON", "LOC1"} <= set(col):
            ctrl.append({"AÑO": anio, "MES": mes, "HOJA": ws.title, "PARTE": "HOJA", "ESTADO": "SIN CABECERA RECONOCIBLE"})
            continue
        tc = next((float(m.group(1)) for r in rows[:8] for v in r if isinstance(v, str)
                   for m in [re.search(r"Tipo de Cambio\s*=\s*([\d.]+)", v)] if m), None)
        cu = col.get("UNID")
        cq = [cu + 1, cu + 2] if cu is not None else []
        zona, d1, d2, en_zona, hoja_filas = "", "", "", [], []
        for r in rows[ih + 1:]:
            t0, t1, t2 = _txt(r[0]), _txt(r[1]), _txt(r[2])
            if t0.startswith("TOTAL GENERAL"):
                break
            if t0 and t0 != "-" and r[1] is None and not t0.startswith("TOTAL"):
                # encabezado de zona; en algunas hojas (2024) la de Cusco no repite el nombre en «Localidad 1»
                if isinstance(r[col["LOC1"]], str):
                    d1 = _dep_loc(r[col["LOC1"]])
                    d2 = _dep_loc(r[col["LOC2"]]) if "LOC2" in col and isinstance(r[col["LOC2"]], str) else ""
                else:
                    deps = [_dep_loc(d) for d in ("PIURA", "TUMBES", "HUANUCO", "LORETO", "UCAYALI", "CUSCO", "CUZCO") if d in t0]
                    d1, d2 = (deps + ["", ""])[:2]
                zona, en_zona = t0, []
                continue
            es_lote = r[1] is not None and not t1.startswith("SUBTOTAL") and (
                t0 == "-" or (r[0] is None and _txt(r[2]) in ("LICENCIA", "SERVICIOS", "OPERACIONES")))  # «VII/VI» jun-2024 sin «-»
            if es_lote and zona:
                unidad = _txt(r[cu]) if cu is not None else ""
                aj = t1.startswith("AJUSTE")
                lote, prod = ("AJUSTES MESES ANTERIORES", "") if aj else _lote_producto(r[1], unidad)
                vol = sum(_num(r[j]) or 0 for j in cq) if cq and not aj else None
                f = {"AÑO": anio, "MES": mes, "FECHA": f"01/{mes:02d}/{anio}", "ZONA": zona,
                     "LOTE_ITEM": str(int(r[1])) if isinstance(r[1], (int, float)) else str(r[1]).strip(),
                     "LOTE": lote, "PRODUCTO": prod, "MODALIDAD": _txt(r[col["MOD"]]) if "MOD" in col else "",
                     "UNIDAD": unidad, "VOLUMEN": vol,
                     "PRECIO_1Q": _num(r[col["PRECIO"]]) if "PRECIO" in col else None,
                     "PRECIO_2Q_MES": _num(r[col["PRECIO"] + 1]) if "PRECIO" in col else None,
                     "DEDUCCIONES_US$": (sum(_num(r[j]) or 0 for j in range(col["DED"], col["VP"]))
                                         if "DED" in col and not aj else None),
                     "VALOR_PRODUCCION_US$": _num(r[col["VP"]]),
                     "CANON_TOTAL_US$": _num(r[col["CANON"]]), "DEPARTAMENTO_1": d1,
                     "CANON_DEP_1_US$": _num(r[col["LOC1"]]), "DEPARTAMENTO_2": d2,
                     "CANON_DEP_2_US$": _num(r[col["LOC2"]]) if "LOC2" in col and d2 else None,
                     "TIPO_CAMBIO": tc, "HOJA": ws.title}
                en_zona.append(f)
                hoja_filas.append(f)
                continue
            if t1.startswith("SUBTOTAL") and zona:
                for clave, campo in (("CANON", "CANON_TOTAL_US$"), ("LOC1", "CANON_DEP_1_US$"), ("LOC2", "CANON_DEP_2_US$")):
                    ref = _num(r[col[clave]]) if clave in col else None
                    if ref is None or (clave == "LOC2" and not d2):
                        continue
                    suma = sum(x[campo] or 0 for x in en_zona)
                    ctrl.append({"AÑO": anio, "MES": mes, "HOJA": ws.title, "PARTE": f"{zona} · {campo}",
                                 "SUMA": round(suma, 2), "TOTAL_EXCEL": round(ref, 2),
                                 "ESTADO": "OK" if abs(suma - ref) <= TOL_SOLES else "REVISAR"})
                continue
            if t2.endswith("S/.") and zona:
                for d, clave in ((d1, "LOC1"), (d2, "LOC2")):
                    v = _num(r[col[clave]]) if clave in col else None
                    if d and v:
                        soles.append({"AÑO": anio, "MES": mes, "DEPARTAMENTO": d, "CANON_SOLES": round(v, 2),
                                      "TIPO_CAMBIO": tc, "HOJA": ws.title})
        if not any((x["VALOR_PRODUCCION_US$"] or 0) != 0 for x in hoja_filas):
            ctrl.append({"AÑO": anio, "MES": mes, "HOJA": ws.title, "PARTE": "HOJA", "ESTADO": "VACIA (sin valores)"})
            soles = [s for s in soles if not (s["MES"] == mes and s["HOJA"] == ws.title)]
            ctrl[:] = [c for c in ctrl if not (c.get("MES") == mes and c.get("HOJA") == ws.title and c["PARTE"] != "HOJA")]
            continue
        filas += hoja_filas
    return filas, soles, ctrl


In [ ]:
# ═══ BLOQUE 2 · SELECTORES (elige y luego ejecuta el Bloque 3) ═══
_años = list(range(date.today().year, 1993, -1))
_ancho = {"description_width": "190px"}
sel_anio_ini = w.Dropdown(options=_años, value=1994, description="Año inicio", style=_ancho)
sel_anio_fin = w.Dropdown(options=_años, value=_años[0], description="Año fin", style=_ancho)
chk_mensual = w.Checkbox(value=True, description="Año en curso con las estadísticas mensuales")
chk_pdf = w.Checkbox(value=True, description="Guardar los PDF en Drive (DATA/PDF)")
chk_diaria = w.Checkbox(value=True, description="Producción diaria (página «Producción Diaria»)")
chk_extra = w.Checkbox(value=True, description="Pozos, regalías, canon, precio de canasta y contratos (de los anuales)")
chk_canon_xls = w.Checkbox(value=True, description="Excel de canon (determinación por lote, distribución y municipalidades)")
display(w.VBox([
    w.HTML("<b>Período</b>. Cada <b>anual</b> trae los 12 meses por lote; para el año en curso "
           "se usa la última <b>mensual</b> publicada (trae el acumulado del año)."),
    w.HBox([sel_anio_ini, sel_anio_fin]), chk_mensual, chk_pdf,
    w.HTML("La <b>producción diaria</b> (líquidos y gas por lote y día) se acumula aparte en "
           "<code>DATA/PP_produccion_diaria.csv</code>; no depende del período elegido."), chk_diaria,
    w.HTML("De cada <b>anual</b> del período también se leen los <b>pozos perforados</b> por lote (series de 10 años: "
           "con 2005 se llega a 1995), las <b>regalías</b> por lote, el <b>canon/sobrecanon</b> por departamento "
           "y el <b>precio de canasta</b> mensual por lote (con él, el valor de la producción)."), chk_extra,
    w.HTML("Los <b>Excel de canon</b> de las páginas «Canon y Sobre Canon» y «Canon Municipalidades» se guardan en "
           "<code>DATA/CANON_EXCEL</code> (canon por lote y mes, reparto por beneficiario y por municipalidad)."), chk_canon_xls]))


In [ ]:
# ═══ BLOQUE 3 · DESCARGA, EXTRACCIÓN, CONTROL Y DRIVE ═══
# Si se corta, vuelve a ejecutarlo: los PDF ya guardados en Drive no se vuelven a descargar.

# ── 1. CONFIGURACIÓN FIJA ────────────────────────────────────────────────────
DRIVE_FOLDER_ID = "1dOOcsVyw915yKg6BzSdICVfCpYTiyeaq"            # DATA (CSV + subcarpeta PDF)
DRIVE_FOLDER_CONTROL_ID = "1TKjS2Wjpnf26oladjhqstmN5c9Irq-Aj"    # CONTROL
PAGINA_EST = ("https://www.perupetro.com.pe/wps/portal/corporativo/PerupetroSite/estadisticas/"
              "estad%C3%ADstica%20petrolera/!ut/p/z1/pdFND8IgDAbgn1TYEPW4xDlc_BiKbvZiOBkSnR6Mv19YvK41sVwged5QKCB0gL1"
              "_h6t_hUfvb_F8Rn3ZZqUyRsm6ajYzYZVb1C5u5FpBOwAxUoUAJPKH_JsfATbTVL48aTqfAJGfiymdT4DIu4bpP4F_3p8A_vL_BEB6PC0gR"
              "5DqweYTBqQZEqAyhrtirxmwVDSQOwboOOjn_ThUJ8IqruIDQCD6fQ!!/dz/d5/L2dBISEvZ0FBIS9nQSEh/")
BASE = "https://www.perupetro.com.pe"
DEPARTAMENTO_ZONA = {"SELVA NORTE": "LORETO"}   # respaldo si el lote no está en DATA/PP_LOTES_GEO.csv (mapa de lotes)
PAUSA = 2                                        # segundos entre descargas
ruta = "/content/Perupetro"; os.makedirs(ruta, exist_ok=True)
A_INI, A_FIN = sel_anio_ini.value, sel_anio_fin.value
if A_INI > A_FIN:
    raise ValueError("El año de inicio es posterior al año fin.")
VERIFICAR_SSL = True     # False solo como último recurso si la cadena no se puede completar (desactiva la verificación)
CAB = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                     "Chrome/124.0 Safari/537.36",
       # sin idioma, el portal responde en inglés y esa versión no trae la tabla de meses de «Producción Diaria»
       "Accept-Language": "es-PE,es;q=0.9"}

# ── 2. GOOGLE DRIVE ──────────────────────────────────────────────────────────
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
import logging
auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

def archivos_en(folder_id):
    out, token = {}, None
    while True:
        r = drive.files().list(q=f"'{folder_id}' in parents and trashed = false",
                               fields="nextPageToken, files(id,name,mimeType)", pageSize=1000, pageToken=token,
                               supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out.update({f["name"]: f["id"] for f in r.get("files", [])})
        token = r.get("nextPageToken")
        if not token:
            return out

def subir(ruta_local, folder_id, existentes, mime="text/csv"):
    nombre = os.path.basename(ruta_local)
    for i in range(1, 6):
        try:
            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if nombre in existentes:
                drive.files().update(fileId=existentes[nombre], media_body=media, supportsAllDrives=True).execute()
            else:
                f = drive.files().create(body={"name": nombre, "parents": [folder_id]}, media_body=media,
                                         fields="id", supportsAllDrives=True).execute()
                existentes[nombre] = f["id"]
            return True
        except Exception as e:
            print(f"   ↻ Drive reintento {i}/5: {e}"); time.sleep(2 ** i)
    return False

def bajar_de_drive(file_id, destino):
    with open(destino, "wb") as fh:
        dl = MediaIoBaseDownload(fh, drive.files().get_media(fileId=file_id, supportsAllDrives=True))
        hecho = False
        while not hecho:
            _, hecho = dl.next_chunk()

datos_drive = archivos_en(DRIVE_FOLDER_ID)
control_drive = archivos_en(DRIVE_FOLDER_CONTROL_ID)
if "PDF" not in datos_drive:
    datos_drive["PDF"] = drive.files().create(body={"name": "PDF", "parents": [DRIVE_FOLDER_ID],
                                                    "mimeType": "application/vnd.google-apps.folder"},
                                              fields="id", supportsAllDrives=True).execute()["id"]
PDF_FOLDER_ID = datos_drive["PDF"]
pdf_drive = archivos_en(PDF_FOLDER_ID)

# Departamento real de cada lote (notebook PP_Mapa_Lotes.ipynb → DATA/PP_LOTES_GEO.csv); si no está, se usa la zona
DEPTO_LOTE = {}
if "PP_LOTES_GEO.csv" in datos_drive:
    try:
        bajar_de_drive(datos_drive["PP_LOTES_GEO.csv"], os.path.join(ruta, "PP_LOTES_GEO.csv"))
        _g = pd.read_csv(os.path.join(ruta, "PP_LOTES_GEO.csv"), encoding="utf-8-sig", dtype=str)
        DEPTO_LOTE = {k: v for k, v in zip(_g["LOTE"], _g["DEPARTAMENTO"]) if isinstance(v, str) and v}
        print(f"🗺️  Departamentos por lote tomados del mapa: {len(DEPTO_LOTE)} lotes")
    except Exception as e:
        print(f"⚠️ No se pudo leer PP_LOTES_GEO.csv ({e}); se usa la zona")

def depto_de(lote, zona):
    base = re.sub(r"\(T+\)$", "", lote)
    for k in (lote, base):
        if k in DEPTO_LOTE:
            return DEPTO_LOTE[k], "MAPA"
    d = DEPARTAMENTO_ZONA.get(zona, "")
    return d, ("ZONA" if d else "")
print(f"☁️  Drive conectado · PDF ya guardados: {len(pdf_drive)}")

# ── 3. CERTIFICADO HTTPS DE PERUPETRO ───────────────────────────────────────
# El servidor no envía su certificado intermedio (los navegadores lo completan solos; Python no).
# Se descarga desde la dirección que indica el propio certificado (AIA) y se verifica con la cadena completa.
import ssl, socket, certifi
from cryptography import x509
from cryptography.hazmat.primitives.serialization import Encoding
from cryptography.x509.oid import ExtensionOID, AuthorityInformationAccessOID

def cert_del_servidor(host):
    ctx = ssl.create_default_context(); ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
    with socket.create_connection((host, 443), timeout=30) as s, ctx.wrap_socket(s, server_hostname=host) as ss:
        return x509.load_der_x509_certificate(ss.getpeercert(binary_form=True))

def emisores(cert):
    try:
        aia = cert.extensions.get_extension_for_oid(ExtensionOID.AUTHORITY_INFORMATION_ACCESS).value
    except x509.ExtensionNotFound:
        return []
    return [d.access_location.value for d in aia if d.access_method == AuthorityInformationAccessOID.CA_ISSUERS]

def leer_cert(datos):
    try:
        return x509.load_der_x509_certificate(datos)
    except ValueError:
        return x509.load_pem_x509_certificate(datos)

def bundle_con_intermedios(host, destino):
    cert, pems = cert_del_servidor(host), []
    for _ in range(3):                            # sube por la cadena hasta 3 niveles
        urls = emisores(cert)
        if not urls or cert.issuer == cert.subject:
            break
        cert = leer_cert(requests.get(urls[0], timeout=30).content)
        pems.append(cert.public_bytes(Encoding.PEM).decode())
    with open(destino, "w") as f:
        f.write(open(certifi.where()).read() + "\n" + "\n".join(pems))
    return destino, len(pems)

def preparar_verificacion():
    try:
        requests.get(BASE, headers=CAB, timeout=30)
        return True                               # la cadena ya es válida
    except requests.exceptions.SSLError:
        pass
    try:
        ruta_ca, n = bundle_con_intermedios("www.perupetro.com.pe", os.path.join(ruta, "ca_perupetro.pem"))
        requests.get(BASE, headers=CAB, timeout=30, verify=ruta_ca)
        print(f"🔒 Certificado de PERUPETRO completado con {n} intermedio(s): conexión verificada")
        return ruta_ca
    except Exception as e:
        if VERIFICAR_SSL:
            print(f"⚠️ No se pudo completar el certificado de PERUPETRO ({type(e).__name__}: {e}).\n"
                  "   Opciones: sube los PDF a mano a DATA/PDF, o pon VERIFICAR_SSL = False arriba (último recurso).")
            return True
        print("⚠️ VERIFICAR_SSL = False: se descarga SIN verificar el certificado (solo PDF públicos)")
        return False

VERIFY = preparar_verificacion()

# ── 4. ENLACES DE LA PÁGINA "ESTADÍSTICA PETROLERA" ─────────────────────────
MES_ABR = {"ENE": 1, "FEB": 2, "MAR": 3, "ABR": 4, "MAY": 5, "JUN": 6, "JUL": 7, "AGO": 8,
           "SET": 9, "SEP": 9, "OCT": 10, "NOV": 11, "DIC": 12}

def clasificar(href, texto):
    """(tipo, año, mes) a partir del nombre del archivo y el texto del enlace; None si no se sabe."""
    nombre = sin_tildes(unquote(href).replace("+", " "))
    if re.search(r"REGLAMENTO|DECRETO|RESOLUCION|\bLEY\b|D\.S|PARTICIPACION", nombre):
        return None, None, None                   # documentos legales enlazados en la misma página
    anio = re.findall(r"(?<!\d)(199\d|20[0-4]\d)(?!\d)", nombre)
    t = sin_tildes(html.unescape(re.sub(r"<[^>]+>", " ", texto))).strip()
    if "MENSUAL" in nombre:
        m = re.search(r"(ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SET|SEP|OCT|NOV|DIC)[A-Z]*\s+(?:DE\s+)?(\d{4})", nombre)
        if m:
            return "MENSUAL", int(m.group(2)), MES_ABR[m.group(1)]
        return "MENSUAL", int(anio[-1]) if anio else None, MES_ABR.get(t[:3])
    if "ANUAL" in nombre or re.fullmatch(r"(19|20)\d\d", t):
        return "ANUAL", int(anio[-1]) if anio else (int(t) if t.isdigit() else None), None
    if anio:                                      # en esta página, un PDF con año y sin "mensual" es el anual
        return "ANUAL", int(anio[-1]), None
    return None, None, None

try:
    pagina = requests.get(PAGINA_EST, headers=CAB, timeout=60, verify=VERIFY).text
except Exception as e:
    pagina = ""
    print(f"⚠️ No se pudo leer la página de PERUPETRO ({e}). Se usarán solo los PDF que ya estén en Drive/PDF.")
enlaces = {}
for href, texto in re.findall(r'<a[^>]+href="([^"]+)"[^>]*>(.*?)</a>', pagina, re.S | re.I):
    href = html.unescape(href)
    if "/wps/wcm/connect/" in href and (".pdf" in href.lower() or "AJPERES" in href.upper()):
        url = urljoin(BASE, href)
        enlaces.setdefault(url, clasificar(href, texto))
print(f"🔗 Enlaces a PDF en la página: {len(enlaces)}")

# ── 5. DESCARGA (o reutiliza lo guardado en Drive) Y EXTRACCIÓN ─────────────
NOMBRE_CATALOGO = "PP_CATALOGO_PDF.csv"
catalogo = pd.DataFrame(columns=["URL", "ARCHIVO"])
if NOMBRE_CATALOGO in control_drive:
    try:                                          # un catálogo vacío (corrida que falló antes) no debe frenar
        leido = pd.read_csv(io.BytesIO(drive.files().get_media(fileId=control_drive[NOMBRE_CATALOGO],
                            supportsAllDrives=True).execute()), dtype=str)
        if {"URL", "ARCHIVO"} <= set(leido.columns):
            catalogo = leido.dropna(subset=["URL", "ARCHIVO"])
    except pd.errors.EmptyDataError:
        pass
url_a_archivo = dict(zip(catalogo["URL"], catalogo["ARCHIVO"]))
año_hoy = date.today().year

def en_rango(tipo, anio):
    if tipo is None:
        return False                              # no es una estadística
    if anio is None:
        return True                               # se decide al leer el PDF
    if tipo == "MENSUAL":
        return chk_mensual.value and A_INI <= anio <= A_FIN
    return A_INI <= anio <= A_FIN

def descargar(url):
    for i in range(1, 4):
        try:
            r = requests.get(url, headers=CAB, timeout=180, verify=VERIFY)
            if r.status_code == 200 and r.content[:4] == b"%PDF":
                return r.content
            print(f"   ↻ HTTP {r.status_code} ({i}/3)")
        except Exception as e:
            print(f"   ↻ {type(e).__name__} ({i}/3)")
        time.sleep(3 * i)
    return None

resultados, control = [], []
pendientes = [(u, c) for u, c in enlaces.items() if en_rango(c[0], c[1])]
# PDF puestos a mano en Drive/PDF que no vienen de la página
manuales = [n for n in pdf_drive if n.lower().endswith(".pdf") and n not in url_a_archivo.values()]
print(f"📄 A procesar: {len(pendientes)} de la página + {len(manuales)} subidos a mano en Drive/PDF\n")

def procesar(local, origen):
    t0 = time.time()
    try:
        r = extraer(local)
    except Exception as e:
        return None, {"ARCHIVO": origen, "ESTADO": f"ERROR: {type(e).__name__}: {e}"}
    if not r:
        return None, {"ARCHIVO": origen, "ESTADO": "NO ES ESTADÍSTICA DE HIDROCARBUROS"}
    if r["TIPO"] is None:                         # PDF antiguo: escaneado o con otro diseño
        return None, {"ARCHIVO": origen, "ESTADO": r["MOTIVO"]}
    try:                                          # gas natural: misma lectura, tabla en MPC
        g = extraer(local, "GAS")
        r["gas"] = {"filas": g.get("filas", []), "totales": g.get("totales", [])}
    except Exception as e:
        r["gas"] = {"filas": [], "totales": [], "error": f"{type(e).__name__}: {e}"}
    if chk_extra.value and r["TIPO"] == "ANUAL":  # pozos, regalías, canon, canasta y contratos del mismo anual
        r["extra"] = tablas_adicionales(local)
    if chk_extra.value and r["TIPO"] == "MENSUAL":  # canon por departamento e ingresos para el Estado, mes a mes
        try:
            r["mensual"] = resumen_mensual(local, r["ANIO"])
        except Exception as e:
            r["mensual"] = {"canon": [], "ingresos": [], "control": [], "error": f"{type(e).__name__}: {e}"}
    return r, {"ARCHIVO": origen, "TIPO": r["TIPO"], "AÑO": r["ANIO"], "MES_REPORTE": r["MES_REPORTE"],
               "SEGUNDOS": round(time.time() - t0, 1)}

trabajos = [("web", u) for u, _ in pendientes] + [("drive", n) for n in manuales]
for n, (fuente, ref) in enumerate(trabajos, 1):
    archivo = url_a_archivo.get(ref) if fuente == "web" else ref
    local = os.path.join(ruta, archivo or f"tmp_{n}.pdf")
    if archivo and archivo in pdf_drive:
        bajar_de_drive(pdf_drive[archivo], local)
    elif fuente == "web":
        contenido = descargar(ref)
        if contenido is None:
            control.append({"ARCHIVO": ref, "ESTADO": "ERROR DE DESCARGA"}); print(f"❌ [{n}] {ref[:90]}")
            continue
        open(local, "wb").write(contenido)
        time.sleep(PAUSA)
    r, fila = procesar(local, archivo or ref)
    tipo_w, anio_w, mes_w = enlaces.get(ref, (None, None, None)) if fuente == "web" else (None, None, None)
    if not r and fila["ESTADO"].startswith(("ESCANEADO", "FORMATO ANTIGUO")) and tipo_w and anio_w:
        # se guarda igual en Drive con nombre ordenado (servirá para la etapa de OCR)
        nombre = f"PP_{tipo_w}_{anio_w}" + (f"_{mes_w:02d}" if tipo_w == "MENSUAL" and mes_w else "") + ".pdf"
        url_a_archivo[ref] = nombre
        if chk_pdf.value and nombre not in pdf_drive:
            os.rename(local, os.path.join(ruta, nombre)); local = os.path.join(ruta, nombre)
            subir(local, PDF_FOLDER_ID, pdf_drive, mime="application/pdf")
        fila.update({"ARCHIVO": nombre, "TIPO": tipo_w, "AÑO": anio_w})
    if r:
        if not en_rango(r["TIPO"], r["ANIO"]):
            os.remove(local); continue
        nombre = f"PP_{r['TIPO']}_{r['ANIO']}" + (f"_{r['MES_REPORTE']:02d}" if r["TIPO"] == "MENSUAL" else "") + ".pdf"
        if fuente == "web":
            url_a_archivo[ref] = nombre
            if chk_pdf.value and nombre not in pdf_drive:
                os.rename(local, os.path.join(ruta, nombre)); local = os.path.join(ruta, nombre)
                subir(local, PDF_FOLDER_ID, pdf_drive, mime="application/pdf")
        fila["ARCHIVO"] = nombre
        resultados.append(r)
    control.append(fila)
    if os.path.exists(local):
        os.remove(local)
    print(f"{'✅' if r else '⚠️'} [{n}/{len(trabajos)}] {fila['ARCHIVO']}: "
          f"{len(r['filas']) if r else 0} filas · {fila.get('ESTADO', '')}")

pd.DataFrame([{"URL": u, "ARCHIVO": a} for u, a in url_a_archivo.items()], columns=["URL", "ARCHIVO"]).to_csv(
    os.path.join(ruta, NOMBRE_CATALOGO), index=False, encoding="utf-8-sig")
subir(os.path.join(ruta, NOMBRE_CATALOGO), DRIVE_FOLDER_CONTROL_ID, control_drive)

# ── 6. TABLA FINAL Y CONTROL ─────────────────────────────────────────────────
# Prioridad: el ANUAL de cada año; para años sin anual, la MENSUAL más reciente (trae el acumulado).
anuales = {r["ANIO"] for r in resultados if r["TIPO"] == "ANUAL"}
elegidos = {}
for r in resultados:
    if r["TIPO"] == "ANUAL":
        elegidos[(r["ANIO"], "A")] = r
    elif r["ANIO"] not in anuales:
        k = (r["ANIO"], "M")
        if k not in elegidos or r["MES_REPORTE"] > elegidos[k]["MES_REPORTE"]:
            elegidos[k] = r

filas, ctrl_final, filas_gas = [], [], []
for (anio, _), r in sorted(elegidos.items()):
    hasta = 12 if r["TIPO"] == "ANUAL" else r["MES_REPORTE"]
    archivo = f"PP_{r['TIPO']}_{anio}" + (f"_{r['MES_REPORTE']:02d}" if r["TIPO"] == "MENSUAL" else "") + ".pdf"
    df = pd.DataFrame(r["filas"])
    est = "OK" if len(df) else "SIN TABLA"
    fila_c = {"ARCHIVO": archivo, "TIPO": r["TIPO"], "AÑO": anio, "MESES": hasta, "LOTES": len(df),
              "FILAS_IMPUTADAS": int(df["ESTADO"].str.startswith("IMPUTADO").sum()) if len(df) else 0,
              "FILAS_REVISAR": int((df["ESTADO"] == "REVISAR").sum()) if len(df) else 0}
    # suma de lotes vs el SUB-TOTAL/TOTAL que imprime el PDF, mes a mes.
    # Anuales recientes: un subtotal por producto. Antiguos (p. ej. 2001): un solo TOTAL general.
    def dif_contra(sub, refs):
        difs = [max(abs(sub[m].sum() - t.get(m, 0)) for m in MESES[:hasta] if m in t) for t in refs]
        return min(difs) if difs else None
    dif_total = dif_contra(df, r["totales"]) if len(df) else None
    fila_c["DIF_MAX_TOTAL"] = dif_total
    # El TOTAL general de los anuales antiguos va al final de la tabla y queda marcado con el último
    # producto (LGN): no es un subtotal de ese producto y no se usa para su control.
    gran_total = float(df["TOTAL"].sum()) if len(df) else 0.0
    es_general = lambda t: df["PRODUCTO"].nunique() > 1 and abs((t.get("TOTAL") or 0) - gran_total) <= 1
    for prod in ("PETROLEO", "LGN", "CONDENSADO"):
        sub = df[df["PRODUCTO"] == prod] if len(df) else df
        refs = [t for t in r["totales"] if t["PRODUCTO"] == prod and not es_general(t)]
        dif = dif_contra(sub, refs) if len(sub) else 0.0
        fila_c[f"BARRILES_{prod}"] = round(float(sub["TOTAL"].sum()), 0) if len(sub) else 0
        fila_c[f"DIF_MAX_{prod}"] = dif
        if (dif is None or dif > 1) and not (dif_total is not None and dif_total <= 1):
            est = "REVISAR"
    if fila_c["FILAS_REVISAR"]:
        est = "REVISAR"
    fila_c["ESTADO"] = est
    ctrl_final.append(fila_c)
    # gas natural del mismo PDF (MPC = miles de pies cúbicos)
    dg = pd.DataFrame(r.get("gas", {}).get("filas", []))
    fila_c["LOTES_GAS"] = len(dg)
    fila_c["GAS_MPC"] = round(float(dg["TOTAL"].sum()), 0) if len(dg) else 0
    fila_c["DIF_MAX_GAS"] = dif_contra(dg, r["gas"]["totales"]) if len(dg) else None
    fila_c["ESTADO_GAS"] = ("SIN TABLA" if not len(dg) else
                            "REVISAR" if (fila_c["DIF_MAX_GAS"] is None or fila_c["DIF_MAX_GAS"] > 12
                                          or (dg["ESTADO"] == "REVISAR").any()) else "OK")
    for _, f in dg.iterrows():
        for k, m in enumerate(MESES[:hasta], 1):
            mpc = float(f[m] or 0)
            filas_gas.append({"AÑO": anio, "MES": k, "FECHA": f"01/{k:02d}/{anio}", "ZONA": f["ZONA"],
                              **dict(zip(("DEPARTAMENTO", "DEPARTAMENTO_FUENTE"), depto_de(f["LOTE"], f["ZONA"]))),
                              "OPERADOR": norm_operador(f["OPERADOR"]), "OPERADOR_ORIGINAL": f["OPERADOR"],
                              "LOTE": f["LOTE"], "LOTE_HISTORICO": lote_historico(f["LOTE"]),
                              "LOTE_ORIGINAL": f["LOTE_ORIGINAL"], "PRODUCTO": "GAS NATURAL",
                              "MPC": mpc, "MMPCD": round(mpc / calendar.monthrange(anio, k)[1] / 1000, 3),
                              "ESTADO_FILA": f["ESTADO"], "FUENTE": archivo})
    for _, f in df.iterrows():
        for k, m in enumerate(MESES[:hasta], 1):
            bls = float(f[m] or 0)
            filas.append({"AÑO": anio, "MES": k, "FECHA": f"01/{k:02d}/{anio}", "ZONA": f["ZONA"],
                          **dict(zip(("DEPARTAMENTO", "DEPARTAMENTO_FUENTE"), depto_de(f["LOTE"], f["ZONA"]))),
                          "OPERADOR": norm_operador(f["OPERADOR"]), "OPERADOR_ORIGINAL": f["OPERADOR"],
                          "LOTE": f["LOTE"], "LOTE_HISTORICO": lote_historico(f["LOTE"]),
                          "LOTE_ORIGINAL": f["LOTE_ORIGINAL"], "PRODUCTO": f["PRODUCTO"],
                          "BARRILES": bls, "BPD": round(bls / calendar.monthrange(anio, k)[1], 1),
                          "ESTADO_FILA": f["ESTADO"], "FUENTE": archivo})

# ── Años escaneados (1994–2000): transcripciones validadas guardadas en el repositorio ──
# Cada tabla se transcribió a mano desde la imagen y se validó: cada lote suma su total anual, cada mes suma
# el TOTAL del mes y el gran total coincide con el PDF.
URL_TRANSC = ("https://raw.githubusercontent.com/adriansivina-droid/scraper-mef/main/HIDROCARBUROS/"
              "transcripciones/PP_mensual_{}_transcrito.csv")
MES_TR = ["ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SET", "OCT", "NOV", "DIC"]
anios_ok = {c["AÑO"] for c in ctrl_final if c["ESTADO"] == "OK"}
anios_transc = set()
for anio in range(A_INI, A_FIN + 1):
    if anio in anios_ok:
        continue
    try:
        rt = requests.get(URL_TRANSC.format(anio), timeout=30)
    except Exception:
        continue
    if rt.status_code != 200:
        continue
    t = pd.read_csv(io.StringIO(rt.text), dtype={"LOTE": str, "LOTE_ORIGINAL": str})
    tot, t = t[t["PRODUCTO"] == "TOTAL"], t[t["PRODUCTO"] != "TOTAL"]
    dif = max(abs(t[m].sum() - tot.iloc[0][m]) for m in MES_TR + ["TOTAL"]) if len(tot) else None
    filas_mal = int((t[MES_TR].sum(axis=1) != t["TOTAL"]).sum())
    fila_c = {"ARCHIVO": t["FUENTE"].iloc[0], "TIPO": "TRANSCRITO", "AÑO": anio, "MESES": 12, "LOTES": len(t),
              "FILAS_IMPUTADAS": 0, "FILAS_REVISAR": filas_mal, "DIF_MAX_TOTAL": dif,
              "ESTADO": "OK (transcrito)" if dif == 0 and filas_mal == 0 else "REVISAR"}
    for prod_ in ("PETROLEO", "LGN", "CONDENSADO"):
        fila_c[f"BARRILES_{prod_}"] = float(t.loc[t["PRODUCTO"] == prod_, "TOTAL"].sum())
    ctrl_final.append(fila_c)
    anios_transc.add(anio)
    for _, f in t.iterrows():
        for k, m in enumerate(MES_TR, 1):
            bls = float(f[m])
            filas.append({"AÑO": anio, "MES": k, "FECHA": f"01/{k:02d}/{anio}", "ZONA": zona_de(f["LOTE"]),
                          **dict(zip(("DEPARTAMENTO", "DEPARTAMENTO_FUENTE"), depto_de(f["LOTE"], zona_de(f["LOTE"])))),
                          "OPERADOR": norm_operador(f["OPERADOR"]), "OPERADOR_ORIGINAL": f["OPERADOR"],
                          "LOTE": f["LOTE"], "LOTE_HISTORICO": lote_historico(f["LOTE"]),
                          "LOTE_ORIGINAL": f["LOTE_ORIGINAL"], "PRODUCTO": f["PRODUCTO"],
                          "BARRILES": bls, "BPD": round(bls / calendar.monthrange(anio, k)[1], 1),
                          "ESTADO_FILA": "TRANSCRITO", "FUENTE": "TRANSCRIPCION " + f["FUENTE"]})
if anios_transc:
    print(f"✍️ Años tomados de transcripciones validadas: {', '.join(map(str, sorted(anios_transc)))}")

# ── Gas natural de los años escaneados (1994–2000): transcripciones validadas igual que las de líquidos ──
URL_TRANSC_GAS = URL_TRANSC.replace("PP_mensual_", "PP_gas_mensual_")
anios_gas = {f["AÑO"] for f in filas_gas}
anios_gas_tr = set()
for anio in range(A_INI, A_FIN + 1):
    if anio in anios_gas:
        continue
    try:
        rt = requests.get(URL_TRANSC_GAS.format(anio), timeout=30)
    except Exception:
        continue
    if rt.status_code != 200:
        continue
    t = pd.read_csv(io.StringIO(rt.text), dtype={"LOTE": str, "LOTE_ORIGINAL": str})
    tot, t = t[t["PRODUCTO"] == "TOTAL"], t[t["PRODUCTO"] != "TOTAL"]
    dif = max(abs(t[m].sum() - tot.iloc[0][m]) for m in MES_TR + ["TOTAL"]) if len(tot) else None
    filas_mal = int((t[MES_TR].sum(axis=1) != t["TOTAL"]).sum())
    c = next((c for c in ctrl_final if c.get("AÑO") == anio), None)
    if c is None:
        c = {"ARCHIVO": t["FUENTE"].iloc[0], "TIPO": "TRANSCRITO", "AÑO": anio, "ESTADO": "SIN LÍQUIDOS"}
        ctrl_final.append(c)
    c.update({"LOTES_GAS": len(t), "GAS_MPC": float(t["TOTAL"].sum()), "DIF_MAX_GAS": dif,
              "ESTADO_GAS": "OK (transcrito)" if dif == 0 and filas_mal == 0 else "REVISAR"})
    anios_gas_tr.add(anio)
    for _, f in t.iterrows():
        for k, m in enumerate(MES_TR, 1):
            mpc = float(f[m])
            filas_gas.append({"AÑO": anio, "MES": k, "FECHA": f"01/{k:02d}/{anio}", "ZONA": zona_de(f["LOTE"]),
                              **dict(zip(("DEPARTAMENTO", "DEPARTAMENTO_FUENTE"), depto_de(f["LOTE"], zona_de(f["LOTE"])))),
                              "OPERADOR": norm_operador(f["OPERADOR"]), "OPERADOR_ORIGINAL": f["OPERADOR"],
                              "LOTE": f["LOTE"], "LOTE_HISTORICO": lote_historico(f["LOTE"]),
                              "LOTE_ORIGINAL": f["LOTE_ORIGINAL"], "PRODUCTO": "GAS NATURAL",
                              "MPC": mpc, "MMPCD": round(mpc / calendar.monthrange(anio, k)[1] / 1000, 3),
                              "ESTADO_FILA": "TRANSCRITO", "FUENTE": "TRANSCRIPCION " + f["FUENTE"]})
if anios_gas_tr:
    print(f"✍️ Gas natural tomado de transcripciones validadas: {', '.join(map(str, sorted(anios_gas_tr)))}")

prod = pd.DataFrame(filas)
ctrl = pd.DataFrame(ctrl_final + [c for c in control if c.get("ESTADO", "").startswith(("ERROR", "NO ES", "ESCANEADO", "FORMATO"))
                                  and c.get("AÑO") not in anios_transc])
hoy = date.today().strftime("%d/%m/%Y")
ctrl["FECHA_EXTRACCION"] = hoy
NOMBRE_DATOS = f"PP_produccion_lote_mes_{A_INI}-{A_FIN}.csv"
NOMBRE_CONTROL = f"CONTROL PP produccion_lote_mes_{A_INI}-{A_FIN}.csv"
prod.to_csv(os.path.join(ruta, NOMBRE_DATOS), index=False, encoding="utf-8-sig")
ctrl.to_csv(os.path.join(ruta, NOMBRE_CONTROL), index=False, encoding="utf-8-sig")
subir(os.path.join(ruta, NOMBRE_DATOS), DRIVE_FOLDER_ID, datos_drive)
subir(os.path.join(ruta, NOMBRE_CONTROL), DRIVE_FOLDER_CONTROL_ID, control_drive)
# Gas natural fiscalizado (2001 en adelante, de los PDF; 1994–2000, de las transcripciones validadas)
gas = pd.DataFrame(filas_gas)
NOMBRE_GAS = f"PP_produccion_gas_lote_mes_{A_INI}-{A_FIN}.csv"
if len(gas):
    gas.to_csv(os.path.join(ruta, NOMBRE_GAS), index=False, encoding="utf-8-sig")
    subir(os.path.join(ruta, NOMBRE_GAS), DRIVE_FOLDER_ID, datos_drive)

# ── 6.1 TABLAS ADICIONALES DE LOS ANUALES: POZOS, REGALÍAS, CANON, PRECIO DE CANASTA Y CONTRATOS ──
# Cada anual trae series (pozos: 10 años; canon: 10 años): para cada año manda la edición más reciente.
# Regalías y canasta son solo del año del anual. Todo se controla contra los totales que imprime el PDF.
ctrl_extra, extra_ok = [], {}
def _ultima_edicion(registros):
    """{año: edición} con la edición (año del anual) más reciente que trae cada año."""
    ed = {}
    for e, f in registros:
        ed[f["AÑO"]] = max(ed.get(f["AÑO"], 0), e)
    return ed
# Canon por departamento e ingresos para el Estado MENSUALES (resumen ejecutivo de cada estadística mensual).
# Se acumulan en Drive: cada corrida agrega o reemplaza los meses que vuelve a leer (manda el reporte más reciente).
def _acumular(nombre, nuevo, claves):
    if nombre in datos_drive:
        try:
            bajar_de_drive(datos_drive[nombre], os.path.join(ruta, nombre))
            viejo = pd.read_csv(os.path.join(ruta, nombre), encoding="utf-8-sig")
            if len(nuevo):
                k_n = set(map(tuple, nuevo[claves].astype(str).values))
                viejo = viejo[[tuple(x) not in k_n for x in viejo[claves].astype(str).values]]
            nuevo = pd.concat([viejo, nuevo], ignore_index=True)
        except Exception as e:
            print(f"⚠️ No se pudo leer {nombre} de Drive ({e}); se crea de nuevo")
    return nuevo.sort_values(claves).reset_index(drop=True) if len(nuevo) else nuevo

mensuales = sorted(((r["ANIO"], r["MES_REPORTE"]), r["mensual"]) for r in resultados if r.get("mensual"))
can_mes = pd.DataFrame(); ing_mes = pd.DataFrame()
if mensuales:
    fil_c, fil_i = {}, {}
    for (a, mm), x in mensuales:                       # del más antiguo al más reciente: el último reporte manda
        fuente = f"PP_MENSUAL_{a}_{mm:02d}.pdf"
        if x.get("error"):
            ctrl_extra.append({"TABLA": "RESUMEN MENSUAL", "EDICION": f"{a}-{mm:02d}", "AÑO": a, "ESTADO": f"ERROR: {x['error']}"})
        for f in x["canon"]:
            fil_c[(f["AÑO"], f["MES"], f["DEPARTAMENTO"])] = {**f, "FUENTE": fuente}
        for f in x["ingresos"]:
            fil_i[(f["AÑO"], f["MES"], f["MODALIDAD"])] = {**f, "FUENTE": fuente}
        for c in x["control"]:
            ok = c["TOTAL_PDF"] is not None and abs(c["SUMA"] - c["TOTAL_PDF"]) <= 0.05
            ctrl_extra.append({**c, "EDICION": f"{a}-{mm:02d}", "USADO": True, "ESTADO": "OK" if ok else "REVISAR"})
    if fil_c:
        can_mes = pd.DataFrame(fil_c.values())
        can_mes["FECHA"] = [f"01/{mm:02d}/{a}" for a, mm in zip(can_mes["AÑO"], can_mes["MES"])]
        can_mes = can_mes[["AÑO", "MES", "FECHA", "DEPARTAMENTO", "CANON_MM_SOLES", "FUENTE"]]
    if fil_i:
        ing_mes = pd.DataFrame(fil_i.values())
        ing_mes["FECHA"] = [f"01/{mm:02d}/{a}" for a, mm in zip(ing_mes["AÑO"], ing_mes["MES"])]
        ing_mes = ing_mes[["AÑO", "MES", "FECHA", "MODALIDAD", "INGRESO_MM_US$", "FUENTE"]]
can_mes = _acumular("PP_canon_departamento_mes.csv", can_mes, ["AÑO", "MES", "DEPARTAMENTO"])
ing_mes = _acumular("PP_ingresos_estado_mes.csv", ing_mes, ["AÑO", "MES", "MODALIDAD"])
if len(can_mes):
    extra_ok["PP_canon_departamento_mes"] = can_mes
if len(ing_mes):
    extra_ok["PP_ingresos_estado_mes"] = ing_mes

ediciones = sorted((r["ANIO"], r["extra"]) for r in resultados if r.get("extra"))
for e, x in ediciones:
    for tabla, msg in x["errores"].items():
        ctrl_extra.append({"TABLA": tabla.upper(), "EDICION": e, "AÑO": None, "ESTADO": f"ERROR: {msg}"})
if ediciones:
    # Pozos perforados por lote. Hasta las ediciones 2008–2011 los confirmatorios van dentro de los exploratorios.
    regs = [(e, f) for e, x in ediciones for f in x["pozos"][0]]
    ed = _ultima_edicion(regs)
    pozos = pd.DataFrame([{**f, "EDICION": e} for e, f in regs if ed[f["AÑO"]] == e])
    for e, x in ediciones:
        for c in x["pozos"][1]:
            ok = c["TOTAL_PDF"] is not None and c["SUMA_LOTES"] == c["TOTAL_PDF"]
            ctrl_extra.append({"TABLA": "POZOS " + c["TIPO"], "EDICION": e, "AÑO": c["AÑO"], "SUMA": c["SUMA_LOTES"],
                               "TOTAL_PDF": c["TOTAL_PDF"], "PAGINA": c["PAGINA"],
                               "USADO": ed.get(c["AÑO"]) == e, "ESTADO": "OK" if ok else "REVISAR"})
    if len(pozos):
        pozos["OPERADOR_ORIGINAL"] = pozos["OPERADOR"]
        pozos["OPERADOR"] = pozos["OPERADOR"].map(norm_operador)
        pozos["LOTE_HISTORICO"] = pozos["LOTE"].map(lote_historico)
        pozos["DEPARTAMENTO"] = [depto_de(l, zona_de(l))[0] for l in pozos["LOTE"]]
        pozos["FUENTE"] = "PP_ANUAL_" + pozos["EDICION"].astype(str) + ".pdf"
        pozos = (pozos.groupby(["AÑO", "TIPO", "DEPARTAMENTO", "OPERADOR", "OPERADOR_ORIGINAL", "LOTE", "LOTE_HISTORICO",
                                "FUENTE"], as_index=False)["POZOS"].sum()
                 .sort_values(["AÑO", "TIPO", "LOTE"]))
        extra_ok["PP_pozos_perforados_lote_anio"] = pozos
    # Regalías (ingresos al fisco) por lote e hidrocarburo, en US$
    # Regalías y canasta son solo del año del anual: el año es la edición (el título del anual 2022 dice «2021» por error)
    reg = pd.DataFrame([{**f, "AÑO": e, "AÑO_TITULO": f["AÑO"], "EDICION": e} for e, x in ediciones for f in x["regalias"][0]])
    for e, x in ediciones:
        for c in x["regalias"][1]:
            ok = c["TOTAL_PDF"] and abs(c["SUMA_LOTES"] - c["TOTAL_PDF"]) <= max(1.0, 0.001 * c["TOTAL_PDF"])
            ctrl_extra.append({"TABLA": "REGALIAS", "EDICION": e, "AÑO": e, "AÑO_TITULO": c["AÑO"], "SUMA": c["SUMA_LOTES"],
                               "TOTAL_PDF": c["TOTAL_PDF"], "PAGINA": c["PAGINA"], "USADO": True,
                               "ESTADO": "OK" if ok else "REVISAR"})
    if len(reg):
        reg["LOTE_HISTORICO"] = reg["LOTE"].map(lote_historico)
        # Operador de cada fila: el PDF distingue contratos del mismo lote igual que en la tabla de producción del mismo
        # anual («8» PPN en liquidación y «8.» Upland; «I (TT)» Petroperú e «I.» Inexport). Si el lote del PDF no basta,
        # se usa el único operador con producción del lote ese año; si hubo varios, queda «VARIOS OPERADORES».
        _base = lambda l: re.sub(r"\(T+\)$", "", str(l).replace(" ", ""))
        _ops = {}
        for r_ in resultados:
            if r_["TIPO"] == "ANUAL":
                for f in r_["filas"]:
                    o_ = norm_operador(f["OPERADOR"])
                    _ops.setdefault((r_["ANIO"], str(f["LOTE_ORIGINAL"]).replace(" ", "")), set()).add(o_)
                    _ops.setdefault((r_["ANIO"], "L:" + _base(f["LOTE"])), set()).add(o_)
        def _op_reg(e, lp, l):
            s_ = _ops.get((e, str(lp).replace(" ", "")))
            if not s_ or len(s_) > 1:
                s_ = _ops.get((e, "L:" + _base(l)), s_)
            return "" if not s_ else next(iter(s_)) if len(s_) == 1 else "VARIOS OPERADORES"
        reg["OPERADOR"] = [_op_reg(e, lp, l) for e, lp, l in zip(reg["EDICION"], reg["LOTE_PDF"], reg["LOTE"])]
        reg["DEPARTAMENTO"] = [depto_de(l, zona_de(l))[0] for l in reg["LOTE"]]
        reg["FUENTE"] = "PP_ANUAL_" + reg["EDICION"].astype(str) + ".pdf"
        extra_ok["PP_regalias_lote_anio"] = reg[["AÑO", "MODALIDAD", "HIDROCARBURO", "DEPARTAMENTO", "LOTE",
                                                 "LOTE_HISTORICO", "LOTE_PDF", "OPERADOR", "REGALIA_PCT", "REGALIA_US$", "FUENTE"]]
    # Canon y sobrecanon por departamento (millones de soles)
    regs = [(e, f) for e, x in ediciones for f in x["canon"][0]]
    ed = _ultima_edicion(regs)
    can = pd.DataFrame([{**f, "EDICION": e} for e, f in regs if ed[f["AÑO"]] == e])
    for e, x in ediciones:
        for c in x["canon"][1]:
            ok = c["TOTAL_PDF"] is not None and abs(c["SUMA_DEPARTAMENTOS"] - c["TOTAL_PDF"]) <= 0.05 + 0.001 * c["TOTAL_PDF"]
            ctrl_extra.append({"TABLA": "CANON", "EDICION": e, "AÑO": c["AÑO"], "SUMA": c["SUMA_DEPARTAMENTOS"],
                               "TOTAL_PDF": c["TOTAL_PDF"], "PAGINA": c["PAGINA"],
                               "USADO": ed.get(c["AÑO"]) == e, "ESTADO": "OK" if ok else "REVISAR"})
    if len(can):
        can["FUENTE"] = "PP_ANUAL_" + can["EDICION"].astype(str) + ".pdf"
        can = can.rename(columns={"CANON": "CANON_MM_SOLES"})
        can["CANON_PDF_ANUAL"] = can["CANON_MM_SOLES"]
        # Si el resumen mensual trae los 12 meses del año, su suma manda (el anual 2025 da 20,19 para Loreto y los
        # meses suman 213,84): se corrige y queda el valor del anual en CANON_PDF_ANUAL.
        if len(can_mes):
            cm = can_mes[can_mes["DEPARTAMENTO"] != "TOTAL"]
            n12 = cm.groupby(["AÑO", "DEPARTAMENTO"])["MES"].nunique()
            s12 = cm.groupby(["AÑO", "DEPARTAMENTO"])["CANON_MM_SOLES"].sum().round(2)
            for i, f in can.iterrows():
                k = (int(f["AÑO"]), sin_tildes(str(f["DEPARTAMENTO"])))
                if n12.get(k, 0) == 12:
                    dif = round(s12[k] - f["CANON_MM_SOLES"], 2)
                    ctrl_extra.append({"TABLA": f"CANON ANUAL vs SUMA MENSUAL {k[1]}", "EDICION": f["EDICION"], "AÑO": k[0],
                                       "SUMA": s12[k], "TOTAL_PDF": f["CANON_MM_SOLES"], "USADO": True,
                                       "ESTADO": "OK" if abs(dif) <= 0.1 else "CORREGIDO CON LA SUMA MENSUAL"})
                    if abs(dif) > 0.1:
                        can.loc[i, "CANON_MM_SOLES"] = s12[k]
                        can.loc[i, "FUENTE"] = f"SUMA MENSUAL ({can_mes.loc[(can_mes['AÑO'] == k[0]), 'FUENTE'].iloc[-1]})"
        extra_ok["PP_canon_departamento_anio"] = can[["AÑO", "DEPARTAMENTO", "CANON_MM_SOLES", "CANON_PDF_ANUAL", "FUENTE"]].sort_values(
            ["AÑO", "DEPARTAMENTO"])
    # Precio de la canasta por lote y mes (US$/bbl)
    cst = pd.DataFrame([{**f, "AÑO": e, "EDICION": e} for e, x in ediciones for f in x["canasta"][0]])
    for e, x in ediciones:
        for c in x["canasta"][1]:
            if c["PROMEDIO_PDF"] is not None:     # promedio del cuadro o, en 2007–2011, del gráfico de la misma página
                ok = any(x is not None and abs(x - c["PROMEDIO_PDF"]) <= 0.011
                         for x in (c["PROMEDIO_CALC"], c.get("PROMEDIO_CALC_12")))
            else:                                 # sin promedio impreso: se pide el año completo
                ok = c.get("MESES_PDF", c["MESES"]) == 12   # los meses con «-» cuentan: el lote empezó o paró
            ctrl_extra.append({"TABLA": "CANASTA " + c["LOTE"] + (f" ({c['VARIANTE']})" if c["VARIANTE"] else ""),
                               "EDICION": e, "AÑO": e, "AÑO_TITULO": c["AÑO"], "SUMA": c["PROMEDIO_CALC"], "TOTAL_PDF": c["PROMEDIO_PDF"],
                               "MESES": c["MESES"], "IMPUTADOS": c.get("IMPUTADOS", 0),
                               "PROMEDIO_FUENTE": c.get("PROMEDIO_FUENTE", "CUADRO" if c["PROMEDIO_PDF"] else ""),
                               "PAGINA": c["PAGINA"], "USADO": True,
                               "ESTADO": "OK" if ok else "REVISAR"})
    if len(cst):
        cst["FECHA"] = [f"01/{m:02d}/{a}" for a, m in zip(cst["AÑO"], cst["MES"])]
        cst["FUENTE"] = "PP_ANUAL_" + cst["EDICION"].astype(str) + ".pdf"
        cst = cst.drop_duplicates(["AÑO", "MES", "LOTE", "VARIANTE"])
        if "ESTADO" not in cst:
            cst["ESTADO"] = "OK"
        cst["ESTADO"] = cst["ESTADO"].fillna("OK")
        extra_ok["PP_precio_canasta_lote_mes"] = cst[["AÑO", "MES", "FECHA", "LOTE", "VARIANTE", "LOTE_PDF",
                                                      "PRECIO_US$_BBL", "ESTADO", "FUENTE"]]
        # Valor de la producción fiscalizada de líquidos = barriles × precio de canasta del lote y mes
        if len(prod):
            precio = {(a, m, l, v): p for a, m, l, v, p in
                      cst[["AÑO", "MES", "LOTE", "VARIANTE", "PRECIO_US$_BBL"]].itertuples(index=False)}
            orden = {"PETROLEO": ("P", ""), "LGN": ("LGN", "L", ""), "CONDENSADO": ("LGN", "L", "")}
            def _precio(a, m, lote, producto):
                base = re.sub(r"\(T+\)$", "", lote)
                # los anuales antiguos dan un solo precio para la zona Noroeste
                cands = (lote, base) + (("NOROESTE",) if zona_de(base) == "NOROESTE" else ())
                for v in orden.get(producto, ("",)):
                    for l in cands:
                        if (a, m, l, v) in precio:
                            return precio[(a, m, l, v)]
                return None
            val = prod[["AÑO", "MES", "FECHA", "DEPARTAMENTO", "OPERADOR", "LOTE", "LOTE_HISTORICO", "PRODUCTO",
                        "BARRILES"]].copy()
            val = val[val["AÑO"].isin(set(cst["AÑO"]))]
            val["PRECIO_US$_BBL"] = [_precio(a, m, l, p) for a, m, l, p in
                                     val[["AÑO", "MES", "LOTE", "PRODUCTO"]].itertuples(index=False)]
            val["VALOR_US$"] = (val["BARRILES"] * val["PRECIO_US$_BBL"]).round(2)
            val["ESTADO_PRECIO"] = ["OK" if p is not None and p == p else ("SIN PRODUCCION" if b == 0 else "SIN PRECIO")
                                    for p, b in zip(val["PRECIO_US$_BBL"], val["BARRILES"])]
            extra_ok["PP_valor_produccion_lote_mes"] = val
# Contratos: empresas socias con su participación, suscripción, término, área y modalidad (vigentes y terminados en el año).
# Se acumulan en Drive por edición: cada corrida reemplaza las ediciones que vuelve a leer.
con = pd.DataFrame([{**f, "EDICION": e} for e, x in ediciones for f in x.get("contratos", ([], []))[0]])
for e, x in ediciones:
    for c in x.get("contratos", ([], []))[1]:
        ctrl_extra.append({"TABLA": f"CONTRATOS {c['ESTADO_CONTRATO']} {c['FASE']}", "EDICION": e, "AÑO": e,
                           "SUMA": c["LOTES"], "TOTAL_PDF": c["TOTAL_PDF"], "PAGINA": c["PAGINA"], "USADO": True,
                           "ESTADO": "OK" if c["TOTAL_PDF"] is not None and c["LOTES"] == c["TOTAL_PDF"] else "REVISAR"})
if len(con):
    con["DEPARTAMENTO"] = [depto_de(l, zona_de(l))[0] for l in con["LOTE"]]
    con["LOTE_HISTORICO"] = con["LOTE"].map(lote_historico)
    con["FUENTE"] = "PP_ANUAL_" + con["EDICION"].astype(str) + ".pdf"
    con = con[["EDICION", "ESTADO", "FASE", "DEPARTAMENTO", "LOTE", "LOTE_HISTORICO", "OPERADOR", "EMPRESA",
               "PARTICIPACION_PCT", "FECHA_SUSCRIPCION", "FECHA_TERMINO", "AREA_HA", "MODALIDAD", "PAGINA", "FUENTE"]]
con = _acumular("PP_contratos.csv", con, ["EDICION"])
if len(con) and ediciones:
    con["LOTE"] = con["LOTE"].astype(str)
    con = con.sort_values(["EDICION", "ESTADO", "FASE", "LOTE"], kind="mergesort").reset_index(drop=True)
    extra_ok["PP_contratos"] = con
for nombre, df_x in extra_ok.items():
    archivo = f"{nombre}.csv"
    df_x.to_csv(os.path.join(ruta, archivo), index=False, encoding="utf-8-sig")
    subir(os.path.join(ruta, archivo), DRIVE_FOLDER_ID, datos_drive)
ctrl_x = pd.DataFrame(ctrl_extra)
if len(ctrl_x):
    ctrl_x["FECHA_EXTRACCION"] = hoy
    NOMBRE_CONTROL_EXTRA = "CONTROL PP tablas_adicionales.csv"
    ctrl_x.to_csv(os.path.join(ruta, NOMBRE_CONTROL_EXTRA), index=False, encoding="utf-8-sig")
    subir(os.path.join(ruta, NOMBRE_CONTROL_EXTRA), DRIVE_FOLDER_CONTROL_ID, control_drive)

# ── 7. RESUMEN ───────────────────────────────────────────────────────────────
print("═" * 70)
print(f"Años con datos: {prod['AÑO'].nunique() if len(prod) else 0} · filas: {len(prod):,}")
if len(ctrl):
    display(ctrl["ESTADO"].value_counts().to_frame("archivos"))
    rev = ctrl[~ctrl["ESTADO"].isin(["OK", "OK (transcrito)"])]
    if len(rev):
        antiguos = rev[rev["ESTADO"].str.startswith(("ESCANEADO", "FORMATO"), na=False)]
        if len(antiguos):
            print(f"🗂️ {len(antiguos)} PDF antiguos (escaneados): quedan guardados en DATA/PDF para la etapa de OCR · "
                  f"años: {', '.join(str(a) for a in sorted(antiguos['AÑO'].dropna().astype(int)))}")
        otros = rev.drop(antiguos.index)
        if len(otros):
            print("⚠️ Archivos a revisar (totales que no cuadran o error): mándalos para ajustar el extractor")
            display(otros)
if len(prod):
    lor = prod[(prod["DEPARTAMENTO"] == "LORETO") & (prod["PRODUCTO"] == "PETROLEO")]
    if len(lor):
        print("🛢️ Loreto · petróleo fiscalizado por lote histórico (millones de barriles; 1-AB y 192 = mismo campo):")
        tabla = (lor.pivot_table(index="LOTE_HISTORICO", columns="AÑO", values="BARRILES", aggfunc="sum") / 1e6).round(2)
        # 0 en los años en que el lote ya existía pero no figura en el PDF; vacío solo antes de su primer año
        tabla = tabla.where(tabla.notna() | ~tabla.notna().cummax(axis=1), 0.0)
        ult = int(lor["AÑO"].max()); meses_ult = int(lor.loc[lor["AÑO"] == ult, "MES"].max())
        if meses_ult < 12:
            tabla = tabla.rename(columns={ult: f"{ult} (ene–{MESES[meses_ult - 1][:3].lower()})"})
        tabla["ACUMULADO"] = tabla.sum(axis=1)
        display(tabla.sort_values("ACUMULADO", ascending=False))
if len(gas):
    print(f"🔥 Gas natural fiscalizado: {gas['AÑO'].nunique()} años · {gas['LOTE'].nunique()} lotes · {NOMBRE_GAS}")
    if "ESTADO_GAS" in ctrl:
        display(ctrl["ESTADO_GAS"].dropna().value_counts().to_frame("años (gas)"))
    tg = (gas.pivot_table(index="LOTE", columns="AÑO", values="MPC", aggfunc="sum") / 1e6).round(1)
    ult = int(gas["AÑO"].max()); meses_ult = int(gas.loc[gas["AÑO"] == ult, "MES"].max())
    if meses_ult < 12:
        tg = tg.rename(columns={ult: f"{ult} (ene–{MESES[meses_ult - 1][:3].lower()})"})
    tg["ACUMULADO"] = tg.sum(axis=1)
    print("Gas natural por lote (miles de millones de pies cúbicos = BCF):")
    display(tg.sort_values("ACUMULADO", ascending=False))
if extra_ok:
    print("═" * 70)
    print("📑 TABLAS ADICIONALES DE LOS ANUALES")
    if {"TABLA", "ESTADO", "EDICION"} <= set(ctrl_x.columns):   # sin anuales en esta corrida el control viene vacío
        display(ctrl_x.assign(TABLA=ctrl_x["TABLA"].str.split(" ").str[0])
                .pivot_table(index="TABLA", columns="ESTADO", values="EDICION", aggfunc="count", fill_value=0))
    rev_x = (ctrl_x[(ctrl_x["ESTADO"] != "OK") & (ctrl_x.get("USADO", True) != False)]
             if "ESTADO" in ctrl_x.columns else ctrl_x.iloc[0:0])
    if len(rev_x):
        print("⚠️ A revisar (la suma de lotes no cuadra con el total del PDF, o faltan meses):")
        display(rev_x)
    if "PP_pozos_perforados_lote_anio" in extra_ok:
        pz = extra_ok["PP_pozos_perforados_lote_anio"]
        print("🛠️ Pozos perforados por año (todas las zonas):")
        display(pz.pivot_table(index="AÑO", columns="TIPO", values="POZOS", aggfunc="sum", fill_value=0).T)
        lp = pz[pz["DEPARTAMENTO"] == "LORETO"]
        if len(lp):
            print("🛠️ Loreto · pozos perforados por lote (exploratorios + confirmatorios | desarrollo):")
            lp = lp.assign(TIPO=lp["TIPO"].replace({"CONFIRMATORIO": "EXPLORATORIO"}))
            display(lp.pivot_table(index=["LOTE_HISTORICO", "TIPO"], columns="AÑO", values="POZOS", aggfunc="sum",
                                   fill_value=0))
    if "PP_regalias_lote_anio" in extra_ok:
        rg = extra_ok["PP_regalias_lote_anio"]
        print("💵 Regalías por departamento (millones de US$):")
        rg = rg.assign(DEPARTAMENTO=rg["DEPARTAMENTO"].fillna("").replace("", "SIN DEPARTAMENTO (zócalo u otros)"))
        display((rg.pivot_table(index="DEPARTAMENTO", columns="AÑO", values="REGALIA_US$", aggfunc="sum") / 1e6).round(1))
    if "PP_canon_departamento_mes" in extra_ok:
        cmes = extra_ok["PP_canon_departamento_mes"]
        ult = cmes[["AÑO", "MES"]].drop_duplicates().sort_values(["AÑO", "MES"]).tail(12)
        cm12 = cmes.merge(ult, on=["AÑO", "MES"])
        cm12 = cm12.assign(PERIODO=[f"{mm:02d}/{a}" for a, mm in zip(cm12["AÑO"], cm12["MES"])])
        print(f"🏛️ Canon y sobrecanon mensual por departamento (millones de soles; {len(cmes[['AÑO','MES']].drop_duplicates())} meses acumulados, últimos 12):")
        display(cm12.pivot_table(index="DEPARTAMENTO", columns="PERIODO", values="CANON_MM_SOLES", sort=False))
    if "PP_ingresos_estado_mes" in extra_ok:
        im = extra_ok["PP_ingresos_estado_mes"]
        print("💵 Ingresos para el Estado por mes (millones de US$; regalías y regalía equivalente, total nacional):")
        display(im.assign(PERIODO=[f"{mm:02d}/{a}" for a, mm in zip(im["AÑO"], im["MES"])])
                .pivot_table(index="MODALIDAD", columns="PERIODO", values="INGRESO_MM_US$", sort=False))
    if "PP_canon_departamento_anio" in extra_ok:
        print("🏛️ Canon y sobrecanon por departamento (millones de soles):")
        display(extra_ok["PP_canon_departamento_anio"].pivot_table(index="DEPARTAMENTO", columns="AÑO",
                                                                    values="CANON_MM_SOLES").round(1))
    if "PP_valor_produccion_lote_mes" in extra_ok:
        vp = extra_ok["PP_valor_produccion_lote_mes"]
        sin = vp[vp["ESTADO_PRECIO"] == "SIN PRECIO"]
        print(f"💰 Valor de la producción de líquidos (millones de US$){'' if not len(sin) else f' · {len(sin)} filas sin precio de canasta'}:")
        vp = vp.assign(DEPARTAMENTO=vp["DEPARTAMENTO"].fillna("").replace("", "SIN DEPARTAMENTO (zócalo u otros)"))
        display((vp.pivot_table(index="DEPARTAMENTO", columns="AÑO", values="VALOR_US$", aggfunc="sum") / 1e6).round(1))
    if "PP_contratos" in extra_ok:
        ct = extra_ok["PP_contratos"]
        ult = int(ct["EDICION"].max())
        v = ct[(ct["EDICION"] == ult) & (ct["ESTADO"] == "VIGENTE") & (ct["FASE"] == "EXPLOTACION")]
        print(f"🏢 Contratos de explotación vigentes al 31/12/{ult} ({v['LOTE'].nunique()} lotes; "
              f"ediciones en la tabla: {', '.join(map(str, sorted(ct['EDICION'].unique())))}):")
        display(v.assign(SOCIOS=[f"{e} ({p:g} %)" if p == p else e for e, p in zip(v["EMPRESA"], v["PARTICIPACION_PCT"])])
                 .groupby(["DEPARTAMENTO", "LOTE", "FECHA_SUSCRIPCION", "FECHA_TERMINO"], sort=False)["SOCIOS"]
                 .agg(" / ".join).reset_index())
print(f"   Datos:   https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
print(f"   Control: https://drive.google.com/drive/folders/{DRIVE_FOLDER_CONTROL_ID}")

# ── 8. PRODUCCIÓN DIARIA (página «Producción Diaria»: líquidos en bbl, gas en Mcf y MMBtu) ──
# Se acumula en DATA/PP_produccion_diaria.csv; cada corrida reemplaza los meses que vuelve a leer.
PAGINA_DIARIA = ("https://www.perupetro.com.pe/wps/portal/corporativo/PerupetroSite/estadisticas/"
                 "producci%C3%B3n%20hidrocarburos/producci%C3%B3n%20diaria/!ut/p/z1/04_Sj9CPykssy0xPLMnMz0vMAfIjo8zi_Yx"
                 "cTTw8TAy93AN8LQwCTUJcvEKADCdfI_1wsAIDHMDRQD8Kt35DHxOofhwKAo3MKNIPUhBFjPvxKIjC771w_ShCSqLwucHM3xS_AnAY4"
                 "FHgbm5krF-QGxoaGmGQ6ZmuqAgAQ52_bw!!/dz/d5/L2dBISEvZ0FBIS9nQSEh/")
NOMBRE_DIARIA = "PP_produccion_diaria.csv"
NOMBRE_CONTROL_DIARIA = "CONTROL PP produccion_diaria.csv"
if chk_diaria.value:
    if "PDF_DIARIA" not in datos_drive:
        datos_drive["PDF_DIARIA"] = drive.files().create(body={"name": "PDF_DIARIA", "parents": [DRIVE_FOLDER_ID],
                                                               "mimeType": "application/vnd.google-apps.folder"},
                                                         fields="id", supportsAllDrives=True).execute()["id"]
    PDF_DIARIA_ID = datos_drive["PDF_DIARIA"]
    pdf_diaria_drive = archivos_en(PDF_DIARIA_ID)
    # ── 8.1 ENLACES DE LA PÁGINA «PRODUCCIÓN DIARIA» ─────────────────────────────
    def tipo_enlace(href, texto):
        """LIQUIDOS, GAS (Mcf; trae también la hoja en MMBtu) o GAS M3 (m3/d)."""
        t = _sin_tildes(html.unescape(re.sub(r"<[^>]+>", " ", texto)) + " " + unquote(href).replace("+", " "))
        if "LIQUIDO" in t:
            return "LIQUIDOS"
        if "GAS" in t:
            return "GAS M3" if re.search(r"M3", t) else "GAS"
        return None

    try:
        sesion_d = requests.Session()             # la sesión guarda la cookie de idioma del portal
        pagina_d = sesion_d.get(PAGINA_DIARIA, headers=CAB, timeout=60, verify=VERIFY).text
        if "Liquid Production" in pagina_d or "Producción al cierre" not in pagina_d:
            # todavía en inglés: segunda lectura con la cookie ya puesta y el idioma pedido otra vez
            pagina_d = sesion_d.get(PAGINA_DIARIA, headers={**CAB, "Cookie": "locale=es; lang=es"}, timeout=60,
                                    verify=VERIFY).text
        if "Liquid Production" in pagina_d:
            print("⚠️ El portal devolvió la página en inglés (sin la tabla de meses): solo se toma el mes en curso.")
    except Exception as e:
        pagina_d = ""
        print(f"⚠️ No se pudo leer la página de PERUPETRO ({e}). Se usarán solo los PDF que ya estén en Drive/PDF_DIARIA.")
    def anclas(texto_html):
        """(href, texto) de cada enlace; acepta href entre comillas dobles, simples o sin comillas."""
        patron = r"""<a\b[^>]*?href\s*=\s*(?:"([^"]*)"|'([^']*)'|([^\s>]+))[^>]*>(.*?)</a>"""
        return [(html.unescape(a or b or c), t) for a, b, c, t in re.findall(patron, texto_html, re.S | re.I)]

    def es_pdf(href):
        h = href.lower()
        return "/wps/wcm/connect/" in h or ".pdf" in h or "ajperes" in h

    # la tabla de meses puede venir dentro de un iframe o de un contenido aparte: también se leen esas páginas
    paginas_d = [pagina_d]
    for src in re.findall(r"""<iframe\b[^>]*?src\s*=\s*["']([^"']+)["']""", pagina_d, re.I):
        try:
            paginas_d.append(requests.get(urljoin(PAGINA_DIARIA, html.unescape(src)), headers=CAB, timeout=60,
                                          verify=VERIFY).text)
        except Exception as e:
            print(f"   ↻ iframe no leído ({type(e).__name__})")
    enlaces_d, todos_d = {}, []
    for pg_html in paginas_d:
        for href, texto in anclas(pg_html):
            txt_ = _sin_tildes(re.sub(r"<[^>]+>", " ", texto))
            # por la dirección (PDF) o por el texto del enlace («Líquidos en b/d», «Gas en Mcfd»), sea cual sea la dirección;
            # al descargar solo se acepta lo que de verdad es un PDF
            if es_pdf(href) or re.search(r"LIQUIDOS\s+(EN\s+)?B/?D|GAS\s+(EN\s+)?(MCF|M3)", txt_):
                todos_d.append((href, re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", texto)).strip()))
                tipo = tipo_enlace(href, texto)
                if tipo:
                    enlaces_d.setdefault(urljoin(PAGINA_DIARIA, href), tipo)
    # diagnóstico: guarda la página en CONTROL y muestra los enlaces vistos si hay pocos reportes
    open(os.path.join(ruta, "PP_pagina_produccion_diaria.html"), "w", encoding="utf-8").write("\n<!-- ===== -->\n".join(paginas_d))
    subir(os.path.join(ruta, "PP_pagina_produccion_diaria.html"), DRIVE_FOLDER_CONTROL_ID, control_drive, mime="text/html")
    if len(enlaces_d) < 4:
        print(f"🔎 Diagnóstico: la página trae {len(todos_d)} enlaces a reportes (copia en CONTROL/PP_pagina_produccion_diaria.html):")
        for href, texto in todos_d[:25]:
            print(f"   · {texto[:40]!r} → {href[:110]}")
    print(f"🔗 Reportes diarios en la página: {sum(1 for t in enlaces_d.values() if t == 'LIQUIDOS')} de líquidos, "
          f"{sum(1 for t in enlaces_d.values() if t == 'GAS')} de gas en Mcf y {sum(1 for t in enlaces_d.values() if t == 'GAS M3')} de gas en m3")


    # ── 8.2 DESCARGA, LECTURA Y CONTROL ──────────────────────────────────────────
    # Cada mes se guarda como PP_DIARIA_<LIQUIDOS|GAS>_<año>_<mes>.pdf; el del mes en curso se reemplaza en cada corrida
    # (PERUPETRO lo actualiza día a día). Los PDF subidos a mano a DATA/PDF_DIARIA también se leen.
    leidos_d, control_d = {}, []

    def procesar_diario(local, origen):
        try:
            paginas = extraer_diario(local)
        except Exception as e:
            control_d.append({"ARCHIVO": origen, "ESTADO": f"ERROR: {type(e).__name__}: {e}"}); return None
        if not paginas:
            control_d.append({"ARCHIVO": origen, "ESTADO": "NO ES REPORTE DIARIO"}); return None
        p0 = paginas[0]
        clase = "LIQUIDOS" if p0["FLUIDO"] == "LIQUIDOS" else ("GAS_M3" if p0["UNIDAD"] == "M3" else "GAS")
        nombre = f"PP_DIARIA_{clase}_{p0['AÑO']}_{p0['MES']:02d}.pdf"
        for pag in paginas:
            filas, ctrl = a_filas(pag, nombre)
            clave = (pag["FLUIDO"], pag["UNIDAD"], pag["AÑO"], pag["MES"])
            if clave in leidos_d and leidos_d[clave]["REPORTE_AL"] >= pag["REPORTE_AL"][6:] + pag["REPORTE_AL"][3:5] + pag["REPORTE_AL"][:2]:
                continue                               # ya hay uno igual o más reciente
            c = pd.DataFrame(ctrl)
            leidos_d[clave] = {"REPORTE_AL": pag["REPORTE_AL"][6:] + pag["REPORTE_AL"][3:5] + pag["REPORTE_AL"][:2], "filas": filas,
                             "ctrl": {"ARCHIVO": nombre, "FLUIDO": pag["FLUIDO"], "UNIDAD": pag["UNIDAD"], "AÑO": pag["AÑO"],
                                      "MES": pag["MES"], "REPORTE_AL": pag["REPORTE_AL"], "DIAS": len(c),
                                      "DIAS_OK": int((c["ESTADO"] == "OK").sum()) if len(c) else 0,
                                      "DIF_MAX": float(c["DIF_MAX"].max()) if len(c) else None,
                                      "LOTES": len({f["LOTE"] for f in filas}),
                                      "ESTADO": "OK" if len(c) and (c["ESTADO"] == "OK").all() else "REVISAR"}}
        return nombre

    for n_, (url, tipo) in enumerate(enlaces_d.items(), 1):
        contenido = descargar(url)
        if contenido is None:
            control_d.append({"ARCHIVO": url, "ESTADO": "ERROR DE DESCARGA"}); print(f"❌ [{n_}] {url[:90]}"); continue
        local = os.path.join(ruta, f"tmp_d{n_}.pdf"); open(local, "wb").write(contenido)
        nombre = procesar_diario(local, url)
        if nombre:
            os.replace(local, os.path.join(ruta, nombre))
            subir(os.path.join(ruta, nombre), PDF_DIARIA_ID, pdf_diaria_drive, mime="application/pdf")
            print(f"✅ [{n_}/{len(enlaces_d)}] {nombre}")
        time.sleep(PAUSA)
    # PDF ya guardados (o subidos a mano) que no vinieron en esta corrida
    nuevos_d = {c["ctrl"]["ARCHIVO"] for c in leidos_d.values()}
    for nombre_d, fid in pdf_diaria_drive.items():
        if nombre_d.lower().endswith(".pdf") and nombre_d not in nuevos_d:
            local = os.path.join(ruta, nombre_d); bajar_de_drive(fid, local); procesar_diario(local, nombre_d)

    # ── 8.3 SERIE ACUMULADA EN DRIVE ──────────────────────────────────────────────
    nuevas_d = pd.DataFrame([f for v in leidos_d.values() for f in v["filas"]])
    if len(nuevas_d):
        # el reporte en m3 trae los nombres de operador cortados («N EXPORT»): se toman del reporte en Mcf del mismo mes
        op_mcf = (nuevas_d[nuevas_d["UNIDAD"] == "MCF"].drop_duplicates(["AÑO", "MES", "LOTE"])
                  .set_index(["AÑO", "MES", "LOTE"])["OPERADOR"].to_dict())
        es_m3 = nuevas_d["UNIDAD"] == "M3"
        nuevas_d.loc[es_m3, "OPERADOR"] = [op_mcf.get(k, o) for k, o in zip(zip(nuevas_d.loc[es_m3, "AÑO"], nuevas_d.loc[es_m3, "MES"],
                                                                                nuevas_d.loc[es_m3, "LOTE"]), nuevas_d.loc[es_m3, "OPERADOR"])]
        nuevas_d["DEPARTAMENTO"] = nuevas_d["LOTE"].map(lambda l: DEPTO_LOTE.get(l, DEPTO_LOTE.get(re.sub(r"\(T+\)$", "", l), "")))
    previa_d = pd.DataFrame()
    if NOMBRE_DIARIA in datos_drive:
        bajar_de_drive(datos_drive[NOMBRE_DIARIA], os.path.join(ruta, NOMBRE_DIARIA))
        previa_d = pd.read_csv(os.path.join(ruta, NOMBRE_DIARIA), encoding="utf-8-sig", dtype={"LOTE": str, "LOTE_ORIGINAL": str})
    if len(previa_d) and len(nuevas_d):                   # los meses leídos ahora reemplazan a los guardados
        k_nuevas_d = set(zip(nuevas_d["FLUIDO"], nuevas_d["UNIDAD"], nuevas_d["AÑO"], nuevas_d["MES"]))
        previa_d = previa_d[[k not in k_nuevas_d for k in zip(previa_d["FLUIDO"], previa_d["UNIDAD"], previa_d["AÑO"], previa_d["MES"])]]
    diaria = pd.concat([previa_d, nuevas_d], ignore_index=True)
    if len(diaria):
        diaria["_f"] = pd.to_datetime(diaria["FECHA"], format="%d/%m/%Y")
        diaria = diaria.sort_values(["_f", "FLUIDO", "UNIDAD", "PRODUCTO", "LOTE"]).drop(columns="_f")
        diaria.to_csv(os.path.join(ruta, NOMBRE_DIARIA), index=False, encoding="utf-8-sig")
        subir(os.path.join(ruta, NOMBRE_DIARIA), DRIVE_FOLDER_ID, datos_drive)
    ctrl_d = pd.DataFrame([v["ctrl"] for v in leidos_d.values()] + control_d)
    for c_ in ("AÑO", "MES", "DIAS", "DIAS_OK", "LOTES"):
        if c_ in ctrl_d:
            ctrl_d[c_] = ctrl_d[c_].astype("Int64")
    ctrl_d["FECHA_EXTRACCION"] = date.today().strftime("%d/%m/%Y")
    ctrl_d.to_csv(os.path.join(ruta, NOMBRE_CONTROL_DIARIA), index=False, encoding="utf-8-sig")
    subir(os.path.join(ruta, NOMBRE_CONTROL_DIARIA), DRIVE_FOLDER_CONTROL_ID, control_drive)

    # ── 8.4 RESUMEN ───────────────────────────────────────────────────────────────
    print("═" * 70)
    print("📅 PRODUCCIÓN DIARIA")
    if len(ctrl_d):
        display(ctrl_d.drop(columns=["FECHA_EXTRACCION"]).sort_values(["AÑO", "MES", "FLUIDO"], na_position="first"))
    if len(diaria):
        print(f"📅 Serie diaria: {diaria['FECHA'].nunique()} días · {len(diaria):,} filas · {NOMBRE_DIARIA}")
        ult_d = diaria[["AÑO", "MES"]].drop_duplicates().sort_values(["AÑO", "MES"]).iloc[-1]
        m_d = diaria[(diaria["AÑO"] == ult_d["AÑO"]) & (diaria["MES"] == ult_d["MES"])]
        dias_d = m_d["DIA"].max()
        liq_d = m_d[m_d["UNIDAD"] == "BBL"].groupby(["PRODUCTO", "LOTE", "OPERADOR"])["VALOR"].sum() / dias_d
        gas_d = m_d[m_d["UNIDAD"] == "MCF"].groupby(["LOTE", "OPERADOR"])["VALOR"].sum() / dias_d / 1000
        print(f"Promedio diario de {MESES_ES[int(ult_d['MES']) - 1].lower()} {int(ult_d['AÑO'])} (días 1–{dias_d}):")
        if len(liq_d):
            display(liq_d[liq_d > 0].round(0).sort_values(ascending=False).to_frame("bpd"))
        if len(gas_d):
            display(gas_d[gas_d > 0].round(2).sort_values(ascending=False).to_frame("MMPCD"))


# ── 9. EXCEL DE CANON (páginas «Canon y Sobre Canon» y «Canon Municipalidades») ──
# Determinación «Canon Petrolero <año>»: canon y sobrecanon por lote y mes (US$) y la parte de cada departamento.
# Distribución «Canon por Año <departamento>»: canon mensual 1993–hoy por beneficiario (gobierno regional, universidades…).
# Municipalidades «Canon por Año <departamento>»: año en curso por municipalidad provincial y distrital.
# Se guardan en DATA/CANON_EXCEL. Los de años cerrados se bajan una sola vez; el año en curso, la distribución y las
# municipalidades se vuelven a bajar en cada corrida (PERUPETRO los actualiza cada mes). Respaldo: poner los Excel a mano
# en esa carpeta (el control los lista como «MANUAL»).
PAGINA_CANON = ("https://www.perupetro.com.pe/wps/portal/corporativo/PerupetroSite/estadisticas/canon%20sobre%20canon/!ut/"
                "p/z1/pZJNDoIwEEbP4gmmrW1hS6PyEypgMMBsDCvTRNGF8fxSdOOCwcRZtcl7mZmvBYQWcOif7tw_3G3oL-O9Q33ai61MEsmzuLQhq2S"
                "9yerxYIyGZgLYTEUMkPAT9fFngEpoyue5pH0PEP6BC9r3ADW_XfA98OUXcblj2lijyjRkYb6wvw8If8mfaPBP_r4B0s_bAC4hSM2gC0UD"
                "0x8ggNgEggYCsYb79fiulrnURasX6dHDBQ!!/dz/d5/L2dBISEvZ0FBIS9nQSEh/")
NOMBRE_CONTROL_CANON = "CONTROL PP canon_excel.csv"
DEPS_CANON = ["UCAYALI", "PIURA", "HUANUCO", "TUMBES", "LORETO", "CUSCO"]

def anclas_canon(texto_html):
    """(href, texto limpio) de cada enlace de la página. Si el enlace no tiene texto (un ícono), se usa su title/alt."""
    patron = r"""(<a\b[^>]*?href\s*=\s*(?:"([^"]*)"|'([^']*)'|([^\s>]+))[^>]*>(.*?)</a>)"""
    limpio = lambda t: re.sub(r"\s+", " ", sin_tildes(html.unescape(t))).strip()
    salida = []
    for todo, a, b, c, t in re.findall(patron, texto_html, re.S | re.I):
        texto = limpio(re.sub(r"<[^>]+>", " ", t))
        if not texto:
            texto = limpio(" ".join(re.findall(r"""(?:title|alt)\s*=\s*["']([^"']*)["']""", todo, re.I)))
        salida.append((html.unescape(a or b or c), texto))
    return salida

def departamento_excel(contenido):
    """(departamento o None, cabecera leída). Busca «DEPARTAMENTO DE <X>» y, si no está, el departamento productor
    que más se nombra en las primeras filas de todas las hojas."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")                   # avisos de openpyxl por encabezados de impresión
            hojas = pd.read_excel(io.BytesIO(contenido), sheet_name=None, header=None, nrows=30)
    except Exception as e:
        return None, f"no se pudo leer ({type(e).__name__})"
    textos = [sin_tildes(" ".join(str(v) for v in h.values.ravel() if pd.notna(v))) for h in hojas.values()]
    cabecera = re.sub(r"\s+", " ", textos[0] if textos else "")[:200]
    for t in textos:
        m = re.search(r"DEPARTAMENTO\s*(?:DE\s+|DEL\s+|:\s*)?([A-Z]+)", t)
        if m and m.group(1) in DEPS_CANON:
            return m.group(1), cabecera
    cuenta = {d: sum(len(re.findall(r"\b" + d + r"\b", t)) for t in textos) for d in DEPS_CANON}
    mejor = max(cuenta, key=cuenta.get)
    return (mejor if cuenta[mejor] else None), cabecera

def clasificar_canon(texto, href, pagina):
    """(tipo, año, departamento, nombre en Drive) o None. Se clasifica por el texto del enlace."""
    t = texto + " " + sin_tildes(unquote(href).replace("+", " "))
    m = re.search(r"CANON\s+PETROLERO\s+((?:19|20)\d\d)", t)
    if m:
        return "DETERMINACION", int(m.group(1)), "", f"PP_CANON_DETERMINACION_{m.group(1)}"
    m = re.search(r"CANON\s+POR\s+A.?O\s+([A-Z ]+?)\s*$", texto) or re.search(r"CANON\s+POR\s+A.?O\s+(\w+)", t)
    if m:
        dep = next((d for d in DEPS_CANON if d in m.group(1)), m.group(1).strip().split(" ")[0])
        tipo = "MUNICIPAL" if pagina == "MUNICIPALIDADES" else "DISTRIBUCION"
        return tipo, None, dep, f"PP_CANON_{tipo}_{dep}"
    return None

def bajar_canon(url):
    """(contenido, error): hasta 3 intentos."""
    err = ""
    for i in range(1, 4):
        try:
            r = sesion_c.get(url, headers=CAB, timeout=120, verify=VERIFY)
            if r.status_code == 200:
                return r.content, ""
            err = f"HTTP {r.status_code}"
        except Exception as e:
            err = type(e).__name__
        time.sleep(3 * i)
    return None, err

def tipo_excel(contenido):
    """Extensión según el contenido: xlsx (zip), xls (OLE) o None si no es un Excel (p. ej., una página de error)."""
    if contenido[:2] == b"PK":
        return ".xlsx"
    if contenido[:4] == b"\xd0\xcf\x11\xe0":
        return ".xls"
    return None

if chk_canon_xls.value:
    print("\n" + "═" * 70 + "\n💰 EXCEL DE CANON (Determinación, Distribución y Municipalidades)")
    if "CANON_EXCEL" not in datos_drive:
        datos_drive["CANON_EXCEL"] = drive.files().create(body={"name": "CANON_EXCEL", "parents": [DRIVE_FOLDER_ID],
                                                                "mimeType": "application/vnd.google-apps.folder"},
                                                          fields="id", supportsAllDrives=True).execute()["id"]
    CANON_FOLDER_ID = datos_drive["CANON_EXCEL"]
    canon_drive = archivos_en(CANON_FOLDER_ID)
    ctrl_canon, paginas_canon = [], {"CANON Y SOBRE CANON": PAGINA_CANON}
    sesion_c = requests.Session()
    htmls = {}
    try:
        htmls["CANON Y SOBRE CANON"] = sesion_c.get(PAGINA_CANON, headers=CAB, timeout=60, verify=VERIFY).text
        # la página «Canon Municipalidades»: por el enlace del menú (texto o dirección) o, si no está, por su dirección directa
        for href, texto in anclas_canon(htmls["CANON Y SOBRE CANON"]):
            if "/wps/" in href and ("CANON MUNICIPALIDADES" in texto or
                                    re.search(r"canon(%20|\+|\s)+municipalidades", unquote(href) + href, re.I)):
                paginas_canon["MUNICIPALIDADES"] = urljoin(PAGINA_CANON, href)
                break
        candidatas = ([paginas_canon["MUNICIPALIDADES"]] if "MUNICIPALIDADES" in paginas_canon else []) + \
                     [PAGINA_CANON.split("canon%20sobre%20canon")[0] + "canon%20municipalidades/"]
        for url_m in candidatas:
            time.sleep(PAUSA)
            try:
                h_m = sesion_c.get(url_m, headers=CAB, timeout=60, verify=VERIFY).text
            except Exception:
                continue
            if re.search(r"CANON\s+POR\s+A.?O", sin_tildes(html.unescape(h_m))):   # la página trae sus Excel
                paginas_canon["MUNICIPALIDADES"], htmls["MUNICIPALIDADES"] = url_m, h_m
                break
        if "MUNICIPALIDADES" not in htmls:
            print("⚠️ No se pudo abrir la página «Canon Municipalidades»: pon esos Excel a mano en DATA/CANON_EXCEL.")
    except Exception as e:
        print(f"⚠️ No se pudo leer la página de canon de PERUPETRO ({type(e).__name__}: {e}). "
              "Se usarán solo los Excel que ya estén en Drive/DATA/CANON_EXCEL.")
    vistos = set()
    for pag, h_ in htmls.items():
        # cada Excel suele tener dos enlaces a la misma dirección (el ícono, sin texto, y el texto): se juntan
        por_url = {}
        for href, texto in anclas_canon(h_):
            if "/wps/wcm/connect/" not in href and not re.search(r"\.xlsx?\b", href, re.I):
                continue                                  # menú y enlaces internos del portal
            u = urljoin(paginas_canon[pag], href)
            por_url.setdefault(u, [href, ""])
            if texto and texto not in por_url[u][1]:
                por_url[u][1] = (por_url[u][1] + " " + texto).strip()
        for url, (href, texto) in por_url.items():
            if re.search(r"REGLAMENTO|DECRETO|RESOLUCION|\bLEY\b|DIRECTIVA", texto):
                continue                                  # documentos legales enlazados en la misma página
            if url in vistos:
                previa = next((f for f in ctrl_canon if f["URL"] == url), None)
                if previa and previa["PAGINA"] != pag:    # la otra página enlaza el mismo archivo: se anota, no se baja
                    ctrl_canon.append({**previa, "PAGINA": pag, "TEXTO_ENLACE": texto,
                                       "ESTADO": f"MISMO ARCHIVO QUE {previa['PAGINA']}"})
                continue
            vistos.add(url)
            c = clasificar_canon(texto, href, pag)
            contenido, err = None, ""
            if not c:
                # enlace sin texto útil (p. ej., un ícono): se baja y se clasifica por la cabecera del Excel
                contenido, err = bajar_canon(url)
                ext_s = tipo_excel(contenido) if contenido else None
                dep, cabecera = departamento_excel(contenido) if ext_s else (None, "")
                tipo = "MUNICIPAL" if pag == "MUNICIPALIDADES" else "DISTRIBUCION"
                if dep:
                    base = f"PP_CANON_{tipo}_{dep}"
                    if any(f["ARCHIVO"].rsplit(".", 1)[0] == base for f in ctrl_canon):   # dos enlaces, mismo departamento
                        base += f"_{sum(1 for f in ctrl_canon if f['ARCHIVO'].startswith(base)) + 1}"
                    c = (tipo, None, dep, base)
                elif ext_s:
                    # Excel sin departamento reconocible: se guarda igual para revisarlo a mano
                    n_s = sum(1 for f in ctrl_canon if f["ESTADO"].startswith("GUARDADO SIN DEPARTAMENTO") and f["PAGINA"] == pag) + 1
                    nombre = f"PP_CANON_{tipo}_SIN_DEPARTAMENTO_{n_s}{ext_s}"
                    with open(os.path.join(ruta, nombre), "wb") as fh:
                        fh.write(contenido)
                    subir(os.path.join(ruta, nombre), CANON_FOLDER_ID, canon_drive,
                          mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet" if ext_s == ".xlsx"
                          else "application/vnd.ms-excel")
            fila = {"PAGINA": pag, "TEXTO_ENLACE": texto, "URL": url, "TIPO": c[0] if c else "",
                    "AÑO": c[1] if c else None, "DEPARTAMENTO": c[2] if c else "", "ARCHIVO": "", "BYTES": None,
                    "ESTADO": "SIN CLASIFICAR", "CABECERA": ""}
            if not c and contenido and tipo_excel(contenido):
                fila.update(ARCHIVO=nombre, BYTES=len(contenido), CABECERA=cabecera,
                            ESTADO="GUARDADO SIN DEPARTAMENTO (ver CABECERA)")
            if c:
                tipo, anio, dep, base = c
                ya = [n for n in canon_drive if n.rsplit(".", 1)[0] == base]
                if ya and tipo == "DETERMINACION" and anio < año_hoy:
                    fila.update(ARCHIVO=ya[0], ESTADO="YA EN DRIVE")          # año cerrado: no se vuelve a bajar
                else:
                    if contenido is None:
                        contenido, err = bajar_canon(url)
                    ext = tipo_excel(contenido) if contenido else None
                    if ext:
                        nombre = base + ext
                        with open(os.path.join(ruta, nombre), "wb") as fh:
                            fh.write(contenido)
                        subir(os.path.join(ruta, nombre), CANON_FOLDER_ID, canon_drive,
                              mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet" if ext == ".xlsx"
                              else "application/vnd.ms-excel")
                        fila.update(ARCHIVO=nombre, BYTES=len(contenido), ESTADO="DESCARGADO")
                    else:
                        fila.update(ESTADO=f"ERROR: {err or 'no es un Excel'}")
                    time.sleep(PAUSA)
            ctrl_canon.append(fila)
    # Excel puestos a mano en la carpeta (respaldo si la descarga automática falla)
    usados = {f["ARCHIVO"] for f in ctrl_canon if f["ARCHIVO"]}
    for n in sorted(canon_drive):
        if re.search(r"\.xlsx?$", n, re.I) and n not in usados:
            ctrl_canon.append({"PAGINA": "DRIVE", "TEXTO_ENLACE": "", "URL": "", "TIPO": "", "AÑO": None,
                               "DEPARTAMENTO": "", "ARCHIVO": n, "BYTES": None, "CABECERA": "",
                               "ESTADO": "MANUAL" if not n.startswith("PP_CANON_") else "EN DRIVE (no enlazado hoy)"})
    ctrl_c = pd.DataFrame(ctrl_canon, columns=["PAGINA", "TEXTO_ENLACE", "URL", "TIPO", "AÑO", "DEPARTAMENTO", "ARCHIVO",
                                               "BYTES", "ESTADO", "CABECERA"])
    ctrl_c["FECHA_EXTRACCION"] = date.today().strftime("%d/%m/%Y")
    ctrl_c.to_csv(os.path.join(ruta, NOMBRE_CONTROL_CANON), index=False, encoding="utf-8-sig")
    subir(os.path.join(ruta, NOMBRE_CONTROL_CANON), DRIVE_FOLDER_CONTROL_ID, control_drive)
    if len(ctrl_c):
        print(f"🔗 Enlaces y archivos: {len(ctrl_c)}")
        display(ctrl_c.groupby(["PAGINA", "TIPO", "ESTADO"], dropna=False).size().to_frame("archivos"))
        rev = ctrl_c[ctrl_c["ESTADO"].str.startswith(("ERROR", "SIN CLASIFICAR", "GUARDADO SIN"))]
        if len(rev):
            print("⚠️ A revisar (manda este cuadro o el CSV de control):")
            with pd.option_context("display.max_colwidth", 200):
                display(rev[["PAGINA", "ARCHIVO", "ESTADO", "CABECERA"]])
    else:
        print("⚠️ No se encontraron Excel de canon ni en la página ni en Drive/DATA/CANON_EXCEL.")
    print(f"   Excel:   https://drive.google.com/drive/folders/{CANON_FOLDER_ID}")
    print(f"   Control: CONTROL/{NOMBRE_CONTROL_CANON}")
    # ── Lectura de los Excel (punto 2): tablas mensuales y control contra los totales del Excel y de los PDF ──
    print("\n📖 Lectura de los Excel de canon")
    locales_c = {}
    for n, fid in sorted(canon_drive.items()):
        m_c = re.fullmatch(r"PP_CANON_(DETERMINACION|DISTRIBUCION|MUNICIPAL)_(\w+?)(_\d+)?\.xlsx?", n)
        if not m_c:
            continue
        p_c = os.path.join(ruta, n)
        if not os.path.exists(p_c):
            try:
                bajar_de_drive(fid, p_c)
            except Exception as e:
                print(f"   ⚠️ No se pudo bajar {n} de Drive ({type(e).__name__})")
                continue
        locales_c[n] = (m_c.group(1), m_c.group(2), p_c)
    det_f, det_s, dis_b, mun_f, ctrl_l = [], [], [], [], []
    for n, (tipo, clave, p_c) in locales_c.items():
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                if tipo == "DETERMINACION":
                    f_, s_, k_ = leer_determinacion(p_c, int(clave))
                    det_f += f_; det_s += s_
                elif tipo == "DISTRIBUCION":
                    b_, c_, k_ = leer_distribucion(p_c, clave)
                    dis_b += b_; mun_f += c_
                else:
                    f_, k_ = leer_municipal(p_c, clave)
                    mun_f += f_
            ctrl_l += [{"FUENTE": tipo, "ARCHIVO": n, **k} for k in k_]
        except Exception as e:
            ctrl_l.append({"FUENTE": tipo, "ARCHIVO": n, "PARTE": "ARCHIVO", "ESTADO": f"ERROR: {type(e).__name__}: {e}"})
    tablas_c = {"PP_canon_lote_mes": pd.DataFrame(det_f), "PP_canon_distribucion_mes": pd.DataFrame(dis_b),
                "PP_canon_municipal_mes": pd.DataFrame(mun_f)}
    # Cruces: (1) soles por departamento de la Determinación = total de la Distribución del mismo mes;
    # (2) Distribución = canon mensual y anual de los PDF; (3) municipalidades = rubro municipal de la Distribución.
    dis_df = tablas_c["PP_canon_distribucion_mes"]
    if len(dis_df):
        tot_d = dis_df.groupby(["DEPARTAMENTO", "AÑO", "MES"])["MONTO_SOLES"].sum()
        for s_ in det_s:
            ref = tot_d.get((s_["DEPARTAMENTO"], s_["AÑO"], s_["MES"]))
            if ref is not None:
                ctrl_l.append({"FUENTE": "CRUCE DETERMINACION S/ vs DISTRIBUCION", "DEPARTAMENTO": s_["DEPARTAMENTO"],
                               "AÑO": s_["AÑO"], "MES": s_["MES"], "SUMA": s_["CANON_SOLES"], "TOTAL_EXCEL": round(ref, 2),
                               "ESTADO": "OK" if abs(s_["CANON_SOLES"] - ref) <= max(10, 1e-4 * ref) else "REVISAR"})
        def _csv_datos(nombre):
            if nombre.rsplit(".", 1)[0] in extra_ok:
                return extra_ok[nombre.rsplit(".", 1)[0]]
            if nombre in datos_drive:
                try:
                    bajar_de_drive(datos_drive[nombre], os.path.join(ruta, nombre))
                    return pd.read_csv(os.path.join(ruta, nombre), encoding="utf-8-sig")
                except Exception:
                    pass
            return pd.DataFrame()
        for nombre, claves in (("PP_canon_departamento_mes.csv", ["AÑO", "MES"]), ("PP_canon_departamento_anio.csv", ["AÑO"])):
            pdf_c = _csv_datos(nombre)
            if not len(pdf_c):
                ctrl_l.append({"FUENTE": f"CRUCE DISTRIBUCION vs {nombre}", "ESTADO": "SIN DATOS DE LOS PDF EN DRIVE"})
                continue
            pdf_c = pdf_c.assign(DEP=pdf_c["DEPARTAMENTO"].astype(str).map(sin_tildes))
            xl = dis_df.groupby(["DEPARTAMENTO"] + claves)["MONTO_SOLES"].sum() / 1e6
            for _, f_ in pdf_c.iterrows():
                k_ = (f_["DEP"],) + tuple(int(f_[c]) for c in claves)
                if k_ not in xl.index:
                    continue
                if len(claves) == 1 and dis_df[(dis_df["DEPARTAMENTO"] == k_[0]) & (dis_df["AÑO"] == k_[1])]["MES"].nunique() < 12:
                    continue                                     # año incompleto en el Excel
                dif = round(xl[k_] - f_["CANON_MM_SOLES"], 2)
                ctrl_l.append({"FUENTE": f"CRUCE DISTRIBUCION vs {nombre}", "DEPARTAMENTO": k_[0], "AÑO": k_[1],
                               "MES": k_[2] if len(k_) > 2 else None, "SUMA": round(xl[k_], 2),
                               "TOTAL_EXCEL": f_["CANON_MM_SOLES"], "ESTADO": "OK" if abs(dif) <= 0.011 else "REVISAR"})
        mun_df = tablas_c["PP_canon_municipal_mes"]
        if len(mun_df):
            rub = dis_df[dis_df["TIPO_BENEFICIARIO"] == "MUNICIPALIDADES"].groupby(["DEPARTAMENTO", "AÑO", "MES"])["MONTO_SOLES"].sum()
            for k_, v_ in mun_df.groupby(["DEPARTAMENTO", "AÑO", "MES"])["MONTO_SOLES"].sum().items():
                if k_ in rub.index:
                    ctrl_l.append({"FUENTE": "CRUCE MUNICIPALIDADES vs DISTRIBUCION", "DEPARTAMENTO": k_[0], "AÑO": k_[1],
                                   "MES": k_[2], "SUMA": round(v_, 2), "TOTAL_EXCEL": round(rub[k_], 2),
                                   "ESTADO": "OK" if abs(v_ - rub[k_]) <= max(10, 1e-4 * rub[k_]) else "REVISAR"})
    # el Excel de Municipalidades solo trae el año en curso: se acumula en Drive para no perder los años anteriores
    tablas_c["PP_canon_municipal_mes"] = _acumular("PP_canon_municipal_mes.csv", tablas_c["PP_canon_municipal_mes"],
                                                   ["DEPARTAMENTO", "AÑO", "MES", "SECCION", "FUENTE_EXCEL"])
    for nombre, df_c in tablas_c.items():
        if len(df_c):
            df_c.to_csv(os.path.join(ruta, nombre + ".csv"), index=False, encoding="utf-8-sig")
            subir(os.path.join(ruta, nombre + ".csv"), DRIVE_FOLDER_ID, datos_drive)
    NOMBRE_CONTROL_LECT = "CONTROL PP canon_excel_lectura.csv"
    ctrl_lc = pd.DataFrame(ctrl_l, columns=["FUENTE", "ARCHIVO", "DEPARTAMENTO", "AÑO", "MES", "HOJA", "PARTE", "SUMA",
                                            "TOTAL_EXCEL", "ESTADO"])
    # Diferencias entre fuentes de PERUPETRO ya revisadas (19/10/2026, ver perupetro_produccion.md): no son del lector
    CONOCIDAS = {
        ("MUNICIPAL", "LORETO", 2025, 9): "el TOTAL GOBIERNOS LOCALES del Excel omite la provincia de Ucayali",
        ("CRUCE MUNICIPALIDADES", "HUANUCO", 2025, 5): "el Excel de Municipalidades no da monto a Tornavista; la Distribución sí",
        ("CRUCE DETERMINACION", "HUANUCO", 1996, 6): "diferencia entre Determinación y Distribución",
        ("CRUCE DETERMINACION", "LORETO", 2006, 6): "diferencia entre Determinación y Distribución",
        ("CRUCE DETERMINACION", "UCAYALI", 2006, 6): "diferencia entre Determinación y Distribución",
        ("CRUCE DETERMINACION", "TUMBES", 2007, 9): "diferencia entre Determinación y Distribución",
        ("CRUCE DETERMINACION", "TUMBES", 2008, 4): "diferencia entre Determinación y Distribución",
        ("CRUCE DETERMINACION", "PIURA", 2009, 9): "la Distribución tiene S/ 0,53 MM más (igual en el anual)",
        ("CRUCE DETERMINACION", "LORETO", 2009, 12): "diferencia entre Determinación y Distribución",
        ("CRUCE DETERMINACION", "CUSCO", 2010, 6): "diferencia entre Determinación y Distribución",
        ("CRUCE DETERMINACION", "PIURA", 2026, 2): "mismo total Piura + Tumbes, distinto reparto",
        ("CRUCE DETERMINACION", "TUMBES", 2026, 2): "mismo total Piura + Tumbes, distinto reparto",
        ("CRUCE DISTRIBUCION vs PP_canon_departamento_anio", "TUMBES", 2007, None): "Excel 71,69 vs PDF anual 70,27",
        ("CRUCE DISTRIBUCION vs PP_canon_departamento_anio", "PIURA", 2009, None): "Excel 270,75 vs PDF anual 270,22",
        ("CRUCE DISTRIBUCION vs PP_canon_departamento_anio", "CUSCO", 2015, None): "Excel 942,73 vs PDF anual 939,73",
        ("CRUCE DISTRIBUCION vs PP_canon_departamento_anio", "PIURA", 2024, None): "Excel 407,29 vs PDF anual 374,79",
    }
    def _conocida(f):
        if f["ESTADO"] != "REVISAR":
            return f["ESTADO"]
        mes = None if pd.isna(f["MES"]) else int(f["MES"])
        for (fu, dep, a, m), nota in CONOCIDAS.items():
            if str(f["FUENTE"]).startswith(fu) and f["DEPARTAMENTO"] == dep and f["AÑO"] == a and m == mes:
                return f"DIFERENCIA DE LA FUENTE (conocida): {nota}"
        return "REVISAR"
    if len(ctrl_lc):
        ctrl_lc["ESTADO"] = ctrl_lc.apply(_conocida, axis=1)
    ctrl_lc["FECHA_EXTRACCION"] = date.today().strftime("%d/%m/%Y")
    ctrl_lc.to_csv(os.path.join(ruta, NOMBRE_CONTROL_LECT), index=False, encoding="utf-8-sig")
    subir(os.path.join(ruta, NOMBRE_CONTROL_LECT), DRIVE_FOLDER_CONTROL_ID, control_drive)
    for nombre, df_c in tablas_c.items():
        if len(df_c):
            print(f"   {nombre}.csv: {len(df_c):,} filas · {df_c['AÑO'].min()}–{df_c['AÑO'].max()} · "
                  f"{df_c['DEPARTAMENTO'].nunique() if 'DEPARTAMENTO' in df_c else len(set(df_c['DEPARTAMENTO_1']) | set(df_c['DEPARTAMENTO_2'].dropna()) - {''})} departamentos")
        else:
            print(f"   {nombre}.csv: sin filas")
    if len(ctrl_lc):
        display(ctrl_lc.assign(ESTADO=ctrl_lc["ESTADO"].str.split(":").str[0])
                .groupby(["FUENTE", "ESTADO"], dropna=False).size().to_frame("controles"))
        rev_l = ctrl_lc[~ctrl_lc["ESTADO"].fillna("").str.startswith(("OK", "VACIA", "SIN REPARTO", "DIFERENCIA DE LA FUENTE"))]
        if len(rev_l):
            print(f"⚠️ A revisar: {len(rev_l)} (detalle en CONTROL/{NOMBRE_CONTROL_LECT}); primeros 15:")
            display(rev_l.drop(columns=["FECHA_EXTRACCION"]).head(15))
        else:
            n_c = ctrl_lc["ESTADO"].fillna("").str.startswith("DIFERENCIA DE LA FUENTE").sum()
            print(f"✅ Sin casos nuevos a revisar ({n_c} diferencias conocidas de PERUPETRO, detalle en el control).")
    dl = tablas_c["PP_canon_distribucion_mes"]
    if len(dl) and (dl["DEPARTAMENTO"] == "LORETO").any():
        lo = dl[(dl["DEPARTAMENTO"] == "LORETO") & (dl["AÑO"] >= año_hoy - 5)]
        print("Loreto · canon por beneficiario (millones de soles):")
        display((lo.pivot_table(index="TIPO_BENEFICIARIO", columns="AÑO", values="MONTO_SOLES", aggfunc="sum", margins=True,
                                margins_name="TOTAL") / 1e6).round(2))
    dt = tablas_c["PP_canon_lote_mes"]
    if len(dt) and (dt["DEPARTAMENTO_1"] == "LORETO").any():
        lo = dt[(dt["DEPARTAMENTO_1"] == "LORETO") & (dt["AÑO"] >= año_hoy - 1)]
        print("Loreto · canon que le toca por lote (millones de US$; los ajustes de meses anteriores van aparte):")
        display((lo.pivot_table(index="LOTE", columns="AÑO", values="CANON_DEP_1_US$", aggfunc="sum") / 1e6).round(2))
